
# WI-05: Rainfall Model Development — Conclusion

## 1. Objective

This notebook developed and evaluated machine learning models for daily rainfall forecasting using the `PRECTOTCORR` variable.

Forecasts were evaluated for T+1 through T+7 at the weather grid-cell level.

## 2. Models Investigated

### Direct Regression
- Ridge Regression
- Random Forest Regressor
- HistGradientBoosting Regressor

### Wet-Day Classification
- Logistic Regression
- Random Forest Classifier
- HistGradientBoosting Classifier

### Conditional Rainfall Regression
- Ridge Regression
- Random Forest Regressor
- HistGradientBoosting Regressor

Only models successfully trained and evaluated are included in the final experiment results.

## 3. Data Splitting

- Training: 2013–2019
- Validation: 2020–2022
- Final holdout: 2023–2024

The final holdout was reserved for WI-07.

## 4. Model Evaluation

Models were evaluated using:

- MAE and RMSE
- Precision, Recall, F1-score and PR-AUC
- Wet-day-only MAE
- Rainfall intensity-band errors
- Seasonal and geographic performance
- Forecast-horizon performance

## 5. Baseline Comparison

The developed models were compared against the WI-04 persistence, seasonal-naive and training-period climatology baselines.

The best-performing candidate for each forecast horizon was documented using validation results.

## 6. Key Findings

[Add findings based on actual experiment results.]

## 7. Selected Candidate Models

[Record the selected rainfall model or models and their validation metrics.]

Final model selection and untouched holdout evaluation will be completed in WI-07.

## 8. Limitations

- Historical weather data ends in 2024.
- Grid-cell predictions do not automatically represent district-level conditions.
- Historical-data-based predictions are not live operational weather forecasts.
- Rainfall forecasts alone do not establish crop damage or disease risk.

## 9. Artifacts Generated

[List actual model files, configuration files, metric reports and evaluation outputs.]

## 10. Final Conclusion

WI-05 completed the rainfall model development and validation experiments.

Production approval is not granted by this notebook. Final model selection and holdout evaluation are reserved for WI-07.


In [1]:

# WI-05 — Cell 1: Environment and Dataset Verification

from pathlib import Path
import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# Find the project directory
PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "phase3b_weather_intelligence").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "phase3b_weather_intelligence"
).exists(), "Project directory not found."

# Feature dataset location
FEATURE_DATASET_DIR = (
    PROJECT_ROOT
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_features_v1"
    / "weather-features-v1"
)

# Experiment report directory
REPORT_DIR = (
    PROJECT_ROOT
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
    / "WI_05_rainfall_model_development"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Check the feature dataset manifest
MANIFEST_PATH = FEATURE_DATASET_DIR / "batch_manifest.json"

assert FEATURE_DATASET_DIR.exists(), "Feature dataset not found."
assert MANIFEST_PATH.exists(), "Batch manifest not found."

# Load the manifest
with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    feature_manifest = json.load(file)

# Find the Parquet batches
FEATURE_BATCH_PATHS = sorted(
    FEATURE_DATASET_DIR.glob("batch_*.parquet")
)

assert len(FEATURE_BATCH_PATHS) == 252, (
    f"Expected 252 batches, found {len(FEATURE_BATCH_PATHS)}"
)

# Inspect the first batch without loading the full dataset
first_batch_schema = pq.read_schema(FEATURE_BATCH_PATHS[0])
feature_columns = first_batch_schema.names

# Experiment configuration
TARGET_VARIABLE = "PRECTOTCORR"
FORECAST_HORIZONS = list(range(1, 8))

TRAIN_START = pd.Timestamp("2013-01-01")
TRAIN_END = pd.Timestamp("2019-12-31")

VALIDATION_START = pd.Timestamp("2020-01-01")
VALIDATION_END = pd.Timestamp("2022-12-31")

HOLDOUT_START = pd.Timestamp("2023-01-01")
HOLDOUT_END = pd.Timestamp("2024-12-31")

WET_DAY_THRESHOLD = 1.0  # mm/day

# Display verification results
print("WI-05 ENVIRONMENT VERIFICATION: COMPLETE")
print("Project root:", PROJECT_ROOT)
print("Feature dataset:", FEATURE_DATASET_DIR)
print("Report directory:", REPORT_DIR)
print("Parquet batches:", len(FEATURE_BATCH_PATHS))
print("Columns:", len(feature_columns))
print("Target:", TARGET_VARIABLE)
print("Forecast horizons:", FORECAST_HORIZONS)
print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Validation period:", VALIDATION_START.date(), "to", VALIDATION_END.date())
print("Final holdout:", HOLDOUT_START.date(), "to", HOLDOUT_END.date())
print("Wet-day threshold:", WET_DAY_THRESHOLD, "mm/day")
print("Final holdout accessed: NO")


WI-05 ENVIRONMENT VERIFICATION: COMPLETE
Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Feature dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1
Report directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development
Parquet batches: 252
Columns: 69
Target: PRECTOTCORR
Forecast horizons: [1, 2, 3, 4, 5, 6, 7]
Training period: 2013-01-01 to 2019-12-31
Validation period: 2020-01-01 to 2022-12-31
Final holdout: 2023-01-01 to 2024-12-31
Wet-day threshold: 1.0 mm/day
Final holdout accessed: NO


In [2]:

# WI-05 — Cell 2: Rainfall Target Distribution Analysis

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Store the required rainfall target columns
rainfall_target_columns = [
    col for col in feature_columns
    if TARGET_VARIABLE in col
    and "target" in col.lower()
]

assert len(rainfall_target_columns) == 7, (
    f"Expected 7 rainfall target columns, found {len(rainfall_target_columns)}"
)

print("Rainfall target columns:")
print(rainfall_target_columns)

# Collect training-period rainfall observations
rainfall_values = {col: [] for col in rainfall_target_columns}

for batch_path in FEATURE_BATCH_PATHS:

    # Read only dates and rainfall targets
    df = pd.read_parquet(
        batch_path,
        columns=["date"] + rainfall_target_columns
    )

    df["date"] = pd.to_datetime(df["date"])

    # Keep only training-period rows based on target date
    for col in rainfall_target_columns:

        # Extract forecast horizon from the column name
        horizon = int(col.split("t_plus_")[-1])

        target_dates = df["date"] + pd.to_timedelta(
            horizon, unit="D"
        )

        training_mask = (
            (target_dates >= TRAIN_START)
            & (target_dates <= TRAIN_END)
        )

        values = df.loc[training_mask, col].dropna()

        rainfall_values[col].append(values.to_numpy())

# Calculate statistics for each forecast horizon
rainfall_summary = []

for col, value_batches in rainfall_values.items():

    values = np.concatenate(value_batches)

    # Basic data-quality checks
    assert np.isfinite(values).all(), (
        f"Non-finite rainfall values found in {col}"
    )

    assert (values >= 0).all(), (
        f"Negative rainfall values found in {col}"
    )

    # Identify dry and wet observations
    dry_count = int((values < WET_DAY_THRESHOLD).sum())
    wet_count = int((values >= WET_DAY_THRESHOLD).sum())

    rainfall_summary.append({
        "target_column": col,
        "observations": int(len(values)),
        "dry_days_below_1mm": dry_count,
        "wet_days_at_least_1mm": wet_count,
        "wet_day_percentage": float(
            wet_count / len(values) * 100
        ),
        "mean_mm": float(np.mean(values)),
        "median_mm": float(np.median(values)),
        "std_mm": float(np.std(values)),
        "p90_mm": float(np.percentile(values, 90)),
        "p95_mm": float(np.percentile(values, 95)),
        "p99_mm": float(np.percentile(values, 99)),
        "maximum_mm": float(np.max(values)),
    })

# Create a readable table
rainfall_summary_df = pd.DataFrame(rainfall_summary)

print("\nRAINFALL DISTRIBUTION SUMMARY")
display(rainfall_summary_df.round(4))

# Save the findings
output_path = REPORT_DIR / "WI_05_rainfall_distribution.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        rainfall_summary,
        file,
        indent=2
    )

print("\nRAINFALL DISTRIBUTION ANALYSIS: COMPLETE")
print("Forecast horizons analyzed:", len(rainfall_summary_df))
print("Wet-day threshold:", WET_DAY_THRESHOLD, "mm/day")
print("Saved:", output_path)
print("Validation data used: NO")
print("Final holdout accessed: NO")


Rainfall target columns:
['PRECTOTCORR_target_t_plus_1', 'PRECTOTCORR_target_t_plus_2', 'PRECTOTCORR_target_t_plus_3', 'PRECTOTCORR_target_t_plus_4', 'PRECTOTCORR_target_t_plus_5', 'PRECTOTCORR_target_t_plus_6', 'PRECTOTCORR_target_t_plus_7']

RAINFALL DISTRIBUTION SUMMARY


,target_column,observations,dry_days_below_1mm,wet_days_at_least_1mm,wet_day_percentage,mean_mm,median_mm,std_mm,p90_mm,p95_mm,p99_mm,maximum_mm
0,PRECTOTCORR_target_t_plus_1,6438600,3961972,2476628,38.4653,3.5976,0.22,9.4996,10.45,17.4700,40.24,1038.34
1,PRECTOTCORR_target_t_plus_2,6436080,3959869,2476211,38.4739,3.5981,0.22,9.4999,10.46,17.4700,40.23,1038.34
2,PRECTOTCORR_target_t_plus_3,6433560,3957715,2475845,38.4833,3.5988,0.23,9.5008,10.46,17.4700,40.24,1038.34
3,PRECTOTCORR_target_t_plus_4,6431040,3955455,2475585,38.4943,3.5996,0.23,9.5019,10.46,17.4700,40.24,1038.34
4,PRECTOTCORR_target_t_plus_5,6428520,3953167,2475353,38.5058,3.6008,0.23,9.5035,10.46,17.4705,40.25,1038.34
5,PRECTOTCORR_target_t_plus_6,6426000,3950955,2475045,38.5161,3.6020,0.23,9.5050,10.47,17.4800,40.26,1038.34
6,PRECTOTCORR_target_t_plus_7,6423480,3948855,2474625,38.5247,3.6029,0.23,9.5061,10.47,17.4800,40.26,1038.34



RAINFALL DISTRIBUTION ANALYSIS: COMPLETE
Forecast horizons analyzed: 7
Wet-day threshold: 1.0 mm/day
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_rainfall_distribution.json
Validation data used: NO
Final holdout accessed: NO


In [3]:

# WI-05 — Cell 3: Predictor Feature Selection

# Columns that must never be used as predictors
excluded_columns = [
    "cell_id",
    "date",
]

# Exclude every target column from both weather variables
target_columns_to_exclude = [
    col for col in feature_columns
    if "target" in col.lower()
]

# Exclude any other future-oriented columns if present
future_columns = [
    col for col in feature_columns
    if "future" in col.lower()
]

excluded_columns = list(set(
    excluded_columns
    + target_columns_to_exclude
    + future_columns
))

# Keep numerical predictor columns only
predictor_columns = [
    col for col in feature_columns
    if col not in excluded_columns
]

# Confirm that the feature list is valid
assert len(predictor_columns) > 0
assert not any("target" in col.lower() for col in predictor_columns)
assert "cell_id" not in predictor_columns
assert "date" not in predictor_columns

# Inspect predictor data types
schema = pq.read_schema(FEATURE_BATCH_PATHS[0])

numeric_predictors = [
    col for col in predictor_columns
    if str(schema.field(col).type) in [
        "double", "float", "int64", "int32",
        "int16", "int8", "uint64", "uint32"
    ]
]

assert len(numeric_predictors) == len(predictor_columns), (
    "Non-numeric predictors found. Inspect before proceeding."
)

# Save feature specification
feature_specification = {
    "target_variable": TARGET_VARIABLE,
    "predictor_count": len(predictor_columns),
    "predictors": predictor_columns,
    "excluded_columns": excluded_columns,
    "target_columns_excluded": target_columns_to_exclude,
    "future_columns_excluded": future_columns,
    "identifier_columns_excluded": ["cell_id", "date"],
    "feature_selection": "Numerical predictors from WI-03 feature dataset",
    "final_holdout_accessed": False
}

output_path = REPORT_DIR / "WI_05_feature_specification.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(feature_specification, file, indent=2)

print("PREDICTOR FEATURE SELECTION: COMPLETE")
print("Total dataset columns:", len(feature_columns))
print("Excluded columns:", len(excluded_columns))
print("Selected numerical predictors:", len(predictor_columns))
print("\nSelected predictors:")
for col in predictor_columns:
    print("-", col)

print("\nSaved:", output_path)
print("Future target columns included: NO")
print("Final holdout accessed: NO")


PREDICTOR FEATURE SELECTION: COMPLETE
Total dataset columns: 69
Excluded columns: 16
Selected numerical predictors: 53

Selected predictors:
- latitude
- longitude
- PRECTOTCORR_current
- T2M_MAX_current
- T2M_MIN_current
- PRECTOTCORR_lag_1
- PRECTOTCORR_lag_2
- PRECTOTCORR_lag_3
- PRECTOTCORR_lag_7
- PRECTOTCORR_lag_14
- PRECTOTCORR_lag_30
- T2M_MAX_lag_1
- T2M_MAX_lag_2
- T2M_MAX_lag_3
- T2M_MAX_lag_7
- T2M_MAX_lag_14
- T2M_MAX_lag_30
- T2M_MIN_lag_1
- T2M_MIN_lag_2
- T2M_MIN_lag_3
- T2M_MIN_lag_7
- T2M_MIN_lag_14
- T2M_MIN_lag_30
- PRECTOTCORR_rolling_mean_3
- PRECTOTCORR_rolling_std_3
- PRECTOTCORR_rolling_mean_7
- PRECTOTCORR_rolling_std_7
- PRECTOTCORR_rolling_mean_14
- PRECTOTCORR_rolling_std_14
- PRECTOTCORR_rolling_mean_30
- PRECTOTCORR_rolling_std_30
- T2M_MAX_rolling_mean_3
- T2M_MAX_rolling_std_3
- T2M_MAX_rolling_mean_7
- T2M_MAX_rolling_std_7
- T2M_MAX_rolling_mean_14
- T2M_MAX_rolling_std_14
- T2M_MAX_rolling_mean_30
- T2M_MAX_rolling_std_30
- T2M_MIN_rolling_mean_3
- T

In [4]:

# WI-05 — Cell 4: Rainfall Experiment Configuration

import json

# Models planned for direct rainfall regression
REGRESSION_MODELS = [
    "Ridge",
    "RandomForest",
    "HistGradientBoosting"
]

# Models planned for wet-day classification
CLASSIFICATION_MODELS = [
    "LogisticRegression",
    "RandomForestClassifier",
    "HistGradientBoostingClassifier"
]

# Models planned for conditional rainfall regression
CONDITIONAL_REGRESSION_MODELS = [
    "Ridge",
    "RandomForest",
    "HistGradientBoosting"
]

# Evaluation metrics
REGRESSION_METRICS = [
    "MAE",
    "RMSE"
]

CLASSIFICATION_METRICS = [
    "Precision",
    "Recall",
    "F1-score",
    "PR-AUC"
]

ADDITIONAL_EVALUATIONS = [
    "Wet-day-only MAE",
    "Rainfall intensity-band MAE",
    "Seasonal performance",
    "Geographic performance",
    "Forecast-horizon performance"
]

# Experiment settings
EXPERIMENT_CONFIG = {
    "experiment_name": "WI-05 Rainfall Model Development",
    "target_variable": TARGET_VARIABLE,
    "forecast_horizons": FORECAST_HORIZONS,
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "training_period": {
        "start": str(TRAIN_START.date()),
        "end": str(TRAIN_END.date())
    },
    "validation_period": {
        "start": str(VALIDATION_START.date()),
        "end": str(VALIDATION_END.date())
    },
    "final_holdout_period": {
        "start": str(HOLDOUT_START.date()),
        "end": str(HOLDOUT_END.date())
    },
    "regression_models": REGRESSION_MODELS,
    "classification_models": CLASSIFICATION_MODELS,
    "conditional_regression_models": CONDITIONAL_REGRESSION_MODELS,
    "regression_metrics": REGRESSION_METRICS,
    "classification_metrics": CLASSIFICATION_METRICS,
    "additional_evaluations": ADDITIONAL_EVALUATIONS,
    "predictor_count": len(predictor_columns),
    "random_split": False,
    "final_holdout_accessed": False,
    "production_approved": False
}

# Save configuration
config_path = REPORT_DIR / "WI_05_experiment_configuration.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(EXPERIMENT_CONFIG, file, indent=2)

# Verify saved configuration
with open(config_path, "r", encoding="utf-8") as file:
    verified_config = json.load(file)

assert verified_config["target_variable"] == "PRECTOTCORR"
assert verified_config["predictor_count"] == 53
assert verified_config["final_holdout_accessed"] is False

print("WI-05 EXPERIMENT CONFIGURATION: SAVED AND VERIFIED")
print("Regression models:", REGRESSION_MODELS)
print("Classification models:", CLASSIFICATION_MODELS)
print("Conditional regression models:", CONDITIONAL_REGRESSION_MODELS)
print("Predictors:", len(predictor_columns))
print("Forecast horizons:", FORECAST_HORIZONS)
print("Saved:", config_path)
print("Final holdout accessed: NO")


WI-05 EXPERIMENT CONFIGURATION: SAVED AND VERIFIED
Regression models: ['Ridge', 'RandomForest', 'HistGradientBoosting']
Classification models: ['LogisticRegression', 'RandomForestClassifier', 'HistGradientBoostingClassifier']
Conditional regression models: ['Ridge', 'RandomForest', 'HistGradientBoosting']
Predictors: 53
Forecast horizons: [1, 2, 3, 4, 5, 6, 7]
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_experiment_configuration.json
Final holdout accessed: NO


In [5]:

# WI-05 — Cell 5: Prepare a Small Training Pilot

import numpy as np
import pandas as pd

# Select 10 batches spread across the dataset
pilot_indices = np.linspace(
    0,
    len(FEATURE_BATCH_PATHS) - 1,
    10,
    dtype=int
)

PILOT_BATCH_PATHS = [
    FEATURE_BATCH_PATHS[i] for i in pilot_indices
]

# Use T+1 for the initial pilot
PILOT_HORIZON = 1

PILOT_TARGET_COLUMN = (
    f"{TARGET_VARIABLE}_target_t_plus_{PILOT_HORIZON}"
)

# Store a small sample from each batch
pilot_frames = []

for batch_path in PILOT_BATCH_PATHS:

    # Read only the required columns
    df = pd.read_parquet(
        batch_path,
        columns=["date"] + predictor_columns + [PILOT_TARGET_COLUMN]
    )

    df["date"] = pd.to_datetime(df["date"])

    # Calculate the actual target date
    df["target_date"] = (
        df["date"] + pd.to_timedelta(PILOT_HORIZON, unit="D")
    )

    # Keep training and validation periods only
    train_mask = (
        (df["target_date"] >= TRAIN_START)
        & (df["target_date"] <= TRAIN_END)
    )

    validation_mask = (
        (df["target_date"] >= VALIDATION_START)
        & (df["target_date"] <= VALIDATION_END)
    )

    df = df.loc[train_mask | validation_mask].copy()

    # Remove rows with missing predictors or target
    df = df.dropna(
        subset=predictor_columns + [PILOT_TARGET_COLUMN]
    )

    # Sample a maximum of 5,000 rows per batch
    if len(df) > 5000:
        df = df.sample(
            n=5000,
            random_state=42
        )

    pilot_frames.append(df)

# Combine the pilot samples
pilot_df = pd.concat(
    pilot_frames,
    ignore_index=True
)

# Separate training and validation by target date
pilot_train = pilot_df[
    (pilot_df["target_date"] >= TRAIN_START)
    & (pilot_df["target_date"] <= TRAIN_END)
].copy()

pilot_validation = pilot_df[
    (pilot_df["target_date"] >= VALIDATION_START)
    & (pilot_df["target_date"] <= VALIDATION_END)
].copy()

# Prepare predictor and target arrays
X_train_pilot = pilot_train[predictor_columns]
y_train_pilot = pilot_train[PILOT_TARGET_COLUMN]

X_validation_pilot = pilot_validation[predictor_columns]
y_validation_pilot = pilot_validation[PILOT_TARGET_COLUMN]

# Check that the pilot is usable
assert len(X_train_pilot) > 0
assert len(X_validation_pilot) > 0

assert X_train_pilot.notna().all().all()
assert X_validation_pilot.notna().all().all()

assert (y_train_pilot >= 0).all()
assert (y_validation_pilot >= 0).all()

# Estimate memory usage
train_memory_mb = (
    X_train_pilot.memory_usage(deep=True).sum()
    + y_train_pilot.memory_usage(deep=True)
) / (1024 ** 2)

validation_memory_mb = (
    X_validation_pilot.memory_usage(deep=True).sum()
    + y_validation_pilot.memory_usage(deep=True)
) / (1024 ** 2)

# Wet-day distribution in the pilot
train_wet_percentage = (
    (y_train_pilot >= WET_DAY_THRESHOLD).mean() * 100
)

validation_wet_percentage = (
    (y_validation_pilot >= WET_DAY_THRESHOLD).mean() * 100
)

print("RAINFALL TRAINING PILOT: PREPARED")
print("Selected batches:", len(PILOT_BATCH_PATHS))
print("Predictors:", len(predictor_columns))
print("Forecast horizon: T+1")

print("\nTraining samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))

print("\nTraining wet-day percentage:", round(train_wet_percentage, 2))
print("Validation wet-day percentage:", round(validation_wet_percentage, 2))

print("\nTraining memory (MB):", round(train_memory_mb, 2))
print("Validation memory (MB):", round(validation_memory_mb, 2))

print("\nMissing training values:", int(X_train_pilot.isna().sum().sum()))
print("Missing validation values:", int(X_validation_pilot.isna().sum().sum()))

print("\nFinal holdout accessed: NO")
print("Pilot performance evaluation: NOT PERFORMED")


RAINFALL TRAINING PILOT: PREPARED
Selected batches: 10
Predictors: 53
Forecast horizon: T+1

Training samples: 34930
Validation samples: 15070

Training wet-day percentage: 38.17
Validation wet-day percentage: 39.83

Training memory (MB): 14.52
Validation memory (MB): 6.27

Missing training values: 0
Missing validation values: 0

Final holdout accessed: NO
Pilot performance evaluation: NOT PERFORMED


In [6]:

# WI-05 — Cell 6: Train and Evaluate Ridge Regression (Pilot)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import json
import numpy as np

# Create a pipeline:
# 1. Standardize input features
# 2. Train Ridge Regression
ridge_pilot = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

# Train using training data only
ridge_pilot.fit(
    X_train_pilot,
    y_train_pilot
)

# Predict rainfall on validation data
ridge_predictions = ridge_pilot.predict(
    X_validation_pilot
)

# Rainfall cannot be negative.
# Keep raw predictions for diagnostics and evaluate clipped predictions
# as the physically constrained version.
ridge_predictions_raw = ridge_predictions.copy()
ridge_predictions_clipped = np.maximum(
    ridge_predictions_raw, 0
)

# Calculate evaluation metrics
ridge_mae = mean_absolute_error(
    y_validation_pilot,
    ridge_predictions_clipped
)

ridge_rmse = np.sqrt(
    mean_squared_error(
        y_validation_pilot,
        ridge_predictions_clipped
    )
)

negative_prediction_count = int(
    (ridge_predictions_raw < 0).sum()
)

# Save the pilot model
MODEL_DIR = REPORT_DIR / "pilot_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ridge_model_path = MODEL_DIR / "WI_05_ridge_t1_pilot.joblib"
joblib.dump(ridge_pilot, ridge_model_path)

# Save the pilot evaluation metrics
ridge_pilot_metrics = {
    "model": "Ridge Regression",
    "target": TARGET_VARIABLE,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "predictor_count": int(len(predictor_columns)),
    "alpha": 1.0,
    "MAE": float(ridge_mae),
    "RMSE": float(ridge_rmse),
    "negative_raw_predictions": negative_prediction_count,
    "negative_predictions_clipped_for_evaluation": True,
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

metrics_path = REPORT_DIR / "WI_05_ridge_t1_pilot_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(ridge_pilot_metrics, file, indent=2)

# Display results
print("RIDGE REGRESSION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("MAE:", round(ridge_mae, 4), "mm/day")
print("RMSE:", round(ridge_rmse, 4), "mm/day")
print("Negative raw predictions:", negative_prediction_count)
print("Model saved:", ridge_model_path)
print("Metrics saved:", metrics_path)
print("Final holdout accessed: NO")


RIDGE REGRESSION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
MAE: 2.7722 mm/day
RMSE: 7.9462 mm/day
Negative raw predictions: 2170
Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_ridge_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_ridge_t1_pilot_metrics.json
Final holdout accessed: NO


In [7]:

# WI-05 — Cell 7: Train and Evaluate Random Forest (Pilot)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import json
import joblib

# Create the Random Forest model
rf_pilot = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train using the pilot training data
rf_pilot.fit(
    X_train_pilot,
    y_train_pilot
)

# Predict rainfall on pilot validation data
rf_predictions_raw = rf_pilot.predict(
    X_validation_pilot
)

# Apply non-negative rainfall constraint
rf_predictions_clipped = np.maximum(
    rf_predictions_raw, 0
)

# Calculate validation metrics
rf_mae = mean_absolute_error(
    y_validation_pilot,
    rf_predictions_clipped
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_validation_pilot,
        rf_predictions_clipped
    )
)

negative_prediction_count = int(
    (rf_predictions_raw < 0).sum()
)

# Save the pilot model
rf_model_path = MODEL_DIR / "WI_05_random_forest_t1_pilot.joblib"
joblib.dump(rf_pilot, rf_model_path)

# Save evaluation metrics
rf_pilot_metrics = {
    "model": "Random Forest Regressor",
    "target": TARGET_VARIABLE,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "predictor_count": int(len(predictor_columns)),
    "n_estimators": 100,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "MAE": float(rf_mae),
    "RMSE": float(rf_rmse),
    "negative_raw_predictions": negative_prediction_count,
    "negative_predictions_clipped_for_evaluation": True,
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

rf_metrics_path = (
    REPORT_DIR / "WI_05_random_forest_t1_pilot_metrics.json"
)

with open(rf_metrics_path, "w", encoding="utf-8") as file:
    json.dump(rf_pilot_metrics, file, indent=2)

# Display results
print("RANDOM FOREST REGRESSION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("MAE:", round(rf_mae, 4), "mm/day")
print("RMSE:", round(rf_rmse, 4), "mm/day")
print("Negative raw predictions:", negative_prediction_count)
print("Model saved:", rf_model_path)
print("Metrics saved:", rf_metrics_path)
print("Final holdout accessed: NO")


RANDOM FOREST REGRESSION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
MAE: 2.6737 mm/day
RMSE: 8.0137 mm/day
Negative raw predictions: 0
Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_random_forest_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_random_forest_t1_pilot_metrics.json
Final holdout accessed: NO


In [8]:

# WI-05 — Cell 8: Train HistGradientBoosting Regressor (Pilot)

from sklearn.ensemble import HistGradientBoostingRegressor

# Create the HistGradientBoosting model
hgb_pilot = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train using pilot training data
hgb_pilot.fit(
    X_train_pilot,
    y_train_pilot
)

# Predict rainfall on validation data
hgb_predictions_raw = hgb_pilot.predict(
    X_validation_pilot
)

# Apply non-negative rainfall constraint
hgb_predictions_clipped = np.maximum(
    hgb_predictions_raw, 0
)

# Calculate validation metrics
hgb_mae = mean_absolute_error(
    y_validation_pilot,
    hgb_predictions_clipped
)

hgb_rmse = np.sqrt(
    mean_squared_error(
        y_validation_pilot,
        hgb_predictions_clipped
    )
)

negative_prediction_count = int(
    (hgb_predictions_raw < 0).sum()
)

# Save the trained pilot model
hgb_model_path = MODEL_DIR / "WI_05_hist_gradient_boosting_t1_pilot.joblib"
joblib.dump(hgb_pilot, hgb_model_path)

# Save pilot evaluation metrics
hgb_pilot_metrics = {
    "model": "HistGradientBoosting Regressor",
    "target": TARGET_VARIABLE,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "predictor_count": int(len(predictor_columns)),
    "max_iter": 100,
    "learning_rate": 0.1,
    "max_leaf_nodes": 31,
    "l2_regularization": 1.0,
    "MAE": float(hgb_mae),
    "RMSE": float(hgb_rmse),
    "negative_raw_predictions": negative_prediction_count,
    "negative_predictions_clipped_for_evaluation": True,
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

hgb_metrics_path = (
    REPORT_DIR / "WI_05_hist_gradient_boosting_t1_pilot_metrics.json"
)

with open(hgb_metrics_path, "w", encoding="utf-8") as file:
    json.dump(hgb_pilot_metrics, file, indent=2)

print("HISTGRADIENTBOOSTING REGRESSION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("MAE:", round(hgb_mae, 4), "mm/day")
print("RMSE:", round(hgb_rmse, 4), "mm/day")
print("Negative raw predictions:", negative_prediction_count)
print("Model saved:", hgb_model_path)
print("Metrics saved:", hgb_metrics_path)
print("Final holdout accessed: NO")


HISTGRADIENTBOOSTING REGRESSION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
MAE: 2.7044 mm/day
RMSE: 7.9588 mm/day
Negative raw predictions: 0
Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_hist_gradient_boosting_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_hist_gradient_boosting_t1_pilot_metrics.json
Final holdout accessed: NO


In [9]:

# WI-05 — Cell 9: Compare Direct Regression Pilot Models

# Collect metrics from the three trained models
pilot_comparison = pd.DataFrame([
    {
        "model": "Ridge Regression",
        "MAE": ridge_mae,
        "RMSE": ridge_rmse,
        "negative_predictions": int(
            (ridge_predictions_raw < 0).sum()
        )
    },
    {
        "model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "negative_predictions": int(
            (rf_predictions_raw < 0).sum()
        )
    },
    {
        "model": "HistGradientBoosting",
        "MAE": hgb_mae,
        "RMSE": hgb_rmse,
        "negative_predictions": int(
            (hgb_predictions_raw < 0).sum()
        )
    }
])

# Rank models by MAE
pilot_comparison = pilot_comparison.sort_values(
    by="MAE"
).reset_index(drop=True)

# Save comparison results
comparison_csv = (
    REPORT_DIR / "WI_05_direct_regression_pilot_comparison.csv"
)

comparison_json = (
    REPORT_DIR / "WI_05_direct_regression_pilot_comparison.json"
)

pilot_comparison.to_csv(comparison_csv, index=False)

with open(comparison_json, "w", encoding="utf-8") as file:
    json.dump(
        pilot_comparison.to_dict(orient="records"),
        file,
        indent=2
    )

print("DIRECT REGRESSION PILOT COMPARISON: COMPLETE")
display(pilot_comparison.round(4))

print("\nCurrent MAE leader:", pilot_comparison.iloc[0]["model"])
print("Saved CSV:", comparison_csv)
print("Saved JSON:", comparison_json)
print("Final holdout accessed: NO")


DIRECT REGRESSION PILOT COMPARISON: COMPLETE


,model,MAE,RMSE,negative_predictions
0,Random Forest,2.6737,8.0137,0
1,HistGradientBoosting,2.7044,7.9588,0
2,Ridge Regression,2.7722,7.9462,2170



Current MAE leader: Random Forest
Saved CSV: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_direct_regression_pilot_comparison.csv
Saved JSON: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_direct_regression_pilot_comparison.json
Final holdout accessed: NO


In [10]:

# WI-05 — Cell 10: Train Logistic Regression Classifier (Pilot)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Convert rainfall into binary wet-day labels
y_train_wet = (
    y_train_pilot >= WET_DAY_THRESHOLD
).astype(int)

y_validation_wet = (
    y_validation_pilot >= WET_DAY_THRESHOLD
).astype(int)

# Create a pipeline with feature scaling and Logistic Regression
logistic_pilot = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=42
    ))
])

# Train using training data only
logistic_pilot.fit(
    X_train_pilot,
    y_train_wet
)

# Predict wet-day probabilities
logistic_probabilities = logistic_pilot.predict_proba(
    X_validation_pilot
)[:, 1]

# Initial classification threshold
CLASSIFICATION_THRESHOLD = 0.5

logistic_predictions = (
    logistic_probabilities >= CLASSIFICATION_THRESHOLD
).astype(int)

# Calculate classification metrics
logistic_precision = precision_score(
    y_validation_wet,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_validation_wet,
    logistic_predictions,
    zero_division=0
)

logistic_f1 = f1_score(
    y_validation_wet,
    logistic_predictions,
    zero_division=0
)

logistic_pr_auc = average_precision_score(
    y_validation_wet,
    logistic_probabilities
)

logistic_cm = confusion_matrix(
    y_validation_wet,
    logistic_predictions
)

# Save the trained pilot model
logistic_model_path = (
    MODEL_DIR / "WI_05_logistic_classifier_t1_pilot.joblib"
)

joblib.dump(logistic_pilot, logistic_model_path)

# Save pilot metrics
logistic_pilot_metrics = {
    "model": "Logistic Regression",
    "target": "Wet-day classification",
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "precision": float(logistic_precision),
    "recall": float(logistic_recall),
    "f1_score": float(logistic_f1),
    "pr_auc": float(logistic_pr_auc),
    "confusion_matrix": logistic_cm.tolist(),
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

logistic_metrics_path = (
    REPORT_DIR / "WI_05_logistic_t1_pilot_metrics.json"
)

with open(logistic_metrics_path, "w", encoding="utf-8") as file:
    json.dump(logistic_pilot_metrics, file, indent=2)

# Display results
print("LOGISTIC REGRESSION CLASSIFICATION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("Classification threshold:", CLASSIFICATION_THRESHOLD)
print("Precision:", round(logistic_precision, 4))
print("Recall:", round(logistic_recall, 4))
print("F1-score:", round(logistic_f1, 4))
print("PR-AUC:", round(logistic_pr_auc, 4))

print("\nConfusion matrix:")
print(logistic_cm)
print("Rows: actual [dry, wet]")
print("Columns: predicted [dry, wet]")

print("\nModel saved:", logistic_model_path)
print("Metrics saved:", logistic_metrics_path)
print("Final holdout accessed: NO")


LOGISTIC REGRESSION CLASSIFICATION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
Classification threshold: 0.5
Precision: 0.8357
Recall: 0.7195
F1-score: 0.7733
PR-AUC: 0.8794

Confusion matrix:
[[8218  849]
 [1684 4319]]
Rows: actual [dry, wet]
Columns: predicted [dry, wet]

Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_logistic_classifier_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_logistic_t1_pilot_metrics.json
Final holdout accessed: NO


In [11]:
# WI-05 — Cell 11: Train Random Forest Classifier (Pilot)

from sklearn.ensemble import RandomForestClassifier

# Convert rainfall into binary wet-day labels
y_train_wet = (
    y_train_pilot >= WET_DAY_THRESHOLD
).astype(int)

y_validation_wet = (
    y_validation_pilot >= WET_DAY_THRESHOLD
).astype(int)

# Create Random Forest classifier
rf_classifier_pilot = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

# Train using training data only
rf_classifier_pilot.fit(
    X_train_pilot,
    y_train_wet
)

# Predict wet-day probabilities
rf_probabilities = rf_classifier_pilot.predict_proba(
    X_validation_pilot
)[:, 1]

# Use the initial classification threshold
CLASSIFICATION_THRESHOLD = 0.5

rf_predictions = (
    rf_probabilities >= CLASSIFICATION_THRESHOLD
).astype(int)

# Calculate classification metrics
rf_precision = precision_score(
    y_validation_wet,
    rf_predictions,
    zero_division=0
)

rf_recall = recall_score(
    y_validation_wet,
    rf_predictions,
    zero_division=0
)

rf_f1 = f1_score(
    y_validation_wet,
    rf_predictions,
    zero_division=0
)

rf_pr_auc = average_precision_score(
    y_validation_wet,
    rf_probabilities
)

rf_cm = confusion_matrix(
    y_validation_wet,
    rf_predictions
)

# Save the trained pilot model
rf_classifier_path = (
    MODEL_DIR / "WI_05_random_forest_classifier_t1_pilot.joblib"
)

joblib.dump(
    rf_classifier_pilot,
    rf_classifier_path
)

# Save pilot metrics
rf_classifier_metrics = {
    "model": "Random Forest Classifier",
    "target": "Wet-day classification",
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "precision": float(rf_precision),
    "recall": float(rf_recall),
    "f1_score": float(rf_f1),
    "pr_auc": float(rf_pr_auc),
    "confusion_matrix": rf_cm.tolist(),
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

rf_classifier_metrics_path = (
    REPORT_DIR / "WI_05_random_forest_classifier_t1_pilot_metrics.json"
)

with open(rf_classifier_metrics_path, "w", encoding="utf-8") as file:
    json.dump(rf_classifier_metrics, file, indent=2)

# Display results
print("RANDOM FOREST CLASSIFICATION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("Classification threshold:", CLASSIFICATION_THRESHOLD)
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1-score:", round(rf_f1, 4))
print("PR-AUC:", round(rf_pr_auc, 4))

print("\nConfusion matrix:")
print(rf_cm)
print("Rows: actual [dry, wet]")
print("Columns: predicted [dry, wet]")

print("\nModel saved:", rf_classifier_path)
print("Metrics saved:", rf_classifier_metrics_path)
print("Final holdout accessed: NO")

RANDOM FOREST CLASSIFICATION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
Classification threshold: 0.5
Precision: 0.8281
Recall: 0.8028
F1-score: 0.8153
PR-AUC: 0.8898

Confusion matrix:
[[8067 1000]
 [1184 4819]]
Rows: actual [dry, wet]
Columns: predicted [dry, wet]

Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_random_forest_classifier_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_random_forest_classifier_t1_pilot_metrics.json
Final holdout accessed: NO


In [12]:
# WI-05 — Cell 12: Train HistGradientBoosting Classifier (Pilot)

from sklearn.ensemble import HistGradientBoostingClassifier

# Convert rainfall into binary wet-day labels
y_train_wet = (
    y_train_pilot >= WET_DAY_THRESHOLD
).astype(int)

y_validation_wet = (
    y_validation_pilot >= WET_DAY_THRESHOLD
).astype(int)

# Create HistGradientBoosting classifier
hgb_classifier_pilot = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train using training data only
hgb_classifier_pilot.fit(
    X_train_pilot,
    y_train_wet
)

# Predict wet-day probabilities
hgb_probabilities = hgb_classifier_pilot.predict_proba(
    X_validation_pilot
)[:, 1]

# Use the initial classification threshold
CLASSIFICATION_THRESHOLD = 0.5

hgb_predictions = (
    hgb_probabilities >= CLASSIFICATION_THRESHOLD
).astype(int)

# Calculate classification metrics
hgb_precision = precision_score(
    y_validation_wet,
    hgb_predictions,
    zero_division=0
)

hgb_recall = recall_score(
    y_validation_wet,
    hgb_predictions,
    zero_division=0
)

hgb_f1 = f1_score(
    y_validation_wet,
    hgb_predictions,
    zero_division=0
)

hgb_pr_auc = average_precision_score(
    y_validation_wet,
    hgb_probabilities
)

hgb_cm = confusion_matrix(
    y_validation_wet,
    hgb_predictions
)

# Save the trained pilot model
hgb_classifier_path = (
    MODEL_DIR / "WI_05_hist_gradient_boosting_classifier_t1_pilot.joblib"
)

joblib.dump(
    hgb_classifier_pilot,
    hgb_classifier_path
)

# Save pilot metrics
hgb_classifier_metrics = {
    "model": "HistGradientBoosting Classifier",
    "target": "Wet-day classification",
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "horizon": "T+1",
    "training_samples": int(len(X_train_pilot)),
    "validation_samples": int(len(X_validation_pilot)),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "precision": float(hgb_precision),
    "recall": float(hgb_recall),
    "f1_score": float(hgb_f1),
    "pr_auc": float(hgb_pr_auc),
    "confusion_matrix": hgb_cm.tolist(),
    "training_scope": "10-batch pilot",
    "final_holdout_accessed": False
}

hgb_classifier_metrics_path = (
    REPORT_DIR / "WI_05_hist_gradient_boosting_classifier_t1_pilot_metrics.json"
)

with open(hgb_classifier_metrics_path, "w", encoding="utf-8") as file:
    json.dump(hgb_classifier_metrics, file, indent=2)

# Display results
print("HISTGRADIENTBOOSTING CLASSIFICATION PILOT: COMPLETE")
print("Training samples:", len(X_train_pilot))
print("Validation samples:", len(X_validation_pilot))
print("Classification threshold:", CLASSIFICATION_THRESHOLD)
print("Precision:", round(hgb_precision, 4))
print("Recall:", round(hgb_recall, 4))
print("F1-score:", round(hgb_f1, 4))
print("PR-AUC:", round(hgb_pr_auc, 4))

print("\nConfusion matrix:")
print(hgb_cm)
print("Rows: actual [dry, wet]")
print("Columns: predicted [dry, wet]")

print("\nModel saved:", hgb_classifier_path)
print("Metrics saved:", hgb_classifier_metrics_path)
print("Final holdout accessed: NO")

HISTGRADIENTBOOSTING CLASSIFICATION PILOT: COMPLETE
Training samples: 34930
Validation samples: 15070
Classification threshold: 0.5
Precision: 0.8305
Recall: 0.8064
F1-score: 0.8183
PR-AUC: 0.8952

Confusion matrix:
[[8079  988]
 [1162 4841]]
Rows: actual [dry, wet]
Columns: predicted [dry, wet]

Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_hist_gradient_boosting_classifier_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_hist_gradient_boosting_classifier_t1_pilot_metrics.json
Final holdout accessed: NO


In [13]:
# WI-05 — Cell 13: Compare Wet-Day Classifier Pilots

classifier_pilot_results = [
    {
        "model": "Logistic Regression",
        "precision": logistic_precision,
        "recall": logistic_recall,
        "f1_score": logistic_f1,
        "pr_auc": logistic_pr_auc,
        "false_positives": int(logistic_cm[0, 1]),
        "false_negatives": int(logistic_cm[1, 0])
    },
    {
        "model": "Random Forest",
        "precision": rf_precision,
        "recall": rf_recall,
        "f1_score": rf_f1,
        "pr_auc": rf_pr_auc,
        "false_positives": int(rf_cm[0, 1]),
        "false_negatives": int(rf_cm[1, 0])
    },
    {
        "model": "HistGradientBoosting",
        "precision": hgb_precision,
        "recall": hgb_recall,
        "f1_score": hgb_f1,
        "pr_auc": hgb_pr_auc,
        "false_positives": int(hgb_cm[0, 1]),
        "false_negatives": int(hgb_cm[1, 0])
    }
]

# Convert results into a DataFrame
classifier_comparison_df = pd.DataFrame(
    classifier_pilot_results
)

# Sort by F1-score, highest first
classifier_comparison_df = classifier_comparison_df.sort_values(
    by="f1_score",
    ascending=False
).reset_index(drop=True)

# Save comparison results
classifier_comparison_csv = (
    REPORT_DIR / "WI_05_classifier_pilot_comparison.csv"
)

classifier_comparison_json = (
    REPORT_DIR / "WI_05_classifier_pilot_comparison.json"
)

classifier_comparison_df.to_csv(
    classifier_comparison_csv,
    index=False
)

with open(classifier_comparison_json, "w", encoding="utf-8") as file:
    json.dump(
        classifier_pilot_results,
        file,
        indent=2
    )

# Display comparison
print("WET-DAY CLASSIFIER PILOT COMPARISON")
print(classifier_comparison_df.round(4).to_string(index=False))

print("\nBest pilot model by F1-score:")
print(classifier_comparison_df.iloc[0]["model"])

print("\nComparison CSV saved:", classifier_comparison_csv)
print("Comparison JSON saved:", classifier_comparison_json)

print("\nScope: 10-batch pilot, T+1 only")
print("Final holdout accessed: NO")

WET-DAY CLASSIFIER PILOT COMPARISON
               model  precision  recall  f1_score  pr_auc  false_positives  false_negatives
HistGradientBoosting     0.8305  0.8064    0.8183  0.8952              988             1162
       Random Forest     0.8281  0.8028    0.8153  0.8898             1000             1184
 Logistic Regression     0.8357  0.7195    0.7733  0.8794              849             1684

Best pilot model by F1-score:
HistGradientBoosting

Comparison CSV saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_classifier_pilot_comparison.csv
Comparison JSON saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_classifier_pilot_comparison.json

Scope: 10-batch pilot, T+1 only
Final holdout accessed: NO


In [14]:
# WI-05 — Cell 14: Classification Threshold Analysis

# Define thresholds to evaluate
thresholds = [
    0.20, 0.25, 0.30, 0.35, 0.40,
    0.45, 0.50, 0.55, 0.60, 0.65,
    0.70, 0.75, 0.80
]

threshold_results = []

# Evaluate each threshold
for threshold in thresholds:

    # Convert probabilities into wet/dry predictions
    predictions = (
        hgb_probabilities >= threshold
    ).astype(int)

    # Calculate metrics
    precision = precision_score(
        y_validation_wet,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation_wet,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation_wet,
        predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        y_validation_wet,
        predictions
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0])
    })

# Create comparison DataFrame
threshold_comparison_df = pd.DataFrame(
    threshold_results
)

# Save threshold analysis
threshold_csv_path = (
    REPORT_DIR / "WI_05_hgb_classifier_threshold_analysis.csv"
)

threshold_json_path = (
    REPORT_DIR / "WI_05_hgb_classifier_threshold_analysis.json"
)

threshold_comparison_df.to_csv(
    threshold_csv_path,
    index=False
)

with open(threshold_json_path, "w", encoding="utf-8") as file:
    json.dump(
        threshold_results,
        file,
        indent=2
    )

# Display results
print("HISTGRADIENTBOOSTING THRESHOLD ANALYSIS")
print(
    threshold_comparison_df.round(4).to_string(index=False)
)

# Identify the threshold with the highest F1-score
best_f1_row = threshold_comparison_df.loc[
    threshold_comparison_df["f1_score"].idxmax()
]

print("\nBest threshold by F1-score:")
print(best_f1_row.to_dict())

print("\nThreshold CSV saved:", threshold_csv_path)
print("Threshold JSON saved:", threshold_json_path)

print("\nScope: 10-batch pilot, T+1 only")
print("Final holdout accessed: NO")

HISTGRADIENTBOOSTING THRESHOLD ANALYSIS
 threshold  precision  recall  f1_score  false_positives  false_negatives
      0.20     0.6995  0.9347    0.8002             2410              392
      0.25     0.7271  0.9207    0.8126             2074              476
      0.30     0.7516  0.8992    0.8188             1784              605
      0.35     0.7718  0.8771    0.8211             1557              738
      0.40     0.7931  0.8556    0.8231             1340              867
      0.45     0.8113  0.8321    0.8215             1162             1008
      0.50     0.8305  0.8064    0.8183              988             1162
      0.55     0.8426  0.7788    0.8095              873             1328
      0.60     0.8573  0.7455    0.7975              745             1528
      0.65     0.8702  0.7058    0.7794              632             1766
      0.70     0.8889  0.6610    0.7582              496             2035
      0.75     0.9001  0.6152    0.7309              410             231

In [15]:
# WI-05 — Cell 15: Prepare Wet-Day Data for Conditional Regression

# Identify wet-day observations in the training and validation sets
train_wet_mask = (
    y_train_pilot >= WET_DAY_THRESHOLD
)

validation_wet_mask = (
    y_validation_pilot >= WET_DAY_THRESHOLD
)

# Keep only wet-day samples for conditional regression training
X_train_wet = X_train_pilot.loc[train_wet_mask]
y_train_wet_amount = y_train_pilot.loc[train_wet_mask]

# Keep actual wet-day samples for conditional evaluation
X_validation_wet = X_validation_pilot.loc[validation_wet_mask]
y_validation_wet_amount = y_validation_pilot.loc[validation_wet_mask]

# Display sample counts
print("CONDITIONAL REGRESSION DATA PREPARATION")
print("Training samples before filtering:", len(X_train_pilot))
print("Training wet-day samples:", len(X_train_wet))
print("Training wet-day percentage:", round(
    len(X_train_wet) / len(X_train_pilot) * 100, 2
))

print("\nValidation samples before filtering:", len(X_validation_pilot))
print("Validation wet-day samples:", len(X_validation_wet))
print("Validation wet-day percentage:", round(
    len(X_validation_wet) / len(X_validation_pilot) * 100, 2
))

print("\nTraining target mean:", round(
    float(y_train_wet_amount.mean()), 4
), "mm/day")

print("Training target median:", round(
    float(y_train_wet_amount.median()), 4
), "mm/day")

print("\nValidation target mean:", round(
    float(y_validation_wet_amount.mean()), 4
), "mm/day")

print("Final holdout accessed: NO")

CONDITIONAL REGRESSION DATA PREPARATION
Training samples before filtering: 34930
Training wet-day samples: 13334
Training wet-day percentage: 38.17

Validation samples before filtering: 15070
Validation wet-day samples: 6003
Validation wet-day percentage: 39.83

Training target mean: 8.7766 mm/day
Training target median: 4.56 mm/day

Validation target mean: 8.3108 mm/day
Final holdout accessed: NO


In [16]:
# WI-05 — Cell 16: Train Conditional HistGradientBoosting Regressor

# Create the conditional rainfall regression model
conditional_hgb_pilot = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train only on wet-day observations
conditional_hgb_pilot.fit(
    X_train_wet,
    y_train_wet_amount
)

# Predict rainfall amount for actual wet validation days
conditional_predictions = conditional_hgb_pilot.predict(
    X_validation_wet
)

# Rainfall cannot be negative, so clip negative predictions
conditional_predictions = np.maximum(
    conditional_predictions,
    0
)

# Calculate conditional regression metrics
conditional_mae = mean_absolute_error(
    y_validation_wet_amount,
    conditional_predictions
)

conditional_rmse = np.sqrt(
    mean_squared_error(
        y_validation_wet_amount,
        conditional_predictions
    )
)

# Calculate prediction statistics
conditional_prediction_mean = float(
    np.mean(conditional_predictions)
)

conditional_prediction_median = float(
    np.median(conditional_predictions)
)

# Save the trained model
conditional_model_path = (
    MODEL_DIR / "WI_05_conditional_hgb_t1_pilot.joblib"
)

joblib.dump(
    conditional_hgb_pilot,
    conditional_model_path
)

# Save conditional regression metrics
conditional_metrics = {
    "model": "HistGradientBoosting Regressor",
    "model_type": "Conditional wet-day rainfall regression",
    "target": "PRECTOTCORR",
    "horizon": "T+1",
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "training_samples": int(len(X_train_wet)),
    "validation_samples": int(len(X_validation_wet)),
    "mae_mm": float(conditional_mae),
    "rmse_mm": float(conditional_rmse),
    "actual_wet_day_mean_mm": float(
        y_validation_wet_amount.mean()
    ),
    "predicted_wet_day_mean_mm": conditional_prediction_mean,
    "predicted_wet_day_median_mm": conditional_prediction_median,
    "training_scope": "10-batch pilot",
    "evaluation_scope": "Actual wet validation days only",
    "final_holdout_accessed": False
}

conditional_metrics_path = (
    REPORT_DIR / "WI_05_conditional_hgb_t1_pilot_metrics.json"
)

with open(conditional_metrics_path, "w", encoding="utf-8") as file:
    json.dump(
        conditional_metrics,
        file,
        indent=2
    )

# Display results
print("CONDITIONAL HISTGRADIENTBOOSTING REGRESSION PILOT: COMPLETE")
print("Training wet-day samples:", len(X_train_wet))
print("Validation wet-day samples:", len(X_validation_wet))

print("\nConditional rainfall MAE:", round(conditional_mae, 4), "mm/day")
print("Conditional rainfall RMSE:", round(conditional_rmse, 4), "mm/day")

print("\nActual wet-day mean:", round(
    float(y_validation_wet_amount.mean()), 4
), "mm/day")

print("Predicted wet-day mean:", round(
    conditional_prediction_mean, 4
), "mm/day")

print("\nModel saved:", conditional_model_path)
print("Metrics saved:", conditional_metrics_path)
print("Final holdout accessed: NO")

CONDITIONAL HISTGRADIENTBOOSTING REGRESSION PILOT: COMPLETE
Training wet-day samples: 13334
Validation wet-day samples: 6003

Conditional rainfall MAE: 5.6016 mm/day
Conditional rainfall RMSE: 12.3768 mm/day

Actual wet-day mean: 8.3108 mm/day
Predicted wet-day mean: 8.236 mm/day

Model saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\pilot_models\WI_05_conditional_hgb_t1_pilot.joblib
Metrics saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_conditional_hgb_t1_pilot_metrics.json
Final holdout accessed: NO


In [17]:
# WI-05 — Cell 17: Evaluate Complete Two-Stage Rainfall Model

# Get wet-day probabilities for every validation observation
two_stage_probabilities = hgb_classifier_pilot.predict_proba(
    X_validation_pilot
)[:, 1]

# Predict conditional rainfall amount for every validation observation
two_stage_amount_predictions = conditional_hgb_pilot.predict(
    X_validation_pilot
)

# Rainfall predictions cannot be negative
two_stage_amount_predictions = np.maximum(
    two_stage_amount_predictions,
    0
)

# Approach 1: Expected rainfall amount
expected_rainfall_predictions = (
    two_stage_probabilities * two_stage_amount_predictions
)

# Approach 2: Hard threshold rainfall prediction
PILOT_SELECTED_THRESHOLD = 0.40

hard_threshold_predictions = np.where(
    two_stage_probabilities >= PILOT_SELECTED_THRESHOLD,
    two_stage_amount_predictions,
    0
)

# Calculate actual wet/dry labels
actual_wet_labels = (
    y_validation_pilot >= WET_DAY_THRESHOLD
).astype(int)

# Calculate predicted wet/dry labels using the selected threshold
predicted_wet_labels = (
    two_stage_probabilities >= PILOT_SELECTED_THRESHOLD
).astype(int)

# Evaluate expected rainfall predictions
expected_mae = mean_absolute_error(
    y_validation_pilot,
    expected_rainfall_predictions
)

expected_rmse = np.sqrt(
    mean_squared_error(
        y_validation_pilot,
        expected_rainfall_predictions
    )
)

# Evaluate hard threshold rainfall predictions
hard_mae = mean_absolute_error(
    y_validation_pilot,
    hard_threshold_predictions
)

hard_rmse = np.sqrt(
    mean_squared_error(
        y_validation_pilot,
        hard_threshold_predictions
    )
)

# Calculate wet-day classification metrics
two_stage_precision = precision_score(
    actual_wet_labels,
    predicted_wet_labels,
    zero_division=0
)

two_stage_recall = recall_score(
    actual_wet_labels,
    predicted_wet_labels,
    zero_division=0
)

two_stage_f1 = f1_score(
    actual_wet_labels,
    predicted_wet_labels,
    zero_division=0
)

two_stage_cm = confusion_matrix(
    actual_wet_labels,
    predicted_wet_labels
)

# Create comparison table
two_stage_comparison = pd.DataFrame([
    {
        "approach": "Expected rainfall amount",
        "mae_mm": expected_mae,
        "rmse_mm": expected_rmse
    },
    {
        "approach": "Hard threshold rainfall amount",
        "mae_mm": hard_mae,
        "rmse_mm": hard_rmse
    }
])

# Display results
print("COMPLETE TWO-STAGE RAINFALL MODEL EVALUATION")
print("Validation samples:", len(y_validation_pilot))

print("\nRainfall amount comparison:")
print(two_stage_comparison.round(4).to_string(index=False))

print("\nWet-day classification metrics at threshold 0.40:")
print("Precision:", round(two_stage_precision, 4))
print("Recall:", round(two_stage_recall, 4))
print("F1-score:", round(two_stage_f1, 4))

print("\nConfusion matrix:")
print(two_stage_cm)
print("Rows: actual [dry, wet]")
print("Columns: predicted [dry, wet]")

print("\nFinal holdout accessed: NO")

COMPLETE TWO-STAGE RAINFALL MODEL EVALUATION
Validation samples: 15070

Rainfall amount comparison:
                      approach  mae_mm  rmse_mm
      Expected rainfall amount  2.5950   7.9743
Hard threshold rainfall amount  2.7774   8.0503

Wet-day classification metrics at threshold 0.40:
Precision: 0.7931
Recall: 0.8556
F1-score: 0.8231

Confusion matrix:
[[7727 1340]
 [ 867 5136]]
Rows: actual [dry, wet]
Columns: predicted [dry, wet]

Final holdout accessed: NO


In [18]:
# WI-05 — Cell 18: Compare Direct and Two-Stage Regression Pilots

# Reuse the actual validation rainfall values
actual_rainfall = y_validation_pilot

# Store direct regression predictions
direct_predictions = {
    "Ridge": np.maximum(
        ridge_pilot.predict(X_validation_pilot),
        0
    ),
    "Random Forest": rf_pilot.predict(
        X_validation_pilot
    ),
    "HistGradientBoosting": hgb_pilot.predict(
        X_validation_pilot
    )
}

# Store all regression approaches
regression_comparison_results = []

# Evaluate direct regression models
for model_name, predictions in direct_predictions.items():

    regression_comparison_results.append({
        "model": model_name,
        "approach": "Direct regression",
        "mae_mm": mean_absolute_error(
            actual_rainfall,
            predictions
        ),
        "rmse_mm": np.sqrt(
            mean_squared_error(
                actual_rainfall,
                predictions
            )
        )
    })

# Evaluate expected-amount two-stage approach
regression_comparison_results.append({
    "model": "HistGradientBoosting",
    "approach": "Two-stage expected amount",
    "mae_mm": mean_absolute_error(
        actual_rainfall,
        expected_rainfall_predictions
    ),
    "rmse_mm": np.sqrt(
        mean_squared_error(
            actual_rainfall,
            expected_rainfall_predictions
        )
    )
})

# Evaluate hard-threshold two-stage approach
regression_comparison_results.append({
    "model": "HistGradientBoosting",
    "approach": "Two-stage hard threshold",
    "mae_mm": mean_absolute_error(
        actual_rainfall,
        hard_threshold_predictions
    ),
    "rmse_mm": np.sqrt(
        mean_squared_error(
            actual_rainfall,
            hard_threshold_predictions
        )
    )
})

# Create comparison DataFrame
regression_comparison_df = pd.DataFrame(
    regression_comparison_results
)

# Sort by MAE
regression_comparison_df = regression_comparison_df.sort_values(
    by="mae_mm",
    ascending=True
).reset_index(drop=True)

# Save comparison results
regression_comparison_csv = (
    REPORT_DIR / "WI_05_direct_vs_two_stage_pilot_comparison.csv"
)

regression_comparison_json = (
    REPORT_DIR / "WI_05_direct_vs_two_stage_pilot_comparison.json"
)

regression_comparison_df.to_csv(
    regression_comparison_csv,
    index=False
)

with open(regression_comparison_json, "w", encoding="utf-8") as file:
    json.dump(
        regression_comparison_results,
        file,
        indent=2
    )

# Display results
print("DIRECT VS TWO-STAGE REGRESSION PILOT COMPARISON")
print(regression_comparison_df.round(4).to_string(index=False))

print("\nBest pilot approach by MAE:")
print(regression_comparison_df.iloc[0]["approach"])
print("Model:", regression_comparison_df.iloc[0]["model"])

print("\nComparison CSV saved:", regression_comparison_csv)
print("Comparison JSON saved:", regression_comparison_json)

print("\nScope: 10-batch pilot, T+1 only")
print("Final holdout accessed: NO")

DIRECT VS TWO-STAGE REGRESSION PILOT COMPARISON
               model                  approach  mae_mm  rmse_mm
HistGradientBoosting Two-stage expected amount  2.5950   7.9743
       Random Forest         Direct regression  2.6737   8.0137
HistGradientBoosting         Direct regression  2.7044   7.9588
               Ridge         Direct regression  2.7722   7.9462
HistGradientBoosting  Two-stage hard threshold  2.7774   8.0503

Best pilot approach by MAE:
Two-stage expected amount
Model: HistGradientBoosting

Comparison CSV saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_direct_vs_two_stage_pilot_comparison.csv
Comparison JSON saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_direct_vs_two_stage_pilot_comparison.json

Scope: 10-batch pilot, T+1 only
Final holdo

In [20]:
# WI-05 — Cell 19: Save Consolidated Pilot Findings

wi05_pilot_findings = {
    "work_item": "WI-05",
    "title": "Rainfall Model Development",

    "experiment_scope": {
        "training_scope": "10-batch pilot",
        "horizon": "T+1",
        "training_samples": int(len(X_train_pilot)),
        "validation_samples": int(len(X_validation_pilot)),
        "final_holdout_accessed": False
    },

    "direct_regression": {
        "Ridge": {
            "mae_mm": float(ridge_mae),
            "rmse_mm": float(ridge_rmse)
        },
        "Random Forest": {
            "mae_mm": float(rf_mae),
            "rmse_mm": float(rf_rmse)
        },
        "HistGradientBoosting": {
            "mae_mm": float(hgb_mae),
            "rmse_mm": float(hgb_rmse)
        }
    },

    "wet_day_classification": {
        "wet_day_threshold_mm": WET_DAY_THRESHOLD,
        "models": classifier_pilot_results,
        "leading_model": "HistGradientBoosting"
    },

    "classification_threshold_analysis": {
        "candidate_model": "HistGradientBoosting",
        "pilot_candidate_threshold": 0.40,
        "precision": float(two_stage_precision),
        "recall": float(two_stage_recall),
        "f1_score": float(two_stage_f1),
        "false_positives": int(two_stage_cm[0, 1]),
        "false_negatives": int(two_stage_cm[1, 0])
    },

    "conditional_regression": {
        "model": "HistGradientBoosting Regressor",
        "training_wet_samples": int(len(X_train_wet)),
        "validation_wet_samples": int(len(X_validation_wet)),
        "mae_mm": float(conditional_mae),
        "rmse_mm": float(conditional_rmse)
    },

    "two_stage_evaluation": {
        "expected_amount": {
            "mae_mm": float(expected_mae),
            "rmse_mm": float(expected_rmse)
        },
        "hard_threshold": {
            "mae_mm": float(hard_mae),
            "rmse_mm": float(hard_rmse)
        }
    },

    "pilot_conclusion": {
        "leading_approach": "Two-stage HistGradientBoosting expected rainfall amount",
        "mae_mm": float(expected_mae),
        "rmse_mm": float(expected_rmse),
        "status": "Promising pilot candidate; not production selected",
        "limitations": [
            "Only 10 of 252 feature batches were used",
            "Only T+1 was evaluated",
            "Pilot results are not representative full-dataset validation results",
            "Classification threshold has not been operationally calibrated",
            "Final holdout has not been accessed"
        ]
    }
}

# Save consolidated findings
pilot_findings_path = (
    REPORT_DIR / "WI_05_consolidated_pilot_findings.json"
)

with open(pilot_findings_path, "w", encoding="utf-8") as file:
    json.dump(wi05_pilot_findings, file, indent=2)

print("WI-05 CONSOLIDATED PILOT FINDINGS: SAVED")
print("Leading approach:", wi05_pilot_findings["pilot_conclusion"]["leading_approach"])
print("Pilot MAE:", round(expected_mae, 4), "mm/day")
print("Pilot RMSE:", round(expected_rmse, 4), "mm/day")
print("Final holdout accessed: NO")
print("Artifact:", pilot_findings_path)

WI-05 CONSOLIDATED PILOT FINDINGS: SAVED
Leading approach: Two-stage HistGradientBoosting expected rainfall amount
Pilot MAE: 2.595 mm/day
Pilot RMSE: 7.9743 mm/day
Final holdout accessed: NO
Artifact: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_consolidated_pilot_findings.json


In [21]:
# WI-05 — Cell 20: Inspect Feature Dataset Batches

# Find all Parquet batch files
feature_batch_files = sorted(
    FEATURE_DATASET_DIR.glob("*.parquet")
)

# Display dataset information
print("FULL VALIDATION DATASET INSPECTION")
print("----------------------------------")

print("Feature dataset directory:", FEATURE_DATASET_DIR)
print("Total Parquet batches:", len(feature_batch_files))

# Display a few sample filenames
print("\nFirst 5 batch files:")

for file_path in feature_batch_files[:5]:
    print(file_path.name)

# Inspect one batch without loading the complete dataset
sample_batch = pq.read_table(
    feature_batch_files[0]
)

print("\nSample batch information:")
print("Rows:", sample_batch.num_rows)
print("Columns:", sample_batch.num_columns)
print("Approximate size (MiB):", round(
    sample_batch.nbytes / (1024 ** 2), 2
))

print("\nColumn names:")
print(sample_batch.column_names)

print("\nFull validation dataset inspection complete.")

FULL VALIDATION DATASET INSPECTION
----------------------------------
Feature dataset directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1
Total Parquet batches: 252

First 5 batch files:
batch_0000.parquet
batch_0001.parquet
batch_0002.parquet
batch_0003.parquet
batch_0004.parquet

Sample batch information:
Rows: 43830
Columns: 69
Approximate size (MiB): 23.14

Column names:
['cell_id', 'date', 'latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'T2M_MAX_lag_1', 'T2M_MAX_lag_2', 'T2M_MAX_lag_3', 'T2M_MAX_lag_7', 'T2M_MAX_lag_14', 'T2M_MAX_lag_30', 'T2M_MIN_lag_1', 'T2M_MIN_lag_2', 'T2M_MIN_lag_3', 'T2M_MIN_lag_7', 'T2M_MIN_lag_14', 'T2M_MIN_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_m

In [22]:
# WI-05 — Cell 21: Inspect System Memory and Batch Memory

import psutil

# Check system memory
memory_info = psutil.virtual_memory()

total_ram_gb = memory_info.total / (1024 ** 3)
available_ram_gb = memory_info.available / (1024 ** 3)
used_ram_gb = memory_info.used / (1024 ** 3)

# Convert one Parquet batch to pandas
sample_batch_df = sample_batch.to_pandas()

# Calculate pandas memory usage
batch_memory_mb = (
    sample_batch_df.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

# Estimate memory for all batches if loaded together
estimated_full_memory_gb = (
    batch_memory_mb * len(feature_batch_files)
    / 1024
)

# Display results
print("SYSTEM MEMORY INSPECTION")
print("------------------------")
print("Total RAM:", round(total_ram_gb, 2), "GB")
print("Used RAM:", round(used_ram_gb, 2), "GB")
print("Available RAM:", round(available_ram_gb, 2), "GB")

print("\nBATCH MEMORY INSPECTION")
print("----------------------")
print("One batch pandas memory:", round(batch_memory_mb, 2), "MB")

print(
    "Estimated memory for all 252 batches:",
    round(estimated_full_memory_gb, 2),
    "GB"
)

print("\nTRAINING STRATEGY INPUT")
print("-----------------------")

if available_ram_gb < 4:
    print("Recommendation: Very conservative sampling required.")
elif available_ram_gb < 8:
    print("Recommendation: Controlled sampling and memory monitoring required.")
else:
    print("Recommendation: Evaluate a larger in-memory training matrix.")

print("\nNo model training performed.")
print("No dataset files modified.")

SYSTEM MEMORY INSPECTION
------------------------
Total RAM: 15.68 GB
Used RAM: 11.8 GB
Available RAM: 3.88 GB

BATCH MEMORY INSPECTION
----------------------
One batch pandas memory: 24.66 MB
Estimated memory for all 252 batches: 6.07 GB

TRAINING STRATEGY INPUT
-----------------------
Recommendation: Very conservative sampling required.

No model training performed.
No dataset files modified.


In [23]:
# WI-05 — Cell 22: Verify Sampling and Chronological Boundaries

# Read one batch
sampling_test_df = sample_batch.to_pandas()

# Convert date column to datetime
sampling_test_df["date"] = pd.to_datetime(
    sampling_test_df["date"]
)

# Calculate target dates for T+1
sampling_test_df["target_date_t1"] = (
    sampling_test_df["date"] + pd.Timedelta(days=1)
)

# Assign chronological splits using target date
sampling_test_df["split"] = np.select(
    [
        sampling_test_df["target_date_t1"].between(
            TRAIN_START, TRAIN_END
        ),
        sampling_test_df["target_date_t1"].between(
            VALIDATION_START, VALIDATION_END
        ),
        sampling_test_df["target_date_t1"].between(
            HOLDOUT_START, HOLDOUT_END
        )
    ],
    [
        "train",
        "validation",
        "holdout"
    ],
    default="excluded"
)

# Display split counts
print("CHRONOLOGICAL SAMPLING BOUNDARY CHECK")
print("-------------------------------------")

print("\nDate range:")
print("Minimum origin date:", sampling_test_df["date"].min())
print("Maximum origin date:", sampling_test_df["date"].max())

print("\nT+1 target-date split counts:")
print(
    sampling_test_df["split"].value_counts().sort_index().to_string()
)

print("\nTraining target-date range:")
print(
    sampling_test_df.loc[
        sampling_test_df["split"] == "train",
        "target_date_t1"
    ].min(),
    "to",
    sampling_test_df.loc[
        sampling_test_df["split"] == "train",
        "target_date_t1"
    ].max()
)

print("\nValidation target-date range:")
print(
    sampling_test_df.loc[
        sampling_test_df["split"] == "validation",
        "target_date_t1"
    ].min(),
    "to",
    sampling_test_df.loc[
        sampling_test_df["split"] == "validation",
        "target_date_t1"
    ].max()
)

print("\nFinal holdout accessed for model evaluation: NO")
print("Only one batch inspected.")

CHRONOLOGICAL SAMPLING BOUNDARY CHECK
-------------------------------------

Date range:
Minimum origin date: 2013-01-01 00:00:00
Maximum origin date: 2024-12-31 00:00:00

T+1 target-date split counts:
split
excluded         10
holdout        7310
train         25550
validation    10960

Training target-date range:
2013-01-02 00:00:00 to 2019-12-31 00:00:00

Validation target-date range:
2020-01-01 00:00:00 to 2022-12-31 00:00:00

Final holdout accessed for model evaluation: NO
Only one batch inspected.


C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2607923087.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  sampling_test_df["date"] + pd.Timedelta(days=1)


In [25]:
# WI-05 — Cell 23: Corrected All-Horizon Boundary Verification

boundary_check_passed = True

for index, row in horizon_boundary_df.iterrows():

    horizon_number = index + 1

    expected_train = 25550 - ((horizon_number - 1) * 10)
    expected_validation = 10960
    expected_holdout = 7310
    expected_excluded = horizon_number * 10

    checks = (
        row["train_rows"] == expected_train
        and row["validation_rows"] == expected_validation
        and row["holdout_rows"] == expected_holdout
        and row["excluded_rows"] == expected_excluded
    )

    if not checks:
        boundary_check_passed = False

    print(
        row["horizon"],
        ":",
        "PASSED" if checks else "FAILED"
    )

print("\nOverall boundary verification:",
      "PASSED" if boundary_check_passed else "FAILED")

print("Final holdout used for training: NO")

T+1 : PASSED
T+2 : PASSED
T+3 : PASSED
T+4 : PASSED
T+5 : PASSED
T+6 : PASSED
T+7 : PASSED

Overall boundary verification: PASSED
Final holdout used for training: NO


In [27]:
# WI-05 — Cell 24: Configure Full-Validation Training Sample

# Find all Parquet batches in the feature dataset
parquet_files = sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet"))

SAMPLES_PER_CELL_YEAR = 20
RANDOM_STATE = 42

print("TRAINING SAMPLE CONFIGURATION")
print("--------------------------------")
print("Feature dataset:", FEATURE_DATASET_DIR)
print("Batches found:", len(parquet_files))
print("Grid cells per batch: 10")
print("Training years: 2013–2019")
print("Samples per cell per year:", SAMPLES_PER_CELL_YEAR)
print("Random state:", RANDOM_STATE)

estimated_sample_size = (
    len(parquet_files)
    * 10
    * 7
    * SAMPLES_PER_CELL_YEAR
)

print("Estimated training samples:", estimated_sample_size)
print("Validation strategy: Full validation, batch-wise")
print("Final holdout: Excluded")

assert len(parquet_files) == 252, "Expected 252 Parquet batches."

TRAINING SAMPLE CONFIGURATION
--------------------------------
Feature dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1
Batches found: 252
Grid cells per batch: 10
Training years: 2013–2019
Samples per cell per year: 20
Random state: 42
Estimated training samples: 352800
Validation strategy: Full validation, batch-wise
Final holdout: Excluded


In [31]:
# WI-05 — Cell 25: Build T+1 Training Sample

horizon = 1
target_column = f"PRECTOTCORR_target_t_plus_{horizon}"

training_sample_parts = []

for file_path in parquet_files:

    batch_df = pq.read_table(
        file_path,
        columns=["cell_id", "date", target_column] + numerical_features
    ).to_pandas()

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.Timedelta(days=int(horizon))
    )

    # Keep only observations whose target date belongs to training
    train_df = batch_df[
        batch_df["target_date"].between(TRAIN_START, TRAIN_END)
    ].copy()

    train_df["target_year"] = train_df["target_date"].dt.year

    # Sample separately for each grid cell and year
    sampled_groups = []

    for (cell_id, year), group in train_df.groupby(
        ["cell_id", "target_year"]
    ):
        sample_count = min(SAMPLES_PER_CELL_YEAR, len(group))

        sampled_groups.append(
            group.sample(
                n=sample_count,
                random_state=RANDOM_STATE
            )
        )

    if sampled_groups:
        training_sample_parts.append(
            pd.concat(sampled_groups, ignore_index=True)
        )

    del batch_df, train_df

training_sample_df = pd.concat(
    training_sample_parts,
    ignore_index=True
)

print("T+1 TRAINING SAMPLE")
print("-------------------")
print("Rows:", len(training_sample_df))
print("Unique grid cells:", training_sample_df["cell_id"].nunique())
print("Training years:", sorted(training_sample_df["target_year"].unique()))
print("Wet-day percentage:",
      round(
          (training_sample_df[target_column] >= WET_DAY_THRESHOLD).mean() * 100,
          2
      ))
print("Missing target values:", training_sample_df[target_column].isna().sum())

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2783854188.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2783854188.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2783854188.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horiz

T+1 TRAINING SAMPLE
-------------------
Rows: 352800
Unique grid cells: 2520
Training years: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Wet-day percentage: 31.03
Missing target values: 0


In [29]:
# WI-05 — Cell 25A: Load the Feature Specification

feature_spec_path = REPORT_DIR / "WI_05_feature_specification.json"

with open(feature_spec_path, "r") as file:
    feature_spec = json.load(file)

print("Feature specification keys:")
print(feature_spec.keys())

Feature specification keys:
dict_keys(['target_variable', 'predictor_count', 'predictors', 'excluded_columns', 'target_columns_excluded', 'future_columns_excluded', 'identifier_columns_excluded', 'feature_selection', 'final_holdout_accessed'])


In [30]:
# WI-05 — Cell 25A: Recover Predictor Names

numerical_features = feature_spec["predictors"]

print("Predictor count:", len(numerical_features))
print("Predictors:", numerical_features)

assert len(numerical_features) == 53, "Expected exactly 53 predictors."

Predictor count: 53
Predictors: ['latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'T2M_MAX_lag_1', 'T2M_MAX_lag_2', 'T2M_MAX_lag_3', 'T2M_MAX_lag_7', 'T2M_MAX_lag_14', 'T2M_MAX_lag_30', 'T2M_MIN_lag_1', 'T2M_MIN_lag_2', 'T2M_MIN_lag_3', 'T2M_MIN_lag_7', 'T2M_MIN_lag_14', 'T2M_MIN_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_mean_7', 'PRECTOTCORR_rolling_std_7', 'PRECTOTCORR_rolling_mean_14', 'PRECTOTCORR_rolling_std_14', 'PRECTOTCORR_rolling_mean_30', 'PRECTOTCORR_rolling_std_30', 'T2M_MAX_rolling_mean_3', 'T2M_MAX_rolling_std_3', 'T2M_MAX_rolling_mean_7', 'T2M_MAX_rolling_std_7', 'T2M_MAX_rolling_mean_14', 'T2M_MAX_rolling_std_14', 'T2M_MAX_rolling_mean_30', 'T2M_MAX_rolling_std_30', 'T2M_MIN_rolling_mean_3', 'T2M_MIN_rolling_std_3', 'T2M_MIN_rolling_mean_7', 'T2M_MIN_rolling_std

In [32]:
# WI-05 — Cell 25B: Rebuild T+1 Training Sample

training_sample_parts = []

for batch_index, file_path in enumerate(parquet_files):

    batch_df = pq.read_table(
        file_path,
        columns=["cell_id", "date", target_column] + numerical_features
    ).to_pandas()

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.Timedelta(days=int(horizon))
    )

    train_df = batch_df[
        batch_df["target_date"].between(TRAIN_START, TRAIN_END)
    ].copy()

    train_df["target_year"] = train_df["target_date"].dt.year

    sampled_groups = []

    for group_index, (_, group) in enumerate(
        train_df.groupby(["cell_id", "target_year"])
    ):
        sample_count = min(SAMPLES_PER_CELL_YEAR, len(group))

        sampled_groups.append(
            group.sample(
                n=sample_count,
                random_state=RANDOM_STATE + batch_index * 1000 + group_index
            )
        )

    if sampled_groups:
        training_sample_parts.append(
            pd.concat(sampled_groups, ignore_index=True)
        )

    del batch_df, train_df

training_sample_df = pd.concat(
    training_sample_parts,
    ignore_index=True
)

sample_wet_rate = (
    training_sample_df[target_column] >= WET_DAY_THRESHOLD
).mean() * 100

print("CORRECTED T+1 TRAINING SAMPLE")
print("--------------------------------")
print("Rows:", len(training_sample_df))
print("Unique grid cells:", training_sample_df["cell_id"].nunique())
print("Training years:", sorted(training_sample_df["target_year"].unique()))
print("Wet-day percentage:", round(sample_wet_rate, 2))
print("Missing target values:", training_sample_df[target_column].isna().sum())

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2957328107.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2957328107.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\2957328107.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch_df["date"] + pd.Timedelta(days=int(horiz

CORRECTED T+1 TRAINING SAMPLE
--------------------------------
Rows: 352800
Unique grid cells: 2520
Training years: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Wet-day percentage: 38.46
Missing target values: 0


In [33]:
# WI-05 — Cell 26: Audit T+1 Training Sample

print("RAINFALL SAMPLE QUALITY AUDIT")
print("--------------------------------")

# Rainfall intensity distribution
intensity_bins = [-0.001, 1, 5, 20, 50, float("inf")]
intensity_labels = [
    "Below 1 mm",
    "1–5 mm",
    "5–20 mm",
    "20–50 mm",
    "50+ mm"
]

training_sample_df["intensity_band"] = pd.cut(
    training_sample_df[target_column],
    bins=intensity_bins,
    labels=intensity_labels
)

print("\nRainfall intensity distribution (%):")
print(
    training_sample_df["intensity_band"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# Seasonal representation
training_sample_df["month"] = (
    training_sample_df["target_date"].dt.month
)

training_sample_df["season"] = np.select(
    [
        training_sample_df["month"].isin([12, 1, 2]),
        training_sample_df["month"].isin([3, 4, 5]),
        training_sample_df["month"].isin([6, 7, 8, 9])
    ],
    ["Winter", "Pre-monsoon", "Monsoon"],
    default="Post-monsoon"
)

print("\nSeasonal representation (%):")
print(
    training_sample_df["season"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMaximum sampled rainfall:",
      round(training_sample_df[target_column].max(), 2), "mm/day")

RAINFALL SAMPLE QUALITY AUDIT
--------------------------------

Rainfall intensity distribution (%):
intensity_band
Below 1 mm    61.64
1–5 mm        18.89
5–20 mm       15.48
20–50 mm       3.37
50+ mm         0.61
Name: proportion, dtype: float64

Seasonal representation (%):
season
Monsoon         33.46
Pre-monsoon     25.19
Winter          24.70
Post-monsoon    16.65
Name: proportion, dtype: float64

Maximum sampled rainfall: 488.99 mm/day


In [34]:
# WI-05 — Cell 27: Full Population vs Sample Distribution

intensity_bins = [-0.001, 1, 5, 20, 50, float("inf")]
intensity_labels = [
    "Below 1 mm", "1–5 mm", "5–20 mm", "20–50 mm", "50+ mm"
]
season_order = ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"]

full_intensity_counts = pd.Series(0, index=intensity_labels, dtype="int64")
full_season_counts = pd.Series(0, index=season_order, dtype="int64")
full_rows = 0
full_max_rainfall = float("-inf")

for file_path in parquet_files:
    batch = pq.read_table(
        file_path,
        columns=["date", target_column]
    ).to_pandas()

    batch["date"] = pd.to_datetime(batch["date"])
    batch["target_date"] = (
        batch["date"] + pd.to_timedelta(int(horizon), unit="D")
    )

    batch = batch[
        batch["target_date"].between(TRAIN_START, TRAIN_END)
    ]

    rainfall = batch[target_column].dropna()
    full_rows += len(rainfall)

    if len(rainfall):
        full_max_rainfall = max(
            full_max_rainfall, rainfall.max()
        )

        bands = pd.cut(
            rainfall,
            bins=intensity_bins,
            labels=intensity_labels
        )
        full_intensity_counts = (
            full_intensity_counts.add(
                bands.value_counts().reindex(
                    intensity_labels, fill_value=0
                ),
                fill_value=0
            ).astype("int64")
        )

    months = batch.loc[rainfall.index, "target_date"].dt.month

    seasons = np.select(
        [
            months.isin([12, 1, 2]),
            months.isin([3, 4, 5]),
            months.isin([6, 7, 8, 9])
        ],
        ["Winter", "Pre-monsoon", "Monsoon"],
        default="Post-monsoon"
    )

    season_counts = pd.Series(seasons).value_counts()
    full_season_counts = (
        full_season_counts.add(
            season_counts.reindex(season_order, fill_value=0),
            fill_value=0
        ).astype("int64")
    )

    del batch

# Sample distributions
sample_intensity_pct = (
    training_sample_df["intensity_band"]
    .value_counts(normalize=True)
    .reindex(intensity_labels, fill_value=0) * 100
)

full_intensity_pct = (
    full_intensity_counts / full_rows * 100
)

sample_season_pct = (
    training_sample_df["season"]
    .value_counts(normalize=True)
    .reindex(season_order, fill_value=0) * 100
)

full_season_pct = (
    full_season_counts / full_rows * 100
)

print("FULL TRAINING POPULATION VS SAMPLE")
print("-----------------------------------")
print("Full training rows:", full_rows)
print("Sample rows:", len(training_sample_df))

print("\nRainfall intensity distribution (%)")
print(pd.DataFrame({
    "Full population": full_intensity_pct,
    "Sample": sample_intensity_pct
}).round(3))

print("\nSeasonal distribution (%)")
print(pd.DataFrame({
    "Full population": full_season_pct,
    "Sample": sample_season_pct
}).round(3))

print("\nMaximum rainfall")
print("Full population:", round(full_max_rainfall, 2), "mm/day")
print(
    "Sample:",
    round(training_sample_df[target_column].max(), 2),
    "mm/day"
)

FULL TRAINING POPULATION VS SAMPLE
-----------------------------------
Full training rows: 6438600
Sample rows: 352800

Rainfall intensity distribution (%)
            Full population  Sample
Below 1 mm           61.634  61.642
1–5 mm               19.012  18.895
5–20 mm              15.361  15.478
20–50 mm              3.383   3.375
50+ mm                0.610   0.610

Seasonal distribution (%)
              Full population  Sample
Winter                 24.658  24.702
Pre-monsoon            25.205  25.189
Monsoon                33.425  33.460
Post-monsoon           16.712  16.649

Maximum rainfall
Full population: 1038.34 mm/day
Sample: 488.99 mm/day


In [35]:
# WI-05 — Cell 28: Save Training Sample Quality Audit

sample_quality_audit = {
    "experiment": "WI-05 T+1 rainfall training sample",
    "target": target_column,
    "horizon": int(horizon),
    "sampling_method": "Random sampling within each grid-cell and target-year group",
    "full_training_rows": int(full_rows),
    "sample_rows": int(len(training_sample_df)),
    "sample_grid_cells": int(training_sample_df["cell_id"].nunique()),
    "full_max_rainfall_mm": float(full_max_rainfall),
    "sample_max_rainfall_mm": float(
        training_sample_df[target_column].max()
    ),
    "intensity_distribution_percentage": {
        label: {
            "full_population": round(float(full_intensity_pct[label]), 4),
            "sample": round(float(sample_intensity_pct[label]), 4)
        }
        for label in intensity_labels
    },
    "seasonal_distribution_percentage": {
        season: {
            "full_population": round(float(full_season_pct[season]), 4),
            "sample": round(float(sample_season_pct[season]), 4)
        }
        for season in season_order
    },
    "holdout_accessed": False,
    "limitations": [
        "The random sample does not contain the maximum observed rainfall event.",
        "Matching aggregate distributions does not guarantee representation of every extreme event.",
        "The sample is intended for model fitting; validation must use the complete validation population."
    ]
}

audit_path = REPORT_DIR / "WI_05_training_sample_quality_audit.json"

with open(audit_path, "w", encoding="utf-8") as file:
    json.dump(sample_quality_audit, file, indent=2)

print("Sample quality audit saved:")
print(audit_path)
print("Sample rows:", len(training_sample_df))
print("Sample wet-day rate:", round(sample_wet_rate, 3), "%")
print("Holdout accessed:", False)

Sample quality audit saved:
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_training_sample_quality_audit.json
Sample rows: 352800
Sample wet-day rate: 38.457 %
Holdout accessed: False


In [36]:
# WI-05 — Cell 29: Train T+1 Two-Stage HGB Candidate

# Prepare training features
X_train_t1 = training_sample_df[numerical_features].copy()

# Actual rainfall target
y_train_t1 = training_sample_df[target_column].copy()

# Stage 1: Wet-day classification (>= 1 mm/day)
y_wet_t1 = (y_train_t1 >= WET_DAY_THRESHOLD).astype("int8")

rainfall_classifier_t1_candidate = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

rainfall_classifier_t1_candidate.fit(
    X_train_t1,
    y_wet_t1
)

# Stage 2: Rainfall amount regression on wet days only
wet_mask_t1 = y_train_t1 >= WET_DAY_THRESHOLD

X_wet_t1 = X_train_t1.loc[wet_mask_t1]
y_wet_amount_t1 = y_train_t1.loc[wet_mask_t1]

rainfall_regressor_t1_candidate = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

rainfall_regressor_t1_candidate.fit(
    X_wet_t1,
    y_wet_amount_t1
)

# Save candidate models separately from pilot artifacts
candidate_model_dir = REPORT_DIR / "candidate_models"
candidate_model_dir.mkdir(parents=True, exist_ok=True)

classifier_path = (
    candidate_model_dir / "WI_05_hgb_wet_classifier_t1_candidate.joblib"
)

regressor_path = (
    candidate_model_dir / "WI_05_hgb_wet_amount_regressor_t1_candidate.joblib"
)

joblib.dump(rainfall_classifier_t1_candidate, classifier_path)
joblib.dump(rainfall_regressor_t1_candidate, regressor_path)

print("T+1 TWO-STAGE CANDIDATE TRAINING COMPLETE")
print("-----------------------------------------")
print("Training sample rows:", len(X_train_t1))
print("Predictor count:", X_train_t1.shape[1])
print("Wet training rows:", int(wet_mask_t1.sum()))
print("Dry training rows:", int((~wet_mask_t1).sum()))
print("Classifier:", classifier_path)
print("Regressor:", regressor_path)
print("Validation data used: No")
print("Final holdout accessed: No")

T+1 TWO-STAGE CANDIDATE TRAINING COMPLETE
-----------------------------------------
Training sample rows: 352800
Predictor count: 53
Wet training rows: 135677
Dry training rows: 217123
Classifier: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_classifier_t1_candidate.joblib
Regressor: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_amount_regressor_t1_candidate.joblib
Validation data used: No
Final holdout accessed: No


In [38]:
# WI-05 — Cell 30A: Diagnose Model Feature Order

classifier_features = list(
    rainfall_classifier_t1_candidate.feature_names_in_
)

regressor_features = list(
    rainfall_regressor_t1_candidate.feature_names_in_
)

print("FEATURE ORDER DIAGNOSTIC")
print("--------------------------------")

print("Expected classifier features:", len(classifier_features))
print("Expected regressor features:", len(regressor_features))
print("Current numerical_features:", len(numerical_features))

print(
    "Classifier matches numerical_features:",
    classifier_features == list(numerical_features)
)

print(
    "Regressor matches numerical_features:",
    regressor_features == list(numerical_features)
)

print("\nFirst 10 classifier features:")
print(classifier_features[:10])

print("\nFirst 10 current numerical_features:")
print(list(numerical_features)[:10])

print("\nClassifier-only or missing features:")
print(set(classifier_features) - set(numerical_features))

print("\nUnexpected current features:")
print(set(numerical_features) - set(classifier_features))

FEATURE ORDER DIAGNOSTIC
--------------------------------
Expected classifier features: 53
Expected regressor features: 53
Current numerical_features: 53
Classifier matches numerical_features: True
Regressor matches numerical_features: True

First 10 classifier features:
['latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14']

First 10 current numerical_features:
['latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14']

Classifier-only or missing features:
set()

Unexpected current features:
set()


In [39]:
# WI-05 — Cell 30B: Inspect Actual Validation Feature Matrix

sample_file = parquet_files[0]

debug_df = pq.read_table(
    sample_file,
    columns=list(dict.fromkeys(
        ["date", target_column] + numerical_features
    ))
).to_pandas()

debug_df["date"] = pd.to_datetime(debug_df["date"])
debug_df["target_date"] = (
    debug_df["date"] + pd.to_timedelta(1, unit="D")
)

debug_validation = debug_df[
    debug_df["target_date"].between(
        VALIDATION_START,
        VALIDATION_END
    )
].copy()

debug_X = debug_validation[numerical_features]

print("VALIDATION FEATURE DIAGNOSTIC")
print("--------------------------------")
print("Validation feature shape:", debug_X.shape)
print("Expected feature count:", len(classifier_features))
print("Duplicate validation columns:", debug_X.columns.duplicated().sum())
print("Validation order matches classifier:",
      list(debug_X.columns) == classifier_features)
print("Validation order matches regressor:",
      list(debug_X.columns) == regressor_features)

print("\nMissing expected columns:")
print(set(classifier_features) - set(debug_X.columns))

print("\nUnexpected columns:")
print(set(debug_X.columns) - set(classifier_features))

print("\nValidation column order:")
print(list(debug_X.columns))

del debug_df, debug_validation, debug_X

VALIDATION FEATURE DIAGNOSTIC
--------------------------------
Validation feature shape: (10960, 53)
Expected feature count: 53
Duplicate validation columns: 0
Validation order matches classifier: True
Validation order matches regressor: True

Missing expected columns:
set()

Unexpected columns:
set()

Validation column order:
['latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'T2M_MAX_lag_1', 'T2M_MAX_lag_2', 'T2M_MAX_lag_3', 'T2M_MAX_lag_7', 'T2M_MAX_lag_14', 'T2M_MAX_lag_30', 'T2M_MIN_lag_1', 'T2M_MIN_lag_2', 'T2M_MIN_lag_3', 'T2M_MIN_lag_7', 'T2M_MIN_lag_14', 'T2M_MIN_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_mean_7', 'PRECTOTCORR_rolling_std_7', 'PRECTOTCORR_rolling_mean_14', 'PRECTOTCORR_rolling_std_14', 'PRECTOTCORR_rolling_mean_30', 'PRECTOTCORR_rolling_std_30', 'T2M_MAX

In [40]:
# WI-05 — Cell 30C: Test Candidate Prediction on One Batch

test_file = parquet_files[0]

test_df = pq.read_table(
    test_file,
    columns=list(dict.fromkeys(
        ["date", target_column] + numerical_features
    ))
).to_pandas()

test_df["date"] = pd.to_datetime(test_df["date"])
test_df["target_date"] = (
    test_df["date"] + pd.to_timedelta(1, unit="D")
)

test_validation = test_df[
    test_df["target_date"].between(
        VALIDATION_START,
        VALIDATION_END
    )
].copy()

X_test = test_validation.loc[:, classifier_features]

print("Classifier feature check:",
      list(X_test.columns) == list(
          rainfall_classifier_t1_candidate.feature_names_in_
      ))

print("Regressor feature check:",
      list(X_test.columns) == list(
          rainfall_regressor_t1_candidate.feature_names_in_
      ))

test_probabilities = (
    rainfall_classifier_t1_candidate.predict_proba(X_test)[:, 1]
)

test_amounts = rainfall_regressor_t1_candidate.predict(X_test)

print("\nSINGLE-BATCH PREDICTION TEST")
print("--------------------------------")
print("Validation rows:", len(X_test))
print("Classifier predictions:", len(test_probabilities))
print("Regressor predictions:", len(test_amounts))
print("Probability range:",
      round(test_probabilities.min(), 4),
      "to",
      round(test_probabilities.max(), 4))
print("Predicted rainfall range:",
      round(test_amounts.min(), 4),
      "to",
      round(test_amounts.max(), 4))

del test_df, test_validation, X_test

Classifier feature check: True
Regressor feature check: True

SINGLE-BATCH PREDICTION TEST
--------------------------------
Validation rows: 10960
Classifier predictions: 10960
Regressor predictions: 10960
Probability range: 0.0108 to 0.9954
Predicted rainfall range: 2.3301 to 34.2204


In [41]:
# WI-05 — Cell 30D: Full T+1 Candidate Validation (Corrected)

actual_parts = []
probability_parts = []
conditional_amount_parts = []
persistence_parts = []

classifier_features = list(
    rainfall_classifier_t1_candidate.feature_names_in_
)

regressor_features = list(
    rainfall_regressor_t1_candidate.feature_names_in_
)

for batch_number, file_path in enumerate(parquet_files, start=1):

    # Remove duplicate column names before reading
    columns_to_read = list(dict.fromkeys(
        ["date", target_column, "PRECTOTCORR_current"]
        + classifier_features
    ))

    batch_df = pq.read_table(
        file_path,
        columns=columns_to_read
    ).to_pandas()

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(1, unit="D")
    )

    validation_df = batch_df[
        batch_df["target_date"].between(
            VALIDATION_START,
            VALIDATION_END
        )
    ].copy()

    if not validation_df.empty:

        # Explicitly preserve the feature order used during training
        X_validation = validation_df.loc[
            :, classifier_features
        ]

        if list(X_validation.columns) != regressor_features:
            raise ValueError(
                f"Feature order mismatch in batch {batch_number}"
            )

        if X_validation.columns.duplicated().any():
            raise ValueError(
                f"Duplicate features in batch {batch_number}"
            )

        y_actual = validation_df[target_column].to_numpy()

        wet_probability = (
            rainfall_classifier_t1_candidate
            .predict_proba(X_validation)[:, 1]
        )

        conditional_amount = (
            rainfall_regressor_t1_candidate
            .predict(X_validation)
        )

        conditional_amount = np.maximum(
            conditional_amount, 0
        )

        persistence_prediction = (
            validation_df["PRECTOTCORR_current"].to_numpy()
        )

        actual_parts.append(y_actual)
        probability_parts.append(wet_probability)
        conditional_amount_parts.append(conditional_amount)
        persistence_parts.append(persistence_prediction)

    del batch_df, validation_df

    if batch_number % 25 == 0:
        print(
            f"Processed {batch_number}/{len(parquet_files)} batches"
        )

# Combine validation predictions
y_validation_actual = np.concatenate(actual_parts)
y_validation_probability = np.concatenate(probability_parts)
y_validation_conditional = np.concatenate(conditional_amount_parts)
y_validation_persistence = np.concatenate(persistence_parts)

# Expected rainfall
y_validation_expected = (
    y_validation_probability * y_validation_conditional
)

# Wet-day classification
validation_threshold = 0.40

y_validation_actual_wet = (
    y_validation_actual >= WET_DAY_THRESHOLD
).astype("int8")

y_validation_predicted_wet = (
    y_validation_probability >= validation_threshold
).astype("int8")

# Overall regression metrics
candidate_mae = mean_absolute_error(
    y_validation_actual,
    y_validation_expected
)

candidate_rmse = np.sqrt(
    mean_squared_error(
        y_validation_actual,
        y_validation_expected
    )
)

persistence_mae = mean_absolute_error(
    y_validation_actual,
    y_validation_persistence
)

persistence_rmse = np.sqrt(
    mean_squared_error(
        y_validation_actual,
        y_validation_persistence
    )
)

# Classification metrics
candidate_precision = precision_score(
    y_validation_actual_wet,
    y_validation_predicted_wet,
    zero_division=0
)

candidate_recall = recall_score(
    y_validation_actual_wet,
    y_validation_predicted_wet,
    zero_division=0
)

candidate_f1 = f1_score(
    y_validation_actual_wet,
    y_validation_predicted_wet,
    zero_division=0
)

candidate_pr_auc = average_precision_score(
    y_validation_actual_wet,
    y_validation_probability
)

# Conditional amount error on actual wet days
actual_wet_mask = (
    y_validation_actual >= WET_DAY_THRESHOLD
)

conditional_wet_mae = mean_absolute_error(
    y_validation_actual[actual_wet_mask],
    y_validation_conditional[actual_wet_mask]
)

# Intensity-band evaluation
intensity_bins = [-0.001, 1, 5, 20, 50, float("inf")]
intensity_labels = [
    "Below 1 mm", "1–5 mm", "5–20 mm",
    "20–50 mm", "50+ mm"
]

validation_intensity_band = pd.cut(
    y_validation_actual,
    bins=intensity_bins,
    labels=intensity_labels
)

intensity_band_results = {}

for band in intensity_labels:

    band_mask = validation_intensity_band == band

    if band_mask.sum() > 0:
        intensity_band_results[band] = {
            "rows": int(band_mask.sum()),
            "candidate_mae": float(
                mean_absolute_error(
                    y_validation_actual[band_mask],
                    y_validation_expected[band_mask]
                )
            ),
            "persistence_mae": float(
                mean_absolute_error(
                    y_validation_actual[band_mask],
                    y_validation_persistence[band_mask]
                )
            )
        }

# Save results
validation_results = {
    "experiment": "WI-05 T+1 full validation",
    "model": "Two-stage HistGradientBoosting",
    "validation_period": "2020-01-01 to 2022-12-31",
    "validation_rows": int(len(y_validation_actual)),
    "predictor_count": len(classifier_features),
    "wet_day_threshold_mm": float(WET_DAY_THRESHOLD),
    "classification_threshold": validation_threshold,
    "candidate": {
        "mae_mm": float(candidate_mae),
        "rmse_mm": float(candidate_rmse),
        "wet_day_precision": float(candidate_precision),
        "wet_day_recall": float(candidate_recall),
        "wet_day_f1": float(candidate_f1),
        "wet_day_pr_auc": float(candidate_pr_auc),
        "conditional_wet_day_mae_mm": float(conditional_wet_mae)
    },
    "persistence_baseline": {
        "mae_mm": float(persistence_mae),
        "rmse_mm": float(persistence_rmse)
    },
    "intensity_band_results": intensity_band_results,
    "holdout_accessed": False
}

validation_path = (
    REPORT_DIR / "WI_05_t1_full_validation_results.json"
)

with open(validation_path, "w", encoding="utf-8") as file:
    json.dump(validation_results, file, indent=2)

print("\nT+1 FULL VALIDATION RESULTS")
print("--------------------------------")
print("Validation rows:", len(y_validation_actual))

print("\nTwo-stage candidate:")
print("MAE:", round(candidate_mae, 4), "mm/day")
print("RMSE:", round(candidate_rmse, 4), "mm/day")
print("Precision:", round(candidate_precision, 4))
print("Recall:", round(candidate_recall, 4))
print("F1:", round(candidate_f1, 4))
print("PR-AUC:", round(candidate_pr_auc, 4))
print("Conditional wet-day MAE:", round(conditional_wet_mae, 4))

print("\nPersistence baseline:")
print("MAE:", round(persistence_mae, 4), "mm/day")
print("RMSE:", round(persistence_rmse, 4), "mm/day")

print("\nIntensity-band MAE:")
for band, result in intensity_band_results.items():
    print(
        band,
        "| Candidate:", round(result["candidate_mae"], 4),
        "| Persistence:", round(result["persistence_mae"], 4)
    )

print("\nSaved:", validation_path)
print("Holdout accessed: False")

Processed 25/252 batches
Processed 50/252 batches
Processed 75/252 batches
Processed 100/252 batches
Processed 125/252 batches
Processed 150/252 batches
Processed 175/252 batches
Processed 200/252 batches
Processed 225/252 batches
Processed 250/252 batches

T+1 FULL VALIDATION RESULTS
--------------------------------
Validation rows: 2761920

Two-stage candidate:
MAE: 2.9141 mm/day
RMSE: 7.5835 mm/day
Precision: 0.8088
Recall: 0.8654
F1: 0.8361
PR-AUC: 0.9147
Conditional wet-day MAE: 6.0015

Persistence baseline:
MAE: 3.2361 mm/day
RMSE: 9.2107 mm/day

Intensity-band MAE:
Below 1 mm | Candidate: 0.7931 | Persistence: 0.5827
1–5 mm | Candidate: 3.0391 | Persistence: 3.3274
5–20 mm | Candidate: 5.0263 | Persistence: 7.2292
20–50 mm | Candidate: 16.1854 | Persistence: 17.5959
50+ mm | Candidate: 54.1557 | Persistence: 49.1249

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t

In [42]:
# WI-05 — Cell 31: T+1 Full Baseline Comparison

t1_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": candidate_mae,
        "rmse_mm": candidate_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": 3.2361,
        "rmse_mm": 9.2107
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.8002,
        "rmse_mm": 12.1503
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t1_baseline_comparison["mae_improvement_vs_candidate_pct"] = (
    (t1_baseline_comparison["mae_mm"].iloc[1:] - candidate_mae)
    / t1_baseline_comparison["mae_mm"].iloc[1:] * 100
).tolist() + [None] if False else None

t1_baseline_comparison["mae_rank"] = (
    t1_baseline_comparison["mae_mm"].rank(method="min").astype(int)
)

t1_baseline_comparison["rmse_rank"] = (
    t1_baseline_comparison["rmse_mm"].rank(method="min").astype(int)
)

comparison_path = (
    REPORT_DIR / "WI_05_t1_full_baseline_comparison.csv"
)

t1_baseline_comparison.to_csv(comparison_path, index=False)

print("T+1 BASELINE COMPARISON")
print("--------------------------------")
print(t1_baseline_comparison.to_string(index=False))

print("\nMAE improvement over persistence:",
      round((3.2361 - candidate_mae) / 3.2361 * 100, 2), "%")

print("RMSE improvement over persistence:",
      round((9.2107 - candidate_rmse) / 9.2107 * 100, 2), "%")

print("\nSaved:", comparison_path)
print("Model status: Provisional T+1 candidate")
print("Final selection: Pending")
print("Holdout accessed: False")

T+1 BASELINE COMPARISON
--------------------------------
                         model   mae_mm   rmse_mm mae_improvement_vs_candidate_pct  mae_rank  rmse_rank
Two-stage HistGradientBoosting 2.914083  7.583502                             None         1          1
                   Persistence 3.236100  9.210700                             None         2          3
                Seasonal naive 4.800200 12.150300                             None         4          4
          Training climatology 3.764900  9.062000                             None         3          2

MAE improvement over persistence: 9.95 %
RMSE improvement over persistence: 17.67 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_full_baseline_comparison.csv
Model status: Provisional T+1 candidate
Final selection: Pending
Holdout accessed: False


In [44]:
# WI-05 — Cell 32A: Inspect Validation Result Structure

print("Type of actual_parts:", type(actual_parts))
print("Number of elements:", len(actual_parts))

if len(actual_parts) > 0:
    print("\nType of first element:", type(actual_parts[0]))

    if isinstance(actual_parts[0], dict):
        print("Keys in first element:", actual_parts[0].keys())
    elif isinstance(actual_parts[0], (list, tuple)):
        print("Number of values in first element:", len(actual_parts[0]))
        print("Types:", [type(x) for x in actual_parts[0]])
    else:
        print("First element:", actual_parts[0])

Type of actual_parts: <class 'list'>
Number of elements: 252

Type of first element: <class 'numpy.ndarray'>
First element: [0.61 0.91 0.73 ... 0.9  0.1  0.  ]


In [45]:
# WI-05 — Cell 32B: Identify Stored Validation Arrays

for name, value in globals().copy().items():
    if any(term in name.lower() for term in [
        "actual", "pred", "season", "expected", "prob", "parts"
    ]):
        if isinstance(value, list):
            if len(value) > 0 and isinstance(value[0], np.ndarray):
                print(
                    f"{name}: list, {len(value)} arrays, "
                    f"first shape={value[0].shape}, "
                    f"first dtype={value[0].dtype}"
                )
            else:
                print(f"{name}: list, length={len(value)}")

predictor_columns: list, length=53
numeric_predictors: list, length=53
training_sample_parts: list, length=252
season_order: list, length=4
actual_parts: list, 252 arrays, first shape=(10960,), first dtype=float64
probability_parts: list, 252 arrays, first shape=(10960,), first dtype=float64
conditional_amount_parts: list, 252 arrays, first shape=(10960,), first dtype=float64
persistence_parts: list, 252 arrays, first shape=(10960,), first dtype=float64
seasonal_results: list, length=0


In [46]:
# WI-05 — Cell 32C: Inspect Validation Batch Dates

sample_batch_path = sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet"))[0]

sample_batch = pd.read_parquet(
    sample_batch_path,
    columns=["date"]
)

print("Sample batch:", sample_batch_path.name)
print("Date dtype:", sample_batch["date"].dtype)
print("First five dates:")
print(sample_batch["date"].head().to_list())

print("\nValidation date range:")
print(
    sample_batch.loc[
        (sample_batch["date"] >= VALIDATION_START) &
        (sample_batch["date"] <= VALIDATION_END),
        "date"
    ].min(),
    "to",
    sample_batch.loc[
        (sample_batch["date"] >= VALIDATION_START) &
        (sample_batch["date"] <= VALIDATION_END),
        "date"
    ].max()
)

Sample batch: batch_0000.parquet
Date dtype: datetime64[ns]
First five dates:
[Timestamp('2013-01-01 00:00:00'), Timestamp('2013-01-02 00:00:00'), Timestamp('2013-01-03 00:00:00'), Timestamp('2013-01-04 00:00:00'), Timestamp('2013-01-05 00:00:00')]

Validation date range:
2020-01-01 00:00:00 to 2022-12-31 00:00:00


In [47]:
# WI-05 — Cell 32D: T+1 Seasonal Validation Performance

season_parts = []

for batch_path in sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet")):

    batch_dates = pd.read_parquet(
        batch_path,
        columns=["date"]
    )["date"]

    # Forecast target date, not forecast origin date
    target_dates = batch_dates + pd.Timedelta(days=horizon)

    # Match the exact validation split used in Cell 30D
    validation_mask = (
        (target_dates >= VALIDATION_START) &
        (target_dates <= VALIDATION_END)
    )

    validation_target_dates = target_dates.loc[validation_mask]

    months = validation_target_dates.dt.month

    seasons = np.select(
        [
            months.isin([12, 1, 2]),
            months.isin([3, 4, 5]),
            months.isin([6, 7, 8, 9]),
            months.isin([10, 11])
        ],
        [
            "Winter",
            "Pre-monsoon",
            "Monsoon",
            "Post-monsoon"
        ],
        default="Unknown"
    )

    season_parts.append(seasons)

# Combine labels in the same order as Cell 30D's batch predictions
validation_seasons = np.concatenate(season_parts)

# Reconstruct full validation arrays
y_true = np.concatenate(actual_parts)

y_pred = (
    np.concatenate(probability_parts) *
    np.concatenate(conditional_amount_parts)
)

# Verify alignment before calculating metrics
assert len(validation_seasons) == len(y_true) == len(y_pred)
assert len(y_true) == 2_761_920
assert "Unknown" not in validation_seasons

seasonal_results = []

for season_name in season_order:

    mask = validation_seasons == season_name

    y_true_season = y_true[mask]
    y_pred_season = y_pred[mask]

    seasonal_results.append({
        "season": season_name,
        "observations": len(y_true_season),
        "actual_mean_mm": np.mean(y_true_season),
        "predicted_mean_mm": np.mean(y_pred_season),
        "mae_mm": mean_absolute_error(
            y_true_season, y_pred_season
        ),
        "rmse_mm": np.sqrt(
            mean_squared_error(y_true_season, y_pred_season)
        )
    })

t1_seasonal_validation = pd.DataFrame(seasonal_results)

seasonal_path = REPORT_DIR / "WI_05_t1_seasonal_validation.csv"
t1_seasonal_validation.to_csv(seasonal_path, index=False)

print("T+1 SEASONAL VALIDATION")
print(t1_seasonal_validation.round(4).to_string(index=False))

print("\nTotal validation observations:", len(y_true))
print("Season label counts:")
print(pd.Series(validation_seasons).value_counts())

print("\nSaved:", seasonal_path)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3836508506.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3836508506.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3836508506.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Time

T+1 SEASONAL VALIDATION
      season  observations  actual_mean_mm  predicted_mean_mm  mae_mm  rmse_mm
      Winter        682920          1.0597             0.9256  1.0056   4.3359
 Pre-monsoon        695520          2.3632             2.1169  1.9257   6.3460
     Monsoon        922320          7.4632             7.3936  5.1807  10.0152
Post-monsoon        461160          3.4938             3.3032  2.6977   7.4324

Total validation observations: 2761920
Season label counts:
Monsoon         922320
Pre-monsoon     695520
Winter          682920
Post-monsoon    461160
Name: count, dtype: int64

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_seasonal_validation.csv


In [48]:
# WI-05 — Cell 33: T+1 Seasonal Comparison Against Persistence

y_persistence = np.concatenate(persistence_parts)

assert len(y_persistence) == len(y_true)

seasonal_comparison = []

for season_name in season_order:

    mask = validation_seasons == season_name

    actual = y_true[mask]
    candidate = y_pred[mask]
    persistence = y_persistence[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    candidate_rmse = np.sqrt(mean_squared_error(actual, candidate))
    persistence_rmse = np.sqrt(mean_squared_error(actual, persistence))

    seasonal_comparison.append({
        "season": season_name,
        "observations": len(actual),
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            (persistence_mae - candidate_mae)
            / persistence_mae * 100
        ),
        "candidate_rmse": candidate_rmse,
        "persistence_rmse": persistence_rmse,
        "rmse_improvement_pct": (
            (persistence_rmse - candidate_rmse)
            / persistence_rmse * 100
        )
    })

t1_seasonal_comparison = pd.DataFrame(seasonal_comparison)

comparison_path = REPORT_DIR / "WI_05_t1_seasonal_baseline_comparison.csv"
t1_seasonal_comparison.to_csv(comparison_path, index=False)

print("T+1 SEASONAL BASELINE COMPARISON")
print(t1_seasonal_comparison.round(4).to_string(index=False))
print("\nSaved:", comparison_path)

T+1 SEASONAL BASELINE COMPARISON
      season  observations  candidate_mae  persistence_mae  mae_improvement_pct  candidate_rmse  persistence_rmse  rmse_improvement_pct
      Winter        682920         1.0056           1.1215              10.3355          4.3359            5.5139               21.3644
 Pre-monsoon        695520         1.9257           2.1497              10.4204          6.3460            7.8577               19.2381
     Monsoon        922320         5.1807           5.7230               9.4755         10.0152           11.9479               16.1758
Post-monsoon        461160         2.6977           3.0325              11.0382          7.4324            9.1898               19.1230

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_seasonal_baseline_comparison.csv


In [49]:
# WI-05 — Cell 34: T+1 Rainfall Intensity Comparison

intensity_bands = [
    ("Below 1 mm", y_true < 1),
    ("1–5 mm", (y_true >= 1) & (y_true < 5)),
    ("5–20 mm", (y_true >= 5) & (y_true < 20)),
    ("20–50 mm", (y_true >= 20) & (y_true < 50)),
    ("50+ mm", y_true >= 50)
]

intensity_results = []

for band_name, mask in intensity_bands:

    actual = y_true[mask]
    candidate = y_pred[mask]
    persistence = y_persistence[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_results.append({
        "intensity_band": band_name,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            (persistence_mae - candidate_mae)
            / persistence_mae * 100
        )
    })

t1_intensity_comparison = pd.DataFrame(intensity_results)

intensity_path = REPORT_DIR / "WI_05_t1_intensity_baseline_comparison.csv"
t1_intensity_comparison.to_csv(intensity_path, index=False)

print("T+1 RAINFALL INTENSITY COMPARISON")
print(t1_intensity_comparison.round(4).to_string(index=False))
print("\nSaved:", intensity_path)

T+1 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         0.7905           0.5803             -36.2327
        1–5 mm        575375          20.8324         3.0340           3.3185               8.5721
       5–20 mm        458822          16.6124         5.0231           7.2234              30.4613
      20–50 mm        105268           3.8114        16.1780          17.5913               8.0339
        50+ mm         18977           0.6871        54.1414          49.1145             -10.2351

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_intensity_baseline_comparison.csv


In [50]:
# WI-05 — Cell 35: T+1 Heavy Rainfall Error Diagnostics

heavy_mask = y_true >= 50

heavy_actual = y_true[heavy_mask]
heavy_candidate = y_pred[heavy_mask]
heavy_persistence = y_persistence[heavy_mask]

heavy_results = {
    "observations": len(heavy_actual),

    "actual_mean_mm": np.mean(heavy_actual),
    "candidate_mean_mm": np.mean(heavy_candidate),
    "persistence_mean_mm": np.mean(heavy_persistence),

    "candidate_bias_mm": np.mean(
        heavy_candidate - heavy_actual
    ),
    "persistence_bias_mm": np.mean(
        heavy_persistence - heavy_actual
    ),

    "candidate_mae_mm": mean_absolute_error(
        heavy_actual, heavy_candidate
    ),
    "persistence_mae_mm": mean_absolute_error(
        heavy_actual, heavy_persistence
    ),

    "candidate_rmse_mm": np.sqrt(
        mean_squared_error(heavy_actual, heavy_candidate)
    ),
    "persistence_rmse_mm": np.sqrt(
        mean_squared_error(heavy_actual, heavy_persistence)
    ),

    "actual_p95_mm": np.quantile(heavy_actual, 0.95),
    "candidate_p95_mm": np.quantile(heavy_candidate, 0.95),
    "persistence_p95_mm": np.quantile(heavy_persistence, 0.95),

    "actual_max_mm": np.max(heavy_actual),
    "candidate_max_mm": np.max(heavy_candidate),
    "persistence_max_mm": np.max(heavy_persistence)
}

for key, value in heavy_results.items():
    print(f"{key}: {value:.4f}" if isinstance(value, (float, np.floating)) else f"{key}: {value}")

heavy_path = REPORT_DIR / "WI_05_t1_heavy_rainfall_diagnostics.json"

with open(heavy_path, "w", encoding="utf-8") as f:
    json.dump(heavy_results, f, indent=2)

print("\nSaved:", heavy_path)

observations: 18977
actual_mean_mm: 77.4719
candidate_mean_mm: 24.7622
persistence_mean_mm: 36.1843
candidate_bias_mm: -52.7097
persistence_bias_mm: -41.2876
candidate_mae_mm: 54.1414
persistence_mae_mm: 49.1145
candidate_rmse_mm: 66.0393
persistence_rmse_mm: 63.0324
actual_p95_mm: 142.4980
candidate_p95_mm: 64.3349
persistence_p95_mm: 101.7420
actual_max_mm: 666.3700
candidate_max_mm: 192.1879
persistence_max_mm: 508.0000

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_heavy_rainfall_diagnostics.json


In [51]:
# WI-05 — Cell 36: Training vs Validation Rainfall Tail Audit

train_rain = training_sample_df[target_column].dropna()
validation_rain = pd.Series(y_true)

rainfall_thresholds = [1, 5, 20, 50, 100, 200, 500]

distribution_results = []

for dataset_name, values in [
    ("Training sample", train_rain),
    ("Validation full population", validation_rain)
]:
    row = {
        "dataset": dataset_name,
        "observations": len(values),
        "mean_mm": values.mean(),
        "median_mm": values.median(),
        "p90_mm": values.quantile(0.90),
        "p95_mm": values.quantile(0.95),
        "p99_mm": values.quantile(0.99),
        "p99_9_mm": values.quantile(0.999),
        "maximum_mm": values.max()
    }

    for threshold in rainfall_thresholds:
        row[f"count_ge_{threshold}mm"] = int(
            (values >= threshold).sum()
        )

        row[f"pct_ge_{threshold}mm"] = (
            (values >= threshold).mean() * 100
        )

    distribution_results.append(row)

rainfall_distribution_audit = pd.DataFrame(distribution_results)

audit_path = REPORT_DIR / "WI_05_t1_training_validation_tail_audit.csv"
rainfall_distribution_audit.to_csv(audit_path, index=False)

print("TRAINING SAMPLE VS VALIDATION DISTRIBUTION")
print(rainfall_distribution_audit.round(4).to_string(index=False))

print("\nSaved:", audit_path)

TRAINING SAMPLE VS VALIDATION DISTRIBUTION
                   dataset  observations  mean_mm  median_mm  p90_mm  p95_mm  p99_mm  p99_9_mm  maximum_mm  count_ge_1mm  pct_ge_1mm  count_ge_5mm  pct_ge_5mm  count_ge_20mm  pct_ge_20mm  count_ge_50mm  pct_ge_50mm  count_ge_100mm  pct_ge_100mm  count_ge_200mm  pct_ge_200mm  count_ge_500mm  pct_ge_500mm
           Training sample        352800   3.6031       0.23   10.45   17.45 40.6001  103.6641      488.99        135677     38.4572         68748     19.4864          14073       3.9889           2153       0.6103             381        0.1080              47        0.0133               0        0.0000
Validation full population       2761920   3.9328       0.40   11.28   18.75 42.6700  100.3508      666.37       1158442     41.9434        583067     21.1109         124245       4.4985          18977       0.6871            2802        0.1015             345        0.0125               3        0.0001

Saved: C:\Users\gsr58\Desktop\Projects\Bh

In [52]:
# WI-05 — Cell 37: Conditional Regressor Training Tail Audit

conditional_train = training_sample_df[
    training_sample_df[target_column] >= WET_DAY_THRESHOLD
][target_column].dropna()

conditional_thresholds = [5, 20, 50, 100, 200, 500]

conditional_audit = {
    "wet_training_observations": len(conditional_train),
    "mean_mm": conditional_train.mean(),
    "median_mm": conditional_train.median(),
    "p90_mm": conditional_train.quantile(0.90),
    "p95_mm": conditional_train.quantile(0.95),
    "p99_mm": conditional_train.quantile(0.99),
    "p99_9_mm": conditional_train.quantile(0.999),
    "maximum_mm": conditional_train.max()
}

for threshold in conditional_thresholds:
    conditional_audit[f"count_ge_{threshold}mm"] = int(
        (conditional_train >= threshold).sum()
    )

    conditional_audit[f"pct_ge_{threshold}mm"] = (
        (conditional_train >= threshold).mean() * 100
    )

print("CONDITIONAL REGRESSOR TRAINING DISTRIBUTION")

for key, value in conditional_audit.items():
    print(
        f"{key}: {value:.4f}"
        if isinstance(value, (float, np.floating))
        else f"{key}: {value}"
    )

conditional_audit_path = (
    REPORT_DIR / "WI_05_t1_conditional_training_tail_audit.json"
)

with open(conditional_audit_path, "w", encoding="utf-8") as f:
    json.dump(conditional_audit, f, indent=2)

print("\nSaved:", conditional_audit_path)

CONDITIONAL REGRESSOR TRAINING DISTRIBUTION
wet_training_observations: 135677
mean_mm: 9.1781
median_mm: 5.0900
p90_mm: 20.4500
p95_mm: 29.7400
p99_mm: 61.1824
p99_9_mm: 144.3431
maximum_mm: 488.9900
count_ge_5mm: 68748
pct_ge_5mm: 50.6703
count_ge_20mm: 14073
pct_ge_20mm: 10.3724
count_ge_50mm: 2153
pct_ge_50mm: 1.5869
count_ge_100mm: 381
pct_ge_100mm: 0.2808
count_ge_200mm: 47
pct_ge_200mm: 0.0346
count_ge_500mm: 0
pct_ge_500mm: 0.0000

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_conditional_training_tail_audit.json


In [53]:
# WI-05 — Cell 38: Train Tail-Weighted Conditional Regressor

# Load the existing conditional regressor to preserve its configuration
baseline_conditional_model = joblib.load(
    REPORT_DIR / "candidate_models" /
    "WI_05_hgb_wet_amount_regressor_t1_candidate.joblib"
)

# Use the exact feature order from the existing model
model_features = list(
    baseline_conditional_model.feature_names_in_
)

# Select wet-day training observations
wet_training_df = training_sample_df[
    training_sample_df[target_column] >= WET_DAY_THRESHOLD
].copy()

X_wet_train = wet_training_df[model_features]
y_wet_train = wet_training_df[target_column]

# Assign experimental tail weights
tail_weights = np.select(
    [
        y_wet_train >= 100,
        y_wet_train >= 50,
        y_wet_train >= 20
    ],
    [
        8.0,
        4.0,
        2.0
    ],
    default=1.0
)

print("Training observations:", len(y_wet_train))
print("Feature count:", len(model_features))
print("\nWeight distribution:")
print(pd.Series(tail_weights).value_counts().sort_index())

# Create a separate model with the original hyperparameters
tail_weighted_regressor = HistGradientBoostingRegressor(
    **baseline_conditional_model.get_params()
)

tail_weighted_regressor.fit(
    X_wet_train,
    y_wet_train,
    sample_weight=tail_weights
)

# Save separately; do not overwrite the original candidate
weighted_model_path = (
    REPORT_DIR / "candidate_models" /
    "WI_05_hgb_wet_amount_regressor_t1_tail_weighted.joblib"
)

joblib.dump(tail_weighted_regressor, weighted_model_path)

print("\nTail-weighted regressor trained successfully.")
print("Saved:", weighted_model_path)
print("Original candidate overwritten: False")
print("Validation used for training: False")
print("Final holdout accessed: False")

Training observations: 135677
Feature count: 53

Weight distribution:
1.0    121604
2.0     11920
4.0      1772
8.0       381
Name: count, dtype: int64

Tail-weighted regressor trained successfully.
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_amount_regressor_t1_tail_weighted.joblib
Original candidate overwritten: False
Validation used for training: False
Final holdout accessed: False


In [54]:
# WI-05 — Cell 39: Full T+1 Tail-Weighted Validation

tail_weighted_probability_parts = []
tail_weighted_amount_parts = []

tail_weighted_model = joblib.load(weighted_model_path)

for batch_path in sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet")):

    batch = pd.read_parquet(
        batch_path,
        columns=list(dict.fromkeys(
            ["date", target_column] + model_features
        ))
    )

    target_dates = batch["date"] + pd.Timedelta(days=horizon)

    validation_mask = (
        (target_dates >= VALIDATION_START) &
        (target_dates <= VALIDATION_END)
    )

    if not validation_mask.any():
        continue

    X_batch = batch.loc[
        validation_mask, model_features
    ]

    weighted_amount = tail_weighted_model.predict(X_batch)

    tail_weighted_amount_parts.append(weighted_amount)

# Reuse the existing classifier probabilities
y_weighted_amount = np.concatenate(
    tail_weighted_amount_parts
)

y_weighted_expected = (
    np.concatenate(probability_parts) *
    y_weighted_amount
)

# Compare against the original candidate and persistence
y_original_expected = y_pred

comparison = []

for model_name, predictions in [
    ("Original two-stage HGB", y_original_expected),
    ("Tail-weighted two-stage HGB", y_weighted_expected),
    ("Persistence", y_persistence)
]:
    comparison.append({
        "model": model_name,
        "mae_mm": mean_absolute_error(y_true, predictions),
        "rmse_mm": np.sqrt(
            mean_squared_error(y_true, predictions)
        )
    })

tail_weighted_comparison = pd.DataFrame(comparison)

comparison_path = (
    REPORT_DIR / "WI_05_t1_tail_weighted_validation_comparison.csv"
)

tail_weighted_comparison.to_csv(comparison_path, index=False)

print("T+1 TAIL-WEIGHTED VALIDATION COMPARISON")
print(tail_weighted_comparison.round(4).to_string(index=False))

print("\nPrediction count:", len(y_weighted_expected))
print("Expected validation count:", len(y_true))
print("Saved:", comparison_path)
print("Final holdout accessed: False")

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\733233125.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\733233125.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\733233125.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.T

T+1 TAIL-WEIGHTED VALIDATION COMPARISON
                      model  mae_mm  rmse_mm
     Original two-stage HGB  2.9141   7.5835
Tail-weighted two-stage HGB  3.4076   8.0989
                Persistence  3.2361   9.2107

Prediction count: 2761920
Expected validation count: 2761920
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_tail_weighted_validation_comparison.csv
Final holdout accessed: False


In [55]:
# WI-05 — Cell 40: Original vs Tail-Weighted Intensity Comparison

intensity_bands = [
    ("Below 1 mm", y_true < 1),
    ("1–5 mm", (y_true >= 1) & (y_true < 5)),
    ("5–20 mm", (y_true >= 5) & (y_true < 20)),
    ("20–50 mm", (y_true >= 20) & (y_true < 50)),
    ("50+ mm", y_true >= 50)
]

intensity_comparison = []

for band_name, mask in intensity_bands:

    actual = y_true[mask]
    original = y_original_expected[mask]
    weighted = y_weighted_expected[mask]
    persistence = y_persistence[mask]

    original_mae = mean_absolute_error(actual, original)
    weighted_mae = mean_absolute_error(actual, weighted)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_comparison.append({
        "intensity_band": band_name,
        "observations": len(actual),
        "original_mae": original_mae,
        "weighted_mae": weighted_mae,
        "persistence_mae": persistence_mae,
        "weighted_vs_original_change_pct": (
            (weighted_mae - original_mae)
            / original_mae * 100
        )
    })

intensity_comparison_df = pd.DataFrame(intensity_comparison)

intensity_path = (
    REPORT_DIR / "WI_05_t1_weighted_intensity_comparison.csv"
)

intensity_comparison_df.to_csv(intensity_path, index=False)

print("T+1 INTENSITY: ORIGINAL VS TAIL-WEIGHTED")
print(intensity_comparison_df.round(4).to_string(index=False))
print("\nSaved:", intensity_path)

T+1 INTENSITY: ORIGINAL VS TAIL-WEIGHTED
intensity_band  observations  original_mae  weighted_mae  persistence_mae  weighted_vs_original_change_pct
    Below 1 mm       1603478        0.7905        0.9901           0.5803                          25.2478
        1–5 mm        575375        3.0340        4.1956           3.3185                          38.2846
       5–20 mm        458822        5.0231        6.4987           7.2234                          29.3780
      20–50 mm        105268       16.1780       14.4173          17.5913                         -10.8837
        50+ mm         18977       54.1414       47.9735          49.1145                         -11.3922

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_weighted_intensity_comparison.csv


In [56]:
# WI-05 — Cell 41: Prediction Bias by Actual Rainfall Intensity

bias_results = []

for band_name, mask in intensity_bands:

    actual = y_true[mask]
    original = y_original_expected[mask]
    weighted = y_weighted_expected[mask]
    persistence = y_persistence[mask]

    bias_results.append({
        "intensity_band": band_name,
        "observations": len(actual),
        "actual_mean": np.mean(actual),
        "original_mean_prediction": np.mean(original),
        "weighted_mean_prediction": np.mean(weighted),
        "persistence_mean_prediction": np.mean(persistence),
        "original_bias": np.mean(original - actual),
        "weighted_bias": np.mean(weighted - actual),
        "persistence_bias": np.mean(persistence - actual)
    })

bias_comparison = pd.DataFrame(bias_results)

bias_path = (
    REPORT_DIR / "WI_05_t1_intensity_prediction_bias.csv"
)

bias_comparison.to_csv(bias_path, index=False)

print("T+1 PREDICTION BIAS BY INTENSITY")
print(bias_comparison.round(4).to_string(index=False))
print("\nSaved:", bias_path)

T+1 PREDICTION BIAS BY INTENSITY
intensity_band  observations  actual_mean  original_mean_prediction  weighted_mean_prediction  persistence_mean_prediction  original_bias  weighted_bias  persistence_bias
    Below 1 mm       1603478       0.1313                    0.8954                    1.0984                       0.6304         0.7641         0.9671            0.4991
        1–5 mm        575375       2.6023                    4.8446                    6.1082                       4.3957         2.2423         3.5059            1.7934
       5–20 mm        458822      10.0224                    9.0889                   11.8708                      10.0022        -0.9335         1.8484           -0.0203
      20–50 mm        105268      29.3106                   15.0447                   20.1457                      19.4506       -14.2659        -9.1649           -9.8599
        50+ mm         18977      77.4719                   24.7622                   32.5375                   

In [57]:
# WI-05 — Cell 42: T+1 Wet-Day Classifier Evaluation

y_probability = np.concatenate(probability_parts)

actual_wet = (y_true >= WET_DAY_THRESHOLD).astype(int)

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

classifier_results = []

for threshold in thresholds:

    predicted_wet = (
        y_probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_wet,
        predicted_wet,
        labels=[0, 1]
    ).ravel()

    classifier_results.append({
        "threshold": threshold,
        "precision": precision_score(
            actual_wet, predicted_wet, zero_division=0
        ),
        "recall": recall_score(
            actual_wet, predicted_wet, zero_division=0
        ),
        "f1": f1_score(
            actual_wet, predicted_wet, zero_division=0
        ),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn)
    })

classifier_comparison = pd.DataFrame(classifier_results)

classifier_path = (
    REPORT_DIR / "WI_05_t1_classifier_threshold_comparison.csv"
)

classifier_comparison.to_csv(classifier_path, index=False)

print("T+1 WET-DAY CLASSIFIER THRESHOLD COMPARISON")
print(classifier_comparison.round(4).to_string(index=False))

print("\nPR-AUC:", round(
    average_precision_score(actual_wet, y_probability), 4
))

print("\nSaved:", classifier_path)

T+1 WET-DAY CLASSIFIER THRESHOLD COMPARISON
 threshold  precision  recall     f1  true_positives  false_positives  false_negatives  true_negatives
       0.3     0.7684  0.9074 0.8322         1051197           316795           107245         1286683
       0.4     0.8088  0.8654 0.8361         1002480           237044           155962         1366434
       0.5     0.8442  0.8169 0.8303          946326           174591           212116         1428887
       0.6     0.8763  0.7591 0.8135          879429           124149           279013         1479329
       0.7     0.9052  0.6885 0.7822          797623            83498           360819         1519980

PR-AUC: 0.9147

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_classifier_threshold_comparison.csv


In [58]:
# WI-05 — Cell 43: Wet-Day Probability Calibration

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fraction_positive, mean_predicted = calibration_curve(
    actual_wet,
    y_probability,
    n_bins=10,
    strategy="quantile"
)

brier_score = brier_score_loss(
    actual_wet,
    y_probability
)

calibration_results = pd.DataFrame({
    "mean_predicted_probability": mean_predicted,
    "observed_wet_frequency": fraction_positive
})

calibration_path = (
    REPORT_DIR / "WI_05_t1_classifier_calibration.csv"
)

calibration_results.to_csv(calibration_path, index=False)

print("T+1 CLASSIFIER CALIBRATION")
print(calibration_results.round(4).to_string(index=False))
print("\nBrier score:", round(brier_score, 6))
print("Saved:", calibration_path)

T+1 CLASSIFIER CALIBRATION
 mean_predicted_probability  observed_wet_frequency
                     0.0105                  0.0081
                     0.0191                  0.0144
                     0.0389                  0.0345
                     0.0952                  0.0912
                     0.2080                  0.2248
                     0.3990                  0.4261
                     0.6295                  0.6470
                     0.8219                  0.8277
                     0.9348                  0.9363
                     0.9815                  0.9843

Brier score: 0.099175
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_classifier_calibration.csv


In [59]:
# WI-05 — Cell 44: T+1 Consolidated Findings

t1_findings = {
    "work_item": "WI-05",
    "experiment": "Rainfall Forecasting T+1",
    "status": "Provisional candidate",
    "target": "PRECTOTCORR",
    "unit": "mm/day",
    "horizon": "T+1",

    "data_policy": {
        "training_period": "2013-2019",
        "validation_period": "2020-2022",
        "final_holdout_period": "2023-2024",
        "validation_observations": 2761920,
        "holdout_accessed": False,
        "chronological_split": True
    },

    "selected_candidate": {
        "model": "Two-stage HistGradientBoosting",
        "classifier": "HistGradientBoostingClassifier",
        "conditional_regressor": "HistGradientBoostingRegressor",
        "wet_day_threshold_mm": 1.0,
        "experimental_classification_threshold": 0.40,
        "mae_mm": 2.914083,
        "rmse_mm": 7.583502
    },

    "baseline_comparison": {
        "persistence": {
            "mae_mm": 3.2361,
            "rmse_mm": 9.2107
        },
        "seasonal_naive": {
            "mae_mm": 4.8002,
            "rmse_mm": 12.1503
        },
        "training_climatology": {
            "mae_mm": 3.7649,
            "rmse_mm": 9.0620
        },
        "improvement_over_persistence": {
            "mae_pct": 9.95,
            "rmse_pct": 17.67
        }
    },

    "seasonal_validation": {
        "Winter": {
            "observations": 682920,
            "mae_mm": 1.0056,
            "rmse_mm": 4.3359,
            "mae_improvement_over_persistence_pct": 10.3355
        },
        "Pre-monsoon": {
            "observations": 695520,
            "mae_mm": 1.9257,
            "rmse_mm": 6.3460,
            "mae_improvement_over_persistence_pct": 10.4204
        },
        "Monsoon": {
            "observations": 922320,
            "mae_mm": 5.1807,
            "rmse_mm": 10.0152,
            "mae_improvement_over_persistence_pct": 9.4755
        },
        "Post-monsoon": {
            "observations": 461160,
            "mae_mm": 2.6977,
            "rmse_mm": 7.4324,
            "mae_improvement_over_persistence_pct": 11.0382
        }
    },

    "intensity_findings": {
        "below_1mm": {
            "observations": 1603478,
            "candidate_mae_mm": 0.7905,
            "persistence_mae_mm": 0.5803,
            "finding": "Candidate underperforms persistence"
        },
        "1_to_5mm": {
            "observations": 575375,
            "candidate_mae_mm": 3.0340,
            "persistence_mae_mm": 3.3185
        },
        "5_to_20mm": {
            "observations": 458822,
            "candidate_mae_mm": 5.0231,
            "persistence_mae_mm": 7.2234
        },
        "20_to_50mm": {
            "observations": 105268,
            "candidate_mae_mm": 16.1780,
            "persistence_mae_mm": 17.5913
        },
        "50mm_and_above": {
            "observations": 18977,
            "candidate_mae_mm": 54.1414,
            "persistence_mae_mm": 49.1145,
            "finding": "Candidate underperforms persistence"
        }
    },

    "classifier_evaluation": {
        "pr_auc": 0.9147,
        "best_tested_f1_threshold": 0.40,
        "precision_at_0_40": 0.8088,
        "recall_at_0_40": 0.8654,
        "f1_at_0_40": 0.8361,
        "brier_score": 0.099175,
        "calibration_decision": "No probability recalibration applied"
    },

    "experimental_tail_weighting": {
        "status": "Not selected",
        "original_mae_mm": 2.9141,
        "weighted_mae_mm": 3.4076,
        "original_rmse_mm": 7.5835,
        "weighted_rmse_mm": 8.0989,
        "finding": (
            "Improved 20mm+ intensity-band MAE but substantially "
            "worsened common rainfall categories and overall metrics"
        )
    },

    "limitations": [
        "Heavy rainfall is systematically underpredicted.",
        "The candidate underperforms persistence below 1mm and at 50mm or above.",
        "Monsoon has the highest absolute prediction error.",
        "The 0.40 classifier threshold was selected using validation results "
        "and remains provisional.",
        "Validation performance does not establish final holdout performance.",
        "Historical weather data ends in 2024 and does not provide live "
        "operational forecasts for 2026."
    ],

    "final_decision": (
        "Retain the original two-stage HistGradientBoosting model as the "
        "provisional T+1 candidate. Do not select the tail-weighted "
        "regressor. Final model selection remains pending."
    )
}

findings_path = REPORT_DIR / "WI_05_t1_consolidated_findings.json"

with open(findings_path, "w", encoding="utf-8") as f:
    json.dump(t1_findings, f, indent=2)

print("T+1 CONSOLIDATED FINDINGS SAVED")
print("Path:", findings_path)
print("Status:", t1_findings["status"])
print("Selected candidate:", t1_findings["selected_candidate"]["model"])
print("Validation observations:", t1_findings["data_policy"]["validation_observations"])
print("Holdout accessed:", t1_findings["data_policy"]["holdout_accessed"])
print("Final selection: Pending")

T+1 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_consolidated_findings.json
Status: Provisional candidate
Selected candidate: Two-stage HistGradientBoosting
Validation observations: 2761920
Holdout accessed: False
Final selection: Pending


In [60]:
# WI-05 — Cell 45: T+2 Experiment Configuration

horizon = 2

target_column = f"PRECTOTCORR_target_t_plus_{horizon}"

assert target_column in predictor_columns + [
    f"PRECTOTCORR_target_t_plus_{i}" for i in range(1, 8)
]

RANDOM_STATE = 42
SAMPLES_PER_CELL_YEAR = 20

print("T+2 EXPERIMENT CONFIGURATION")
print("Forecast horizon:", horizon)
print("Target column:", target_column)
print("Training period:", TRAIN_START, "to", TRAIN_END)
print("Validation period:", VALIDATION_START, "to", VALIDATION_END)
print("Final holdout period:", HOLDOUT_START, "to", HOLDOUT_END)
print("Samples per cell-year:", SAMPLES_PER_CELL_YEAR)
print("Final holdout accessed: False")

T+2 EXPERIMENT CONFIGURATION
Forecast horizon: 2
Target column: PRECTOTCORR_target_t_plus_2
Training period: 2013-01-01 00:00:00 to 2019-12-31 00:00:00
Validation period: 2020-01-01 00:00:00 to 2022-12-31 00:00:00
Final holdout period: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Samples per cell-year: 20
Final holdout accessed: False


In [61]:
# WI-05 — Cell 46: Build T+2 Training Sample

training_sample_parts_t2 = []

batch_files = sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet"))

for batch_index, batch_path in enumerate(batch_files):

    batch = pd.read_parquet(batch_path)

    # Calculate target date for T+2
    batch["target_date"] = batch["date"] + pd.Timedelta(days=horizon)

    # Select training-period observations with complete features and target
    train_mask = (
        (batch["target_date"] >= TRAIN_START) &
        (batch["target_date"] <= TRAIN_END) &
        batch[target_column].notna()
    )

    train_batch = batch.loc[train_mask].copy()

    train_batch["target_year"] = train_batch["target_date"].dt.year

    # Sample 20 observations per grid-cell and target year
    sampled_groups = []

    for group_index, (_, group) in enumerate(
        train_batch.groupby(["cell_id", "target_year"], sort=True)
    ):
        sample_count = min(SAMPLES_PER_CELL_YEAR, len(group))

        sampled_groups.append(
            group.sample(
                n=sample_count,
                random_state=RANDOM_STATE
                + batch_index * 1000
                + group_index
            )
        )

    if sampled_groups:
        training_sample_parts_t2.append(
            pd.concat(sampled_groups, ignore_index=True)
        )

training_sample_df_t2 = pd.concat(
    training_sample_parts_t2,
    ignore_index=True
)

print("T+2 TRAINING SAMPLE")
print("Total observations:", len(training_sample_df_t2))
print("Unique grid cells:", training_sample_df_t2["cell_id"].nunique())
print("Target years:", sorted(training_sample_df_t2["target_year"].unique()))
print("Target column:", target_column)
print("Missing targets:", training_sample_df_t2[target_column].isna().sum())
print("Wet-day rate (%):", round(
    (training_sample_df_t2[target_column] >= WET_DAY_THRESHOLD).mean() * 100,
    4
))
print("Maximum rainfall:", training_sample_df_t2[target_column].max())

print("\nT+1 sample preserved:", "training_sample_df" in globals())
print("Final holdout accessed: False")

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1426895525.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch["target_date"] = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1426895525.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch["target_date"] = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1426895525.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  batch["target_da

T+2 TRAINING SAMPLE
Total observations: 352800
Unique grid cells: 2520
Target years: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Target column: PRECTOTCORR_target_t_plus_2
Missing targets: 0
Wet-day rate (%): 38.4768
Maximum rainfall: 488.99

T+1 sample preserved: True
Final holdout accessed: False


In [62]:
# WI-05 — Cell 47: T+2 Training Sample Quality Audit

t2_rain = training_sample_df_t2[target_column]

t2_quality = {
    "observations": len(t2_rain),
    "mean_mm": t2_rain.mean(),
    "median_mm": t2_rain.median(),
    "p90_mm": t2_rain.quantile(0.90),
    "p95_mm": t2_rain.quantile(0.95),
    "p99_mm": t2_rain.quantile(0.99),
    "p99_9_mm": t2_rain.quantile(0.999),
    "maximum_mm": t2_rain.max(),
    "wet_day_rate_pct": (t2_rain >= WET_DAY_THRESHOLD).mean() * 100
}

for threshold in [1, 5, 20, 50, 100, 200, 500]:
    t2_quality[f"count_ge_{threshold}mm"] = int(
        (t2_rain >= threshold).sum()
    )

    t2_quality[f"pct_ge_{threshold}mm"] = (
        (t2_rain >= threshold).mean() * 100
    )

print("T+2 TRAINING SAMPLE QUALITY")
for key, value in t2_quality.items():
    print(
        f"{key}: {value:.4f}"
        if isinstance(value, (float, np.floating))
        else f"{key}: {value}"
    )

t2_quality_path = (
    REPORT_DIR / "WI_05_t2_training_sample_quality_audit.json"
)

with open(t2_quality_path, "w", encoding="utf-8") as f:
    json.dump(t2_quality, f, indent=2)

print("\nSaved:", t2_quality_path)

T+2 TRAINING SAMPLE QUALITY
observations: 352800
mean_mm: 3.6012
median_mm: 0.2200
p90_mm: 10.4500
p95_mm: 17.4600
p99_mm: 40.4400
p99_9_mm: 104.6960
maximum_mm: 488.9900
wet_day_rate_pct: 38.4768
count_ge_1mm: 135746
pct_ge_1mm: 38.4768
count_ge_5mm: 68733
pct_ge_5mm: 19.4821
count_ge_20mm: 14049
pct_ge_20mm: 3.9821
count_ge_50mm: 2129
pct_ge_50mm: 0.6035
count_ge_100mm: 387
pct_ge_100mm: 0.1097
count_ge_200mm: 51
pct_ge_200mm: 0.0145
count_ge_500mm: 0
pct_ge_500mm: 0.0000

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_training_sample_quality_audit.json


In [63]:
# WI-05 — Cell 48: Train T+2 Two-Stage HGB Candidate

model_features_t2 = list(numerical_features)

# Separate training data
X_train_t2 = training_sample_df_t2[model_features_t2]
y_train_t2 = training_sample_df_t2[target_column]

# Wet-day classification labels
y_wet_t2 = (y_train_t2 >= WET_DAY_THRESHOLD).astype(int)

# Conditional amount training data
wet_mask_t2 = y_train_t2 >= WET_DAY_THRESHOLD

X_wet_t2 = X_train_t2.loc[wet_mask_t2]
y_wet_amount_t2 = y_train_t2.loc[wet_mask_t2]

# Train wet-day classifier
classifier_t2 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

classifier_t2.fit(X_train_t2, y_wet_t2)

# Train conditional rainfall amount regressor
regressor_t2 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

regressor_t2.fit(X_wet_t2, y_wet_amount_t2)

# Save separate T+2 model artifacts
classifier_t2_path = (
    REPORT_DIR / "candidate_models" /
    "WI_05_hgb_wet_classifier_t2_candidate.joblib"
)

regressor_t2_path = (
    REPORT_DIR / "candidate_models" /
    "WI_05_hgb_wet_amount_regressor_t2_candidate.joblib"
)

joblib.dump(classifier_t2, classifier_t2_path)
joblib.dump(regressor_t2, regressor_t2_path)

print("T+2 MODEL TRAINING COMPLETE")
print("Training observations:", len(X_train_t2))
print("Predictor count:", len(model_features_t2))
print("Wet observations:", int(wet_mask_t2.sum()))
print("Dry observations:", int((~wet_mask_t2).sum()))

print("\nClassifier saved:", classifier_t2_path)
print("Regressor saved:", regressor_t2_path)

print("\nT+1 models overwritten: False")
print("Validation used for training: False")
print("Final holdout accessed: False")

T+2 MODEL TRAINING COMPLETE
Training observations: 352800
Predictor count: 53
Wet observations: 135746
Dry observations: 217054

Classifier saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_classifier_t2_candidate.joblib
Regressor saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_amount_regressor_t2_candidate.joblib

T+1 models overwritten: False
Validation used for training: False
Final holdout accessed: False


In [64]:
# WI-05 — Cell 49: T+2 Full Validation

actual_parts_t2 = []
probability_parts_t2 = []
conditional_amount_parts_t2 = []
persistence_parts_t2 = []

for batch_path in sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet")):

    batch = pd.read_parquet(
        batch_path,
        columns=list(dict.fromkeys(
            ["date", target_column, "PRECTOTCORR_current"]
            + model_features_t2
        ))
    )

    target_dates = batch["date"] + pd.Timedelta(days=horizon)

    validation_mask = (
        (target_dates >= VALIDATION_START) &
        (target_dates <= VALIDATION_END)
    )

    if not validation_mask.any():
        continue

    validation_batch = batch.loc[validation_mask]

    X_validation = validation_batch[model_features_t2]

    actual_parts_t2.append(
        validation_batch[target_column].to_numpy()
    )

    probability_parts_t2.append(
        classifier_t2.predict_proba(X_validation)[:, 1]
    )

    conditional_amount_parts_t2.append(
        regressor_t2.predict(X_validation)
    )

    persistence_parts_t2.append(
        validation_batch["PRECTOTCORR_current"].to_numpy()
    )

# Combine predictions
y_true_t2 = np.concatenate(actual_parts_t2)
y_probability_t2 = np.concatenate(probability_parts_t2)
y_conditional_t2 = np.concatenate(conditional_amount_parts_t2)
y_persistence_t2 = np.concatenate(persistence_parts_t2)

# Two-stage expected rainfall
y_pred_t2 = y_probability_t2 * y_conditional_t2

# Full validation metrics
t2_mae = mean_absolute_error(y_true_t2, y_pred_t2)
t2_rmse = np.sqrt(mean_squared_error(y_true_t2, y_pred_t2))

print("T+2 FULL VALIDATION")
print("Observations:", len(y_true_t2))
print("MAE:", round(t2_mae, 4), "mm/day")
print("RMSE:", round(t2_rmse, 4), "mm/day")

print("\nPersistence comparison")
print("Persistence MAE:", round(
    mean_absolute_error(y_true_t2, y_persistence_t2), 4
))
print("Persistence RMSE:", round(
    np.sqrt(mean_squared_error(y_true_t2, y_persistence_t2)), 4
))

print("\nExpected validation observations:", 2761920)
print("Final holdout accessed: False")

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\627385794.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\627385794.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\627385794.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch["date"] + pd.T

T+2 FULL VALIDATION
Observations: 2761920
MAE: 3.418 mm/day
RMSE: 8.3131 mm/day

Persistence comparison
Persistence MAE: 4.0844
Persistence RMSE: 10.843

Expected validation observations: 2761920
Final holdout accessed: False


In [65]:
# WI-05 — Cell 50: Save T+2 Full Validation Results

t2_validation_results = {
    "horizon": "T+2",
    "target": "PRECTOTCORR",
    "validation_period": "2020-2022",
    "validation_observations": len(y_true_t2),

    "candidate": {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": float(t2_mae),
        "rmse_mm": float(t2_rmse)
    },

    "persistence": {
        "mae_mm": float(mean_absolute_error(
            y_true_t2, y_persistence_t2
        )),
        "rmse_mm": float(np.sqrt(mean_squared_error(
            y_true_t2, y_persistence_t2
        )))
    },

    "holdout_accessed": False,
    "model_status": "Provisional T+2 candidate"
}

t2_results_path = REPORT_DIR / "WI_05_t2_full_validation_results.json"

with open(t2_results_path, "w", encoding="utf-8") as f:
    json.dump(t2_validation_results, f, indent=2)

print("T+2 VALIDATION RESULTS SAVED")
print("Path:", t2_results_path)
print("Candidate MAE:", round(t2_mae, 4))
print("Candidate RMSE:", round(t2_rmse, 4))
print("Holdout accessed: False")

T+2 VALIDATION RESULTS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_full_validation_results.json
Candidate MAE: 3.418
Candidate RMSE: 8.3131
Holdout accessed: False


In [66]:
# WI-05 — Cell 51: T+2 Seasonal Validation

season_parts_t2 = []

for batch_path in sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet")):

    batch_dates = pd.read_parquet(
        batch_path,
        columns=["date"]
    )["date"]

    target_dates = batch_dates + pd.Timedelta(days=horizon)

    validation_mask = (
        (target_dates >= VALIDATION_START) &
        (target_dates <= VALIDATION_END)
    )

    validation_target_dates = target_dates.loc[validation_mask]
    months = validation_target_dates.dt.month

    seasons = np.select(
        [
            months.isin([12, 1, 2]),
            months.isin([3, 4, 5]),
            months.isin([6, 7, 8, 9]),
            months.isin([10, 11])
        ],
        [
            "Winter",
            "Pre-monsoon",
            "Monsoon",
            "Post-monsoon"
        ],
        default="Unknown"
    )

    season_parts_t2.append(seasons)

validation_seasons_t2 = np.concatenate(season_parts_t2)

assert len(validation_seasons_t2) == len(y_true_t2) == 2761920
assert "Unknown" not in validation_seasons_t2

seasonal_results_t2 = []

for season_name in season_order:

    mask = validation_seasons_t2 == season_name

    actual = y_true_t2[mask]
    candidate = y_pred_t2[mask]
    persistence = y_persistence_t2[mask]

    seasonal_results_t2.append({
        "season": season_name,
        "observations": len(actual),
        "actual_mean_mm": np.mean(actual),
        "predicted_mean_mm": np.mean(candidate),
        "candidate_mae_mm": mean_absolute_error(actual, candidate),
        "persistence_mae_mm": mean_absolute_error(actual, persistence),
        "candidate_rmse_mm": np.sqrt(
            mean_squared_error(actual, candidate)
        ),
        "persistence_rmse_mm": np.sqrt(
            mean_squared_error(actual, persistence)
        )
    })

t2_seasonal_validation = pd.DataFrame(seasonal_results_t2)

seasonal_path_t2 = (
    REPORT_DIR / "WI_05_t2_seasonal_validation.csv"
)

t2_seasonal_validation.to_csv(seasonal_path_t2, index=False)

print("T+2 SEASONAL VALIDATION")
print(t2_seasonal_validation.round(4).to_string(index=False))

print("\nSeason counts:")
print(pd.Series(validation_seasons_t2).value_counts())

print("\nSaved:", seasonal_path_t2)
print("Final holdout accessed: False")

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\34553378.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\34553378.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Timedelta(days=horizon)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\34553378.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  target_dates = batch_dates + pd.Timedelta(

T+2 SEASONAL VALIDATION
      season  observations  actual_mean_mm  predicted_mean_mm  candidate_mae_mm  persistence_mae_mm  candidate_rmse_mm  persistence_rmse_mm
      Winter        682920          1.0597             0.8995            1.2433              1.4634             4.6924               6.3033
 Pre-monsoon        695520          2.3632             2.0625            2.2427              2.6680             6.8632               8.9853
     Monsoon        922320          7.4632             7.3988            6.0049              7.1626            11.0336              14.2120
Post-monsoon        461160          3.4938             3.2193            3.2375              3.9456             8.1709              10.9348

Season counts:
Monsoon         922320
Pre-monsoon     695520
Winter          682920
Post-monsoon    461160
Name: count, dtype: int64

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_mo

In [67]:
# WI-05 — Cell 52: T+2 Seasonal Baseline Comparison

t2_seasonal_comparison = t2_seasonal_validation.copy()

t2_seasonal_comparison["mae_improvement_pct"] = (
    (
        t2_seasonal_comparison["persistence_mae_mm"]
        - t2_seasonal_comparison["candidate_mae_mm"]
    )
    / t2_seasonal_comparison["persistence_mae_mm"]
) * 100

t2_seasonal_comparison["rmse_improvement_pct"] = (
    (
        t2_seasonal_comparison["persistence_rmse_mm"]
        - t2_seasonal_comparison["candidate_rmse_mm"]
    )
    / t2_seasonal_comparison["persistence_rmse_mm"]
) * 100

seasonal_comparison_path_t2 = (
    REPORT_DIR / "WI_05_t2_seasonal_baseline_comparison.csv"
)

t2_seasonal_comparison.to_csv(
    seasonal_comparison_path_t2,
    index=False
)

print("T+2 SEASONAL BASELINE COMPARISON")
print(
    t2_seasonal_comparison[
        [
            "season",
            "observations",
            "mae_improvement_pct",
            "rmse_improvement_pct"
        ]
    ].round(4).to_string(index=False)
)

print("\nSaved:", seasonal_comparison_path_t2)

T+2 SEASONAL BASELINE COMPARISON
      season  observations  mae_improvement_pct  rmse_improvement_pct
      Winter        682920              15.0410               25.5560
 Pre-monsoon        695520              15.9425               23.6174
     Monsoon        922320              16.1635               22.3646
Post-monsoon        461160              17.9485               25.2758

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_seasonal_baseline_comparison.csv


In [68]:
# WI-05 — Cell 53: T+2 Rainfall Intensity Comparison

intensity_bands_t2 = [
    ("Below 1 mm", y_true_t2 < 1),
    ("1–5 mm", (y_true_t2 >= 1) & (y_true_t2 < 5)),
    ("5–20 mm", (y_true_t2 >= 5) & (y_true_t2 < 20)),
    ("20–50 mm", (y_true_t2 >= 20) & (y_true_t2 < 50)),
    ("50+ mm", y_true_t2 >= 50)
]

intensity_results_t2 = []

for band_name, mask in intensity_bands_t2:

    actual = y_true_t2[mask]
    candidate = y_pred_t2[mask]
    persistence = y_persistence_t2[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_results_t2.append({
        "intensity_band": band_name,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true_t2) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            (persistence_mae - candidate_mae)
            / persistence_mae * 100
        )
    })

t2_intensity_comparison = pd.DataFrame(intensity_results_t2)

intensity_path_t2 = (
    REPORT_DIR / "WI_05_t2_intensity_baseline_comparison.csv"
)

t2_intensity_comparison.to_csv(intensity_path_t2, index=False)

print("T+2 RAINFALL INTENSITY COMPARISON")
print(t2_intensity_comparison.round(4).to_string(index=False))
print("\nSaved:", intensity_path_t2)

T+2 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.2528           1.0876             -15.1954
        1–5 mm        575375          20.8324         3.3881           4.3578              22.2518
       5–20 mm        458822          16.6124         5.0219           8.1743              38.5656
      20–50 mm        105268           3.8114        19.1217          20.5874               7.1193
        50+ mm         18977           0.6871        61.3879          58.5887              -4.7777

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_intensity_baseline_comparison.csv


In [69]:
# WI-05 — Cell 54: T+2 Heavy Rainfall Diagnostics

heavy_mask_t2 = y_true_t2 >= 50

heavy_actual_t2 = y_true_t2[heavy_mask_t2]
heavy_candidate_t2 = y_pred_t2[heavy_mask_t2]
heavy_persistence_t2 = y_persistence_t2[heavy_mask_t2]

heavy_results_t2 = {
    "observations": len(heavy_actual_t2),

    "actual_mean_mm": np.mean(heavy_actual_t2),
    "candidate_mean_mm": np.mean(heavy_candidate_t2),
    "persistence_mean_mm": np.mean(heavy_persistence_t2),

    "candidate_bias_mm": np.mean(
        heavy_candidate_t2 - heavy_actual_t2
    ),
    "persistence_bias_mm": np.mean(
        heavy_persistence_t2 - heavy_actual_t2
    ),

    "candidate_mae_mm": mean_absolute_error(
        heavy_actual_t2, heavy_candidate_t2
    ),
    "persistence_mae_mm": mean_absolute_error(
        heavy_actual_t2, heavy_persistence_t2
    ),

    "candidate_rmse_mm": np.sqrt(
        mean_squared_error(heavy_actual_t2, heavy_candidate_t2)
    ),
    "persistence_rmse_mm": np.sqrt(
        mean_squared_error(heavy_actual_t2, heavy_persistence_t2)
    ),

    "actual_p95_mm": np.quantile(heavy_actual_t2, 0.95),
    "candidate_p95_mm": np.quantile(heavy_candidate_t2, 0.95),
    "persistence_p95_mm": np.quantile(heavy_persistence_t2, 0.95),

    "actual_max_mm": np.max(heavy_actual_t2),
    "candidate_max_mm": np.max(heavy_candidate_t2),
    "persistence_max_mm": np.max(heavy_persistence_t2)
}

for key, value in heavy_results_t2.items():
    print(
        f"{key}: {value:.4f}"
        if isinstance(value, (float, np.floating))
        else f"{key}: {value}"
    )

heavy_path_t2 = (
    REPORT_DIR / "WI_05_t2_heavy_rainfall_diagnostics.json"
)

with open(heavy_path_t2, "w", encoding="utf-8") as f:
    json.dump(heavy_results_t2, f, indent=2)

print("\nSaved:", heavy_path_t2)

observations: 18977
actual_mean_mm: 77.4719
candidate_mean_mm: 16.4462
persistence_mean_mm: 22.6844
candidate_bias_mm: -61.0257
persistence_bias_mm: -54.7875
candidate_mae_mm: 61.3879
persistence_mae_mm: 58.5887
candidate_rmse_mm: 72.4713
persistence_rmse_mm: 71.5838
actual_p95_mm: 142.4980
candidate_p95_mm: 47.3190
persistence_p95_mm: 74.7860
actual_max_mm: 666.3700
candidate_max_mm: 147.5116
persistence_max_mm: 666.3700

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_heavy_rainfall_diagnostics.json


In [70]:
# WI-05 — Cell 55: T+2 Full Baseline Comparison

t2_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": t2_mae,
        "rmse_mm": t2_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": 4.0844,
        "rmse_mm": 10.8430
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.7911,
        "rmse_mm": 12.1318
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t2_baseline_comparison["mae_rank"] = (
    t2_baseline_comparison["mae_mm"]
    .rank(method="min")
    .astype(int)
)

t2_baseline_comparison["rmse_rank"] = (
    t2_baseline_comparison["rmse_mm"]
    .rank(method="min")
    .astype(int)
)

baseline_path_t2 = (
    REPORT_DIR / "WI_05_t2_full_baseline_comparison.csv"
)

t2_baseline_comparison.to_csv(
    baseline_path_t2,
    index=False
)

print("T+2 FULL BASELINE COMPARISON")
print(t2_baseline_comparison.round(4).to_string(index=False))

print("\nMAE improvement over persistence:",
      round((4.0844 - t2_mae) / 4.0844 * 100, 2), "%")

print("MAE improvement over climatology:",
      round((3.7649 - t2_mae) / 3.7649 * 100, 2), "%")

print("\nSaved:", baseline_path_t2)
print("Final holdout accessed: False")

T+2 FULL BASELINE COMPARISON
                         model  mae_mm  rmse_mm  mae_rank  rmse_rank
Two-stage HistGradientBoosting  3.4180   8.3131         1          1
                   Persistence  4.0844  10.8430         3          3
                Seasonal naive  4.7911  12.1318         4          4
          Training climatology  3.7649   9.0620         2          2

MAE improvement over persistence: 16.32 %
MAE improvement over climatology: 9.21 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_full_baseline_comparison.csv
Final holdout accessed: False


In [71]:
# WI-05 — Cell 56: T+2 Consolidated Findings

t2_findings = {
    "work_item": "WI-05",
    "experiment": "Rainfall Forecasting T+2",
    "status": "Provisional candidate",
    "target": "PRECTOTCORR",
    "unit": "mm/day",
    "horizon": "T+2",

    "data_policy": {
        "training_period": "2013-2019",
        "validation_period": "2020-2022",
        "final_holdout_period": "2023-2024",
        "training_sample_observations": len(training_sample_df_t2),
        "validation_observations": len(y_true_t2),
        "holdout_accessed": False,
        "chronological_split": True
    },

    "selected_candidate": {
        "model": "Two-stage HistGradientBoosting",
        "classifier": "HistGradientBoostingClassifier",
        "conditional_regressor": "HistGradientBoostingRegressor",
        "wet_day_threshold_mm": 1.0,
        "mae_mm": float(t2_mae),
        "rmse_mm": float(t2_rmse)
    },

    "baseline_comparison": {
        "persistence": {"mae_mm": 4.0844, "rmse_mm": 10.8430},
        "seasonal_naive": {"mae_mm": 4.7911, "rmse_mm": 12.1318},
        "training_climatology": {"mae_mm": 3.7649, "rmse_mm": 9.0620},
        "improvement_over_persistence_mae_pct": 16.32,
        "improvement_over_climatology_mae_pct": 9.21
    },

    "seasonal_validation": {
        row["season"]: {
            "observations": int(row["observations"]),
            "candidate_mae_mm": float(row["candidate_mae_mm"]),
            "candidate_rmse_mm": float(row["candidate_rmse_mm"]),
            "mae_improvement_over_persistence_pct": float(
                t2_seasonal_comparison.loc[
                    t2_seasonal_comparison["season"] == row["season"],
                    "mae_improvement_pct"
                ].iloc[0]
            ),
            "rmse_improvement_over_persistence_pct": float(
                t2_seasonal_comparison.loc[
                    t2_seasonal_comparison["season"] == row["season"],
                    "rmse_improvement_pct"
                ].iloc[0]
            )
        }
        for row in t2_seasonal_validation.to_dict("records")
    },

    "intensity_findings": {
        row["intensity_band"]: {
            "observations": int(row["observations"]),
            "candidate_mae_mm": float(row["candidate_mae"]),
            "persistence_mae_mm": float(row["persistence_mae"]),
            "mae_improvement_pct": float(row["mae_improvement_pct"])
        }
        for row in t2_intensity_comparison.to_dict("records")
    },

    "heavy_rainfall_diagnostics": {
        "observations": int(heavy_results_t2["observations"]),
        "actual_mean_mm": float(heavy_results_t2["actual_mean_mm"]),
        "candidate_mean_mm": float(heavy_results_t2["candidate_mean_mm"]),
        "candidate_bias_mm": float(heavy_results_t2["candidate_bias_mm"]),
        "candidate_mae_mm": float(heavy_results_t2["candidate_mae_mm"]),
        "persistence_mae_mm": float(heavy_results_t2["persistence_mae_mm"]),
        "candidate_p95_mm": float(heavy_results_t2["candidate_p95_mm"]),
        "actual_p95_mm": float(heavy_results_t2["actual_p95_mm"]),
        "candidate_max_mm": float(heavy_results_t2["candidate_max_mm"]),
        "actual_max_mm": float(heavy_results_t2["actual_max_mm"])
    },

    "limitations": [
        "The candidate underperforms persistence below 1 mm and at 50 mm or above.",
        "Heavy rainfall is systematically underpredicted.",
        "Monsoon has the highest absolute prediction error.",
        "The candidate's maximum prediction is substantially below the observed maximum.",
        "Validation performance does not establish final holdout performance.",
        "Historical weather data ends in 2024 and does not provide live operational forecasts for 2026."
    ],

    "final_decision": (
        "Retain the original two-stage HistGradientBoosting model as the "
        "provisional T+2 candidate. Final model selection remains pending."
    )
}

t2_findings_path = REPORT_DIR / "WI_05_t2_consolidated_findings.json"

with open(t2_findings_path, "w", encoding="utf-8") as f:
    json.dump(t2_findings, f, indent=2)

print("T+2 CONSOLIDATED FINDINGS SAVED")
print("Path:", t2_findings_path)
print("Status:", t2_findings["status"])
print("Selected candidate:", t2_findings["selected_candidate"]["model"])
print("Validation observations:", t2_findings["data_policy"]["validation_observations"])
print("Holdout accessed:", t2_findings["data_policy"]["holdout_accessed"])
print("Final selection: Pending")

T+2 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_consolidated_findings.json
Status: Provisional candidate
Selected candidate: Two-stage HistGradientBoosting
Validation observations: 2761920
Holdout accessed: False
Final selection: Pending


In [72]:
# WI-05 — Cell 57: Configure T+3 Rainfall Forecasting

horizon = 3
target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

print("WI-05 RAINFALL MODEL DEVELOPMENT")
print("Horizon:", f"T+{horizon}")
print("Target:", target_col)
print("Training period:", TRAIN_START, "to", TRAIN_END)
print("Validation period:", VALIDATION_START, "to", VALIDATION_END)
print("Final holdout period:", HOLDOUT_START, "to", HOLDOUT_END)
print("Wet-day threshold:", 1.0, "mm/day")
print("Predictor count:", len(FEATURE_COLS))
print("Final holdout accessed: False")

WI-05 RAINFALL MODEL DEVELOPMENT
Horizon: T+3
Target: PRECTOTCORR_target_t_plus_3
Training period: 2013-01-01 00:00:00 to 2019-12-31 00:00:00
Validation period: 2020-01-01 00:00:00 to 2022-12-31 00:00:00
Final holdout period: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Wet-day threshold: 1.0 mm/day


NameError: name 'FEATURE_COLS' is not defined

In [73]:
# WI-05 — Inspect existing feature-list variables

feature_variables = [
    name for name in globals()
    if "feature" in name.lower() and isinstance(globals()[name], (list, tuple))
]

print("Available feature-list variables:")
for name in feature_variables:
    print(name, "->", len(globals()[name]), "columns")

Available feature-list variables:
FEATURE_BATCH_PATHS -> 252 columns
feature_columns -> 69 columns
feature_batch_files -> 252 columns
numerical_features -> 53 columns
classifier_features -> 53 columns
regressor_features -> 53 columns
model_features -> 53 columns
model_features_t2 -> 53 columns


In [74]:
# WI-05 — Cell 58: Verify Predictors and Configure T+3

print("Feature counts:")
print("T+1 classifier:", len(classifier_features))
print("T+1 regressor:", len(regressor_features))
print("T+2 model:", len(model_features_t2))
print("General model:", len(model_features))
print("Numerical:", len(numerical_features))

assert len(model_features_t2) == 53
assert classifier_features == model_features_t2
assert regressor_features == model_features_t2
assert model_features == model_features_t2

model_features_t3 = model_features_t2.copy()

horizon = 3
target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

print("\nT+3 CONFIGURATION")
print("Horizon:", horizon)
print("Target:", target_col)
print("Predictor count:", len(model_features_t3))
print("Feature consistency: PASSED")
print("Final holdout accessed: False")

Feature counts:
T+1 classifier: 53
T+1 regressor: 53
T+2 model: 53
General model: 53
Numerical: 53

T+3 CONFIGURATION
Horizon: 3
Target: PRECTOTCORR_target_t_plus_3
Predictor count: 53
Feature consistency: PASSED
Final holdout accessed: False


In [76]:
# WI-05 — Inspect actual identifier and date columns

sample_batch = pd.read_parquet(FEATURE_BATCH_PATHS[0])

print("Available columns:")
print(sample_batch.columns.tolist())

print("\nPotential identifier columns:")
print([
    col for col in sample_batch.columns
    if any(term in col.lower() for term in ["grid", "cell", "lat", "lon", "id"])
])

print("\nRows:", len(sample_batch))
print("Date column present:", "date" in sample_batch.columns)

Available columns:
['cell_id', 'date', 'latitude', 'longitude', 'PRECTOTCORR_current', 'T2M_MAX_current', 'T2M_MIN_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'T2M_MAX_lag_1', 'T2M_MAX_lag_2', 'T2M_MAX_lag_3', 'T2M_MAX_lag_7', 'T2M_MAX_lag_14', 'T2M_MAX_lag_30', 'T2M_MIN_lag_1', 'T2M_MIN_lag_2', 'T2M_MIN_lag_3', 'T2M_MIN_lag_7', 'T2M_MIN_lag_14', 'T2M_MIN_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_mean_7', 'PRECTOTCORR_rolling_std_7', 'PRECTOTCORR_rolling_mean_14', 'PRECTOTCORR_rolling_std_14', 'PRECTOTCORR_rolling_mean_30', 'PRECTOTCORR_rolling_std_30', 'T2M_MAX_rolling_mean_3', 'T2M_MAX_rolling_std_3', 'T2M_MAX_rolling_mean_7', 'T2M_MAX_rolling_std_7', 'T2M_MAX_rolling_mean_14', 'T2M_MAX_rolling_std_14', 'T2M_MAX_rolling_mean_30', 'T2M_MAX_rolling_std_30', 'T2M_MIN_rolling_mean_3', 'T2M_MIN_rolling_std_3', 'T2M_MIN_rolling_mean_7', 'T2M_MIN_rolli

In [77]:
# WI-05 — Cell 59B: Generate T+3 Training Sample

sample_parts_t3 = []

for batch_index, batch_path in enumerate(FEATURE_BATCH_PATHS):
    batch_df = pd.read_parquet(batch_path)

    batch_df["target_date"] = (
        pd.to_datetime(batch_df["date"]) +
        pd.Timedelta(days=int(horizon))
    )

    train_batch = batch_df[
        batch_df["target_date"].dt.year.between(2013, 2019)
    ].dropna(subset=model_features_t3 + [target_col])

    for group_index, (_, group) in enumerate(
        train_batch.groupby("cell_id")
    ):
        sample_size = min(140, len(group))

        sampled = group.sample(
            n=sample_size,
            random_state=42 + batch_index * 1000 + group_index
        )

        sample_parts_t3.append(sampled)

    del batch_df, train_batch

training_sample_df_t3 = pd.concat(
    sample_parts_t3,
    ignore_index=True
)

print("T+3 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t3))
print("Grid cells:", training_sample_df_t3["cell_id"].nunique())
print("Training years:",
      sorted(training_sample_df_t3["target_date"].dt.year.unique()))
print("Target:", target_col)
print("Missing target:", training_sample_df_t3[target_col].isna().sum())
print("Wet-day rate (%):", round(
    (training_sample_df_t3[target_col] >= 1.0).mean() * 100, 4
))
print("Maximum rainfall:", training_sample_df_t3[target_col].max())

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3785572317.py:10: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3785572317.py:10: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\3785572317.py:10: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pd.Timedelta(days=int(horizon))
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\37

T+3 TRAINING SAMPLE
Rows: 352800
Grid cells: 2520
Training years: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Target: PRECTOTCORR_target_t_plus_3
Missing target: 0
Wet-day rate (%): 38.746
Maximum rainfall: 509.31


In [78]:
# WI-05 — Cell 60: T+3 Training Sample Quality Audit

t3_target = training_sample_df_t3[target_col]

print("T+3 TARGET DISTRIBUTION")
print(t3_target.describe(percentiles=[
    0.50, 0.90, 0.95, 0.99, 0.999
]).round(4))

print("\nRAINFALL INTENSITY COUNTS")
for threshold in [1, 5, 20, 50, 100, 200, 500]:
    count = int((t3_target >= threshold).sum())
    print(f">= {threshold:>3} mm: {count:>7} ({count / len(t3_target) * 100:.4f}%)")

print("\nWET / DRY COUNTS")
print("Wet (>=1 mm):", int((t3_target >= 1.0).sum()))
print("Dry (<1 mm):", int((t3_target < 1.0).sum()))

T+3 TARGET DISTRIBUTION
count    352800.0000
mean          3.6388
std           9.5056
min           0.0000
50%           0.2400
90%          10.5400
95%          17.5400
99%          40.8800
99.9%       103.4500
max         509.3100
Name: PRECTOTCORR_target_t_plus_3, dtype: float64

RAINFALL INTENSITY COUNTS
>=   1 mm:  136696 (38.7460%)
>=   5 mm:   69228 (19.6224%)
>=  20 mm:   14188 (4.0215%)
>=  50 mm:    2242 (0.6355%)
>= 100 mm:     388 (0.1100%)
>= 200 mm:      58 (0.0164%)
>= 500 mm:       1 (0.0003%)

WET / DRY COUNTS
Wet (>=1 mm): 136696
Dry (<1 mm): 216104


In [79]:
# WI-05 — Cell 61: Train T+3 Two-Stage HGB Candidate

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

X_train_t3 = training_sample_df_t3[model_features_t3]
y_train_t3 = training_sample_df_t3[target_col]

wet_mask_t3 = y_train_t3 >= 1.0

hgb_classifier_t3 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hgb_regressor_t3 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hgb_classifier_t3.fit(
    X_train_t3,
    wet_mask_t3.astype(int)
)

hgb_regressor_t3.fit(
    X_train_t3.loc[wet_mask_t3],
    y_train_t3.loc[wet_mask_t3]
)

classifier_path_t3 = (
    REPORT_DIR.parent / "candidate_models" /
    "WI_05_hgb_wet_classifier_t3_candidate.joblib"
)

regressor_path_t3 = (
    REPORT_DIR.parent / "candidate_models" /
    "WI_05_hgb_wet_amount_regressor_t3_candidate.joblib"
)

classifier_path_t3.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(hgb_classifier_t3, classifier_path_t3)
joblib.dump(hgb_regressor_t3, regressor_path_t3)

print("T+3 MODELS TRAINED")
print("Training observations:", len(X_train_t3))
print("Predictors:", X_train_t3.shape[1])
print("Wet observations:", int(wet_mask_t3.sum()))
print("Dry observations:", int((~wet_mask_t3).sum()))
print("Classifier:", classifier_path_t3)
print("Regressor:", regressor_path_t3)
print("Final holdout accessed: False")

T+3 MODELS TRAINED
Training observations: 352800
Predictors: 53
Wet observations: 136696
Dry observations: 216104
Classifier: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_classifier_t3_candidate.joblib
Regressor: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t3_candidate.joblib
Final holdout accessed: False


In [80]:
# WI-05 — Cell 62: T+3 Full Validation Evaluation

actual_parts_t3 = []
probability_parts_t3 = []
conditional_amount_parts_t3 = []
persistence_parts_t3 = []

for batch_path in FEATURE_BATCH_PATHS:
    batch_df = pd.read_parquet(batch_path)

    batch_df["target_date"] = (
        pd.to_datetime(batch_df["date"]) +
        pd.to_timedelta(horizon, unit="D")
    )

    valid_batch = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t3 + [target_col])

    if len(valid_batch) == 0:
        continue

    X_valid_batch = valid_batch[model_features_t3]

    actual_parts_t3.append(
        valid_batch[target_col].to_numpy()
    )

    probability_parts_t3.append(
        hgb_classifier_t3.predict_proba(X_valid_batch)[:, 1]
    )

    conditional_amount_parts_t3.append(
        hgb_regressor_t3.predict(X_valid_batch)
    )

    persistence_parts_t3.append(
        valid_batch["PRECTOTCORR_current"].to_numpy()
    )

    del batch_df, valid_batch, X_valid_batch

y_true_t3 = np.concatenate(actual_parts_t3)
y_probability_t3 = np.concatenate(probability_parts_t3)
y_conditional_t3 = np.concatenate(conditional_amount_parts_t3)
y_persistence_t3 = np.concatenate(persistence_parts_t3)

y_pred_t3 = y_probability_t3 * y_conditional_t3

t3_mae = mean_absolute_error(y_true_t3, y_pred_t3)
t3_rmse = np.sqrt(mean_squared_error(y_true_t3, y_pred_t3))

persistence_mae_t3 = mean_absolute_error(
    y_true_t3, y_persistence_t3
)
persistence_rmse_t3 = np.sqrt(
    mean_squared_error(y_true_t3, y_persistence_t3)
)

print("T+3 FULL VALIDATION RESULTS")
print("Observations:", len(y_true_t3))
print("Candidate MAE:", round(t3_mae, 4))
print("Candidate RMSE:", round(t3_rmse, 4))
print("Persistence MAE:", round(persistence_mae_t3, 4))
print("Persistence RMSE:", round(persistence_rmse_t3, 4))
print("MAE improvement (%):", round(
    (1 - t3_mae / persistence_mae_t3) * 100, 2
))
print("RMSE improvement (%):", round(
    (1 - t3_rmse / persistence_rmse_t3) * 100, 2
))
print("Final holdout accessed: False")

T+3 FULL VALIDATION RESULTS
Observations: 2761920
Candidate MAE: 3.579
Candidate RMSE: 8.552
Persistence MAE: 4.3918
Persistence RMSE: 11.3489
MAE improvement (%): 18.51
RMSE improvement (%): 24.64
Final holdout accessed: False


In [81]:
# WI-05 — Cell 63: Save T+3 Full Validation Results

t3_validation_results = {
    "work_item": "WI-05",
    "horizon": "T+3",
    "target": target_col,
    "validation_period": "2020-2022",
    "observations": int(len(y_true_t3)),
    "candidate_model": "Two-stage HistGradientBoosting",
    "candidate_mae_mm": float(t3_mae),
    "candidate_rmse_mm": float(t3_rmse),
    "persistence_mae_mm": float(persistence_mae_t3),
    "persistence_rmse_mm": float(persistence_rmse_t3),
    "mae_improvement_over_persistence_pct": float(
        (1 - t3_mae / persistence_mae_t3) * 100
    ),
    "rmse_improvement_over_persistence_pct": float(
        (1 - t3_rmse / persistence_rmse_t3) * 100
    ),
    "holdout_accessed": False,
    "status": "Provisional candidate"
}

t3_results_path = REPORT_DIR / "WI_05_t3_full_validation_results.json"

with open(t3_results_path, "w", encoding="utf-8") as f:
    json.dump(t3_validation_results, f, indent=2)

print("T+3 VALIDATION RESULTS SAVED")
print("Path:", t3_results_path)
print("Observations:", t3_validation_results["observations"])
print("Candidate MAE:", round(t3_mae, 4))
print("Candidate RMSE:", round(t3_rmse, 4))
print("Holdout accessed:", False)

T+3 VALIDATION RESULTS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_full_validation_results.json
Observations: 2761920
Candidate MAE: 3.579
Candidate RMSE: 8.552
Holdout accessed: False


In [82]:
# WI-05 — Cell 64: T+3 Seasonal Validation

season_parts_t3 = []

for batch_path in FEATURE_BATCH_PATHS:
    batch_df = pd.read_parquet(batch_path)

    batch_df["target_date"] = (
        pd.to_datetime(batch_df["date"]) +
        pd.to_timedelta(3, unit="D")
    )

    valid_batch = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t3 + [target_col])

    if len(valid_batch) == 0:
        continue

    month = valid_batch["target_date"].dt.month

    season = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8, 9]),
            month.isin([10, 11])
        ],
        [
            "Winter",
            "Pre-monsoon",
            "Monsoon",
            "Post-monsoon"
        ],
        default="Unknown"
    )

    season_parts_t3.append(season)

    del batch_df, valid_batch

season_labels_t3 = np.concatenate(season_parts_t3)

assert len(season_labels_t3) == len(y_true_t3)

seasonal_rows_t3 = []

for season_name in [
    "Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"
]:
    mask = season_labels_t3 == season_name

    actual = y_true_t3[mask]
    predicted = y_pred_t3[mask]
    persistence = y_persistence_t3[mask]

    candidate_mae = mean_absolute_error(actual, predicted)
    candidate_rmse = np.sqrt(mean_squared_error(actual, predicted))

    persistence_mae = mean_absolute_error(actual, persistence)
    persistence_rmse = np.sqrt(mean_squared_error(actual, persistence))

    seasonal_rows_t3.append({
        "season": season_name,
        "observations": int(mask.sum()),
        "actual_mean_mm": float(actual.mean()),
        "candidate_mean_mm": float(predicted.mean()),
        "candidate_mae_mm": float(candidate_mae),
        "candidate_rmse_mm": float(candidate_rmse),
        "persistence_mae_mm": float(persistence_mae),
        "persistence_rmse_mm": float(persistence_rmse),
        "mae_improvement_pct": float(
            (1 - candidate_mae / persistence_mae) * 100
        ),
        "rmse_improvement_pct": float(
            (1 - candidate_rmse / persistence_rmse) * 100
        )
    })

t3_seasonal_validation = pd.DataFrame(seasonal_rows_t3)

print("T+3 SEASONAL VALIDATION")
print(t3_seasonal_validation.round(4).to_string(index=False))
print("\nTotal observations:", t3_seasonal_validation["observations"].sum())

T+3 SEASONAL VALIDATION
      season  observations  actual_mean_mm  candidate_mean_mm  candidate_mae_mm  candidate_rmse_mm  persistence_mae_mm  persistence_rmse_mm  mae_improvement_pct  rmse_improvement_pct
      Winter        682920          1.0597             0.9061            1.3124             4.7614              1.6047               6.5566              18.2142               27.3808
 Pre-monsoon        695520          2.3632             2.0623            2.3538             7.0095              2.8625               9.3265              17.7707               24.8433
     Monsoon        922320          7.4632             7.4374            6.2606            11.4147              7.6518              14.9215              18.1814               23.5015
Post-monsoon        461160          3.4938             3.2523            3.4204             8.3516              4.3059              11.4552              20.5646               27.0929

Total observations: 2761920


In [83]:
# WI-05 — Cell 65: T+3 Rainfall Intensity Evaluation

intensity_bands_t3 = [
    ("Below 1 mm", y_true_t3 < 1),
    ("1-5 mm", (y_true_t3 >= 1) & (y_true_t3 < 5)),
    ("5-20 mm", (y_true_t3 >= 5) & (y_true_t3 < 20)),
    ("20-50 mm", (y_true_t3 >= 20) & (y_true_t3 < 50)),
    ("50+ mm", y_true_t3 >= 50)
]

intensity_rows_t3 = []

for band_name, mask in intensity_bands_t3:
    actual = y_true_t3[mask]
    predicted = y_pred_t3[mask]
    persistence = y_persistence_t3[mask]

    candidate_mae = mean_absolute_error(actual, predicted)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_rows_t3.append({
        "intensity_band": band_name,
        "observations": int(mask.sum()),
        "observation_pct": float(mask.mean() * 100),
        "candidate_mae": float(candidate_mae),
        "persistence_mae": float(persistence_mae),
        "mae_improvement_pct": float(
            (1 - candidate_mae / persistence_mae) * 100
        )
    })

t3_intensity_comparison = pd.DataFrame(intensity_rows_t3)

print("T+3 RAINFALL INTENSITY COMPARISON")
print(t3_intensity_comparison.round(4).to_string(index=False))
print("\nTotal observations:", t3_intensity_comparison["observations"].sum())

T+3 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.4147           1.3609              -3.9584
        1-5 mm        575375          20.8324         3.3974           4.6158              26.3952
       5-20 mm        458822          16.6124         5.1705           8.4204              38.5958
      20-50 mm        105268           3.8114        19.9522          21.4747               7.0897
        50+ mm         18977           0.6871        62.6588          61.5446              -1.8105

Total observations: 2761920


In [84]:
# WI-05 — Cell 66: T+3 Heavy Rainfall Diagnostics

heavy_mask_t3 = y_true_t3 >= 50

heavy_actual_t3 = y_true_t3[heavy_mask_t3]
heavy_candidate_t3 = y_pred_t3[heavy_mask_t3]
heavy_persistence_t3 = y_persistence_t3[heavy_mask_t3]

heavy_results_t3 = {
    "observations": int(heavy_mask_t3.sum()),

    "actual_mean_mm": float(heavy_actual_t3.mean()),
    "candidate_mean_mm": float(heavy_candidate_t3.mean()),
    "persistence_mean_mm": float(heavy_persistence_t3.mean()),

    "candidate_bias_mm": float(
        (heavy_candidate_t3 - heavy_actual_t3).mean()
    ),
    "persistence_bias_mm": float(
        (heavy_persistence_t3 - heavy_actual_t3).mean()
    ),

    "candidate_mae_mm": float(
        mean_absolute_error(heavy_actual_t3, heavy_candidate_t3)
    ),
    "persistence_mae_mm": float(
        mean_absolute_error(heavy_actual_t3, heavy_persistence_t3)
    ),

    "candidate_rmse_mm": float(
        np.sqrt(mean_squared_error(heavy_actual_t3, heavy_candidate_t3))
    ),
    "persistence_rmse_mm": float(
        np.sqrt(mean_squared_error(heavy_actual_t3, heavy_persistence_t3))
    ),

    "actual_p95_mm": float(np.percentile(heavy_actual_t3, 95)),
    "candidate_p95_mm": float(np.percentile(heavy_candidate_t3, 95)),
    "persistence_p95_mm": float(np.percentile(heavy_persistence_t3, 95)),

    "actual_max_mm": float(heavy_actual_t3.max()),
    "candidate_max_mm": float(heavy_candidate_t3.max()),
    "persistence_max_mm": float(heavy_persistence_t3.max())
}

print("T+3 HEAVY RAINFALL DIAGNOSTICS (ACTUAL >= 50 mm)")
for key, value in heavy_results_t3.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

T+3 HEAVY RAINFALL DIAGNOSTICS (ACTUAL >= 50 mm)
observations: 18977
actual_mean_mm: 77.4719
candidate_mean_mm: 15.2398
persistence_mean_mm: 18.2526
candidate_bias_mm: -62.2321
persistence_bias_mm: -59.2193
candidate_mae_mm: 62.6588
persistence_mae_mm: 61.5446
candidate_rmse_mm: 73.5146
persistence_rmse_mm: 73.5814
actual_p95_mm: 142.4980
candidate_p95_mm: 47.0082
persistence_p95_mm: 65.6220
actual_max_mm: 666.3700
candidate_max_mm: 157.4192
persistence_max_mm: 440.8500


In [85]:
# WI-05 — Cell 67: T+3 Full Baseline Comparison

t3_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": t3_mae,
        "rmse_mm": t3_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": 4.3918,
        "rmse_mm": 11.3489
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.8055,
        "rmse_mm": 12.1839
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t3_baseline_comparison["mae_rank"] = (
    t3_baseline_comparison["mae_mm"]
    .rank(method="min")
    .astype(int)
)

t3_baseline_comparison["rmse_rank"] = (
    t3_baseline_comparison["rmse_mm"]
    .rank(method="min")
    .astype(int)
)

baseline_path_t3 = REPORT_DIR / "WI_05_t3_full_baseline_comparison.csv"

t3_baseline_comparison.to_csv(
    baseline_path_t3,
    index=False
)

print("T+3 FULL BASELINE COMPARISON")
print(t3_baseline_comparison.round(4).to_string(index=False))

print(
    "\nMAE improvement over persistence:",
    round((1 - t3_mae / 4.3918) * 100, 2), "%"
)

print(
    "MAE improvement over climatology:",
    round((1 - t3_mae / 3.7649) * 100, 2), "%"
)

print("\nSaved:", baseline_path_t3)
print("Final holdout accessed: False")

T+3 FULL BASELINE COMPARISON
                         model  mae_mm  rmse_mm  mae_rank  rmse_rank
Two-stage HistGradientBoosting  3.5790   8.5520         1          1
                   Persistence  4.3918  11.3489         3          3
                Seasonal naive  4.8055  12.1839         4          4
          Training climatology  3.7649   9.0620         2          2

MAE improvement over persistence: 18.51 %
MAE improvement over climatology: 4.94 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_full_baseline_comparison.csv
Final holdout accessed: False


In [86]:
# WI-05 — Cell 68: Save T+3 Seasonal Comparison

t3_seasonal_path = REPORT_DIR / "WI_05_t3_seasonal_baseline_comparison.csv"

t3_seasonal_validation.to_csv(
    t3_seasonal_path,
    index=False
)

print("T+3 SEASONAL COMPARISON SAVED")
print("Path:", t3_seasonal_path)
print("Seasons:", len(t3_seasonal_validation))
print(
    "All seasons beat persistence:",
    bool(
        (t3_seasonal_validation["mae_improvement_pct"] > 0).all()
        and
        (t3_seasonal_validation["rmse_improvement_pct"] > 0).all()
    )
)
print("Final holdout accessed: False")

T+3 SEASONAL COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_seasonal_baseline_comparison.csv
Seasons: 4
All seasons beat persistence: True
Final holdout accessed: False


In [87]:
# WI-05 — Cell 69: Save T+3 Intensity Comparison

t3_intensity_path = REPORT_DIR / "WI_05_t3_rainfall_intensity_comparison.csv"

t3_intensity_comparison.to_csv(
    t3_intensity_path,
    index=False
)

print("T+3 INTENSITY COMPARISON SAVED")
print("Path:", t3_intensity_path)
print("\nIntensity comparison:")
print(t3_intensity_comparison.to_string(index=False))
print("\nFinal holdout accessed: False")

T+3 INTENSITY COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_rainfall_intensity_comparison.csv

Intensity comparison:
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478        58.056642       1.414722         1.360855            -3.958364
        1-5 mm        575375        20.832428       3.397442         4.615789            26.395195
       5-20 mm        458822        16.612429       5.170497         8.420427            38.595785
      20-50 mm        105268         3.811407      19.952201        21.474702             7.089742
        50+ mm         18977         0.687094      62.658825        61.544561            -1.810498

Final holdout accessed: False


In [88]:
# WI-05 — Cell 70: Consolidate T+3 Findings

t3_findings = {
    "work_item": "WI-05",
    "target": "PRECTOTCORR",
    "horizon": "T+3",
    "model": "Two-stage HistGradientBoosting",
    "status": "provisional_candidate",
    "validation_observations": 2761920,
    "validation_mae_mm": float(t3_mae),
    "validation_rmse_mm": float(t3_rmse),
    "baseline_comparison": t3_baseline_comparison.to_dict(orient="records"),
    "seasonal_comparison": t3_seasonal_validation.to_dict(orient="records"),
    "intensity_comparison": t3_intensity_comparison.to_dict(orient="records"),
    "heavy_rainfall_diagnostics": heavy_results_t3,
    "final_holdout_accessed": False,
    "final_model_selected": False
}

t3_findings_path = REPORT_DIR / "WI_05_t3_consolidated_findings.json"

with open(t3_findings_path, "w", encoding="utf-8") as f:
    json.dump(
        t3_findings,
        f,
        indent=4,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

print("T+3 CONSOLIDATED FINDINGS SAVED")
print("Path:", t3_findings_path)
print("Status:", t3_findings["status"])
print("Final model selected:", t3_findings["final_model_selected"])
print("Final holdout accessed:", t3_findings["final_holdout_accessed"])

T+3 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_consolidated_findings.json
Status: provisional_candidate
Final model selected: False
Final holdout accessed: False


In [89]:
# WI-05 — Cell 71: T+4 Configuration

horizon = 4
target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

model_features_t4 = model_features_t2.copy()

print("WI-05 T+4 CONFIGURATION")
print("Horizon:", horizon)
print("Target:", target_col)
print("Number of features:", len(model_features_t4))
print("Feature consistency:", model_features_t4 == classifier_features == regressor_features)
print("Training period: 2013–2019")
print("Validation period: 2020–2022")
print("Final holdout: 2023–2024 (untouched)")

WI-05 T+4 CONFIGURATION
Horizon: 4
Target: PRECTOTCORR_target_t_plus_4
Number of features: 53
Feature consistency: True
Training period: 2013–2019
Validation period: 2020–2022
Final holdout: 2023–2024 (untouched)


In [91]:
# WI-05 — Cell 72: Build T+4 Training Sample

FEATURES_DIR = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
    r"\phase3b_weather_intelligence\data\processed\weather_features_v1"
    r"\weather-features-v1"
)

sample_parts_t4 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    train_df = batch_df[
        batch_df["target_date"].dt.year.between(2013, 2019)
    ]

    for group_index, (_, group) in enumerate(
        train_df.groupby("cell_id", sort=False)
    ):
        sampled_group = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + group_index
        )
        sample_parts_t4.append(sampled_group)

    del batch_df, train_df

training_sample_df_t4 = pd.concat(
    sample_parts_t4,
    ignore_index=True
)

print("T+4 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t4))
print("Unique grid cells:", training_sample_df_t4["cell_id"].nunique())
print("Missing targets:", training_sample_df_t4[target_col].isna().sum())
print(
    "Wet-day rate (%):",
    round((training_sample_df_t4[target_col] >= 1.0).mean() * 100, 4)
)
print("Target maximum:", training_sample_df_t4[target_col].max())
print("Target-date years:")
print(training_sample_df_t4["target_date"].dt.year.value_counts().sort_index())

T+4 TRAINING SAMPLE
Rows: 50400
Unique grid cells: 2520
Missing targets: 0
Wet-day rate (%): 38.494
Target maximum: 249.9
Target-date years:
target_date
2013    7088
2014    7274
2015    7185
2016    7219
2017    7204
2018    7202
2019    7228
Name: count, dtype: int64


In [92]:
# WI-05 — Cell 73: Rebuild T+4 Training Sample Correctly

sample_parts_t4 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    train_df = batch_df[
        batch_df["target_date"].dt.year.between(2013, 2019)
    ]

    train_df["target_year"] = train_df["target_date"].dt.year

    for group_index, (_, group) in enumerate(
        train_df.groupby(["cell_id", "target_year"], sort=False)
    ):
        sampled_group = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + group_index
        )
        sample_parts_t4.append(sampled_group)

    del batch_df, train_df

training_sample_df_t4 = pd.concat(
    sample_parts_t4,
    ignore_index=True
)

training_sample_df_t4.drop(columns=["target_year"], inplace=True)

print("CORRECTED T+4 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t4))
print("Unique grid cells:", training_sample_df_t4["cell_id"].nunique())
print("Missing targets:", training_sample_df_t4[target_col].isna().sum())
print(
    "Wet-day rate (%):",
    round((training_sample_df_t4[target_col] >= 1.0).mean() * 100, 4)
)
print("Target maximum:", training_sample_df_t4[target_col].max())
print("Target-date years:")
print(
    training_sample_df_t4["target_date"]
    .dt.year.value_counts()
    .sort_index()
)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1963849549.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["target_year"] = train_df["target_date"].dt.year
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1963849549.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["target_year"] = train_df["target_date"].dt.year
C:\Users\gsr58\AppData\Local\Temp\ipykernel_11020\1963849549.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usin

CORRECTED T+4 TRAINING SAMPLE
Rows: 352800
Unique grid cells: 2520
Missing targets: 0
Wet-day rate (%): 38.5065
Target maximum: 488.99
Target-date years:
target_date
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
2018    50400
2019    50400
Name: count, dtype: int64


In [93]:
# WI-05 — Cell 74: Train T+4 Two-Stage HGB Models

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
import joblib

X_train_t4 = training_sample_df_t4[model_features_t4]
y_train_t4 = training_sample_df_t4[target_col]

wet_train_t4 = (y_train_t4 >= 1.0).astype(int)

classifier_t4 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

regressor_t4 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

classifier_t4.fit(X_train_t4, wet_train_t4)

wet_mask_t4 = y_train_t4 >= 1.0

regressor_t4.fit(
    X_train_t4.loc[wet_mask_t4],
    y_train_t4.loc[wet_mask_t4]
)

candidate_dir = REPORT_DIR.parent / "candidate_models"
candidate_dir.mkdir(parents=True, exist_ok=True)

classifier_path_t4 = candidate_dir / "WI_05_hgb_wet_classifier_t4_candidate.joblib"
regressor_path_t4 = candidate_dir / "WI_05_hgb_wet_amount_regressor_t4_candidate.joblib"

joblib.dump(classifier_t4, classifier_path_t4)
joblib.dump(regressor_t4, regressor_path_t4)

print("T+4 MODEL TRAINING COMPLETE")
print("Training rows:", len(X_train_t4))
print("Features:", X_train_t4.shape[1])
print("Wet observations:", int(wet_mask_t4.sum()))
print("Dry observations:", int((~wet_mask_t4).sum()))
print("Classifier:", classifier_path_t4)
print("Regressor:", regressor_path_t4)
print("Final holdout accessed: False")

T+4 MODEL TRAINING COMPLETE
Training rows: 352800
Features: 53
Wet observations: 135851
Dry observations: 216949
Classifier: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_classifier_t4_candidate.joblib
Regressor: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t4_candidate.joblib
Final holdout accessed: False


In [95]:
# WI-05 — Cell 75: T+4 Full Validation (Corrected)

actual_parts_t4 = []
probability_parts_t4 = []
conditional_amount_parts_t4 = []
persistence_parts_t4 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t4 + [target_col])

    if len(val_df) > 0:
        X_val_batch = val_df[model_features_t4]

        actual_parts_t4.append(val_df[target_col].to_numpy())
        probability_parts_t4.append(
            classifier_t4.predict_proba(X_val_batch)[:, 1]
        )
        conditional_amount_parts_t4.append(
            regressor_t4.predict(X_val_batch)
        )

        # Forecast-origin day's observed rainfall (verify column name)
        persistence_parts_t4.append(
            val_df["PRECTOTCORR_lag_1"].to_numpy()
        )

    del batch_df, val_df

y_true_t4 = np.concatenate(actual_parts_t4)
y_probability_t4 = np.concatenate(probability_parts_t4)
y_conditional_t4 = np.concatenate(conditional_amount_parts_t4)
y_persistence_t4 = np.concatenate(persistence_parts_t4)

y_pred_t4 = y_probability_t4 * y_conditional_t4

t4_mae = mean_absolute_error(y_true_t4, y_pred_t4)
t4_rmse = np.sqrt(mean_squared_error(y_true_t4, y_pred_t4))

t4_persistence_mae = mean_absolute_error(y_true_t4, y_persistence_t4)
t4_persistence_rmse = np.sqrt(
    mean_squared_error(y_true_t4, y_persistence_t4)
)

print("T+4 FULL VALIDATION RESULTS")
print("Validation observations:", len(y_true_t4))
print("Candidate MAE:", round(t4_mae, 4))
print("Candidate RMSE:", round(t4_rmse, 4))
print("Persistence MAE:", round(t4_persistence_mae, 4))
print("Persistence RMSE:", round(t4_persistence_rmse, 4))
print(
    "MAE improvement over persistence (%):",
    round((1 - t4_mae / t4_persistence_mae) * 100, 2)
)
print("Final holdout accessed: False")

T+4 FULL VALIDATION RESULTS
Validation observations: 2761920
Candidate MAE: 3.61
Candidate RMSE: 8.5761
Persistence MAE: 4.6048
Persistence RMSE: 11.6422
MAE improvement over persistence (%): 21.6
Final holdout accessed: False


In [96]:
# WI-05 — Cell 76: Identify Rainfall Feature Columns

print([
    col for col in model_features_t4
    if "PRECTOTCORR" in col
])

['PRECTOTCORR_current', 'PRECTOTCORR_lag_1', 'PRECTOTCORR_lag_2', 'PRECTOTCORR_lag_3', 'PRECTOTCORR_lag_7', 'PRECTOTCORR_lag_14', 'PRECTOTCORR_lag_30', 'PRECTOTCORR_rolling_mean_3', 'PRECTOTCORR_rolling_std_3', 'PRECTOTCORR_rolling_mean_7', 'PRECTOTCORR_rolling_std_7', 'PRECTOTCORR_rolling_mean_14', 'PRECTOTCORR_rolling_std_14', 'PRECTOTCORR_rolling_mean_30', 'PRECTOTCORR_rolling_std_30']


In [97]:
# WI-05 — Cell 77: Correct T+4 Persistence Baseline

persistence_parts_t4 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=["PRECTOTCORR_current", target_col])

    if len(val_df) > 0:
        persistence_parts_t4.append(
            val_df["PRECTOTCORR_current"].to_numpy()
        )

    del batch_df, val_df

y_persistence_t4 = np.concatenate(persistence_parts_t4)

t4_persistence_mae = mean_absolute_error(
    y_true_t4, y_persistence_t4
)
t4_persistence_rmse = np.sqrt(
    mean_squared_error(y_true_t4, y_persistence_t4)
)

print("T+4 CORRECTED PERSISTENCE COMPARISON")
print("Candidate MAE:", round(t4_mae, 4))
print("Persistence MAE:", round(t4_persistence_mae, 4))
print("Candidate RMSE:", round(t4_rmse, 4))
print("Persistence RMSE:", round(t4_persistence_rmse, 4))
print(
    "MAE improvement (%):",
    round((1 - t4_mae / t4_persistence_mae) * 100, 2)
)
print("Final holdout accessed: False")

T+4 CORRECTED PERSISTENCE COMPARISON
Candidate MAE: 3.61
Persistence MAE: 4.5194
Candidate RMSE: 8.5761
Persistence RMSE: 11.5325
MAE improvement (%): 20.12
Final holdout accessed: False


In [98]:
# WI-05 — Cell 78: T+4 Full Baseline Comparison

t4_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": t4_mae,
        "rmse_mm": t4_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": t4_persistence_mae,
        "rmse_mm": t4_persistence_rmse
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.8382,
        "rmse_mm": 12.2575
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t4_baseline_comparison["mae_rank"] = (
    t4_baseline_comparison["mae_mm"].rank(method="min").astype(int)
)
t4_baseline_comparison["rmse_rank"] = (
    t4_baseline_comparison["rmse_mm"].rank(method="min").astype(int)
)

baseline_path_t4 = REPORT_DIR / "WI_05_t4_full_baseline_comparison.csv"
t4_baseline_comparison.to_csv(baseline_path_t4, index=False)

print("T+4 FULL BASELINE COMPARISON")
print(t4_baseline_comparison.to_string(index=False))
print("\nMAE improvement over persistence:",
      round((1 - t4_mae / t4_persistence_mae) * 100, 2), "%")
print("MAE improvement over climatology:",
      round((1 - t4_mae / 3.7649) * 100, 2), "%")
print("\nSaved:", baseline_path_t4)
print("Final holdout accessed: False")

T+4 FULL BASELINE COMPARISON
                         model  mae_mm   rmse_mm  mae_rank  rmse_rank
Two-stage HistGradientBoosting 3.60997  8.576050         1          1
                   Persistence 4.51940 11.532456         3          3
                Seasonal naive 4.83820 12.257500         4          4
          Training climatology 3.76490  9.062000         2          2

MAE improvement over persistence: 20.12 %
MAE improvement over climatology: 4.12 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t4_full_baseline_comparison.csv
Final holdout accessed: False


In [99]:
# WI-05 — Cell 79: T+4 Seasonal Evaluation

t4_validation_df = pd.DataFrame({
    "actual": y_true_t4,
    "prediction": y_pred_t4,
    "persistence": y_persistence_t4
})

# Target date = forecast origin date + 4 days
# Reconstruct target dates in the same batch order used during validation.
target_dates_parts_t4 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t4 + [target_col])

    if len(val_df) > 0:
        target_dates_parts_t4.append(
            val_df["target_date"].to_numpy()
        )

    del batch_df, val_df

t4_validation_df["target_date"] = np.concatenate(target_dates_parts_t4)

month = pd.to_datetime(t4_validation_df["target_date"]).dt.month

t4_validation_df["season"] = np.select(
    [
        month.isin([12, 1, 2]),
        month.isin([3, 4, 5]),
        month.isin([6, 7, 8, 9]),
        month.isin([10, 11])
    ],
    [
        "Winter",
        "Pre-monsoon",
        "Monsoon",
        "Post-monsoon"
    ],
    default="Unknown"
)

seasonal_rows_t4 = []

for season, group in t4_validation_df.groupby("season"):
    candidate_mae = mean_absolute_error(
        group["actual"], group["prediction"]
    )
    candidate_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["prediction"])
    )
    persistence_mae = mean_absolute_error(
        group["actual"], group["persistence"]
    )
    persistence_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["persistence"])
    )

    seasonal_rows_t4.append({
        "season": season,
        "observations": len(group),
        "actual_mean": group["actual"].mean(),
        "prediction_mean": group["prediction"].mean(),
        "candidate_mae": candidate_mae,
        "candidate_rmse": candidate_rmse,
        "persistence_mae": persistence_mae,
        "persistence_rmse": persistence_rmse,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100,
        "rmse_improvement_pct": (
            1 - candidate_rmse / persistence_rmse
        ) * 100
    })

t4_seasonal_validation = pd.DataFrame(seasonal_rows_t4)

print("T+4 SEASONAL VALIDATION")
print(t4_seasonal_validation.round(4).to_string(index=False))
print("Final holdout accessed: False")

T+4 SEASONAL VALIDATION
      season  observations  actual_mean  prediction_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct
     Monsoon        922320       7.4632           7.4088         6.3105         11.4523           7.8275           15.1516              19.3804               24.4154
Post-monsoon        461160       3.4938           3.1608         3.4101          8.3349           4.4675           11.6606              23.6688               28.5209
 Pre-monsoon        695520       2.3632           2.0666         2.4015          7.0452           2.9678            9.5171              19.0811               25.9729
      Winter        682920       1.0597           0.8992         1.3285          4.7809           1.6670            6.6158              20.3026               27.7344
Final holdout accessed: False


In [100]:
# WI-05 — Cell 80: Save T+4 Seasonal Comparison

t4_seasonal_path = REPORT_DIR / "WI_05_t4_seasonal_baseline_comparison.csv"

t4_seasonal_validation.to_csv(
    t4_seasonal_path,
    index=False
)

print("T+4 SEASONAL COMPARISON SAVED")
print("Path:", t4_seasonal_path)
print("Seasons:", len(t4_seasonal_validation))
print(
    "All seasons beat persistence:",
    bool(
        (t4_seasonal_validation["mae_improvement_pct"] > 0).all()
        and
        (t4_seasonal_validation["rmse_improvement_pct"] > 0).all()
    )
)
print("Final holdout accessed: False")

T+4 SEASONAL COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t4_seasonal_baseline_comparison.csv
Seasons: 4
All seasons beat persistence: True
Final holdout accessed: False


In [101]:
# WI-05 — Cell 81: T+4 Rainfall Intensity Evaluation

intensity_conditions_t4 = [
    y_true_t4 < 1,
    (y_true_t4 >= 1) & (y_true_t4 < 5),
    (y_true_t4 >= 5) & (y_true_t4 < 20),
    (y_true_t4 >= 20) & (y_true_t4 < 50),
    y_true_t4 >= 50
]

intensity_labels_t4 = [
    "Below 1 mm",
    "1-5 mm",
    "5-20 mm",
    "20-50 mm",
    "50+ mm"
]

intensity_rows_t4 = []

for label, mask in zip(intensity_labels_t4, intensity_conditions_t4):
    actual = y_true_t4[mask]
    candidate = y_pred_t4[mask]
    persistence = y_persistence_t4[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_rows_t4.append({
        "intensity_band": label,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true_t4) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100
    })

t4_intensity_comparison = pd.DataFrame(intensity_rows_t4)

print("T+4 RAINFALL INTENSITY COMPARISON")
print(t4_intensity_comparison.round(4).to_string(index=False))
print("\nFinal holdout accessed: False")

T+4 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.4594           1.4983               2.5931
        1-5 mm        575375          20.8324         3.3988           4.6978              27.6512
       5-20 mm        458822          16.6124         5.1547           8.4867              39.2614
      20-50 mm        105268           3.8114        19.9814          21.7201               8.0053
        50+ mm         18977           0.6871        63.5643          63.0502              -0.8155

Final holdout accessed: False


In [102]:
# WI-05 — Cell 82: T+4 Intensity and Heavy-Rainfall Diagnostics

t4_intensity_path = REPORT_DIR / "WI_05_t4_rainfall_intensity_comparison.csv"

t4_intensity_comparison.to_csv(
    t4_intensity_path,
    index=False
)

# Heavy rainfall: actual rainfall >= 50 mm
heavy_mask_t4 = y_true_t4 >= 50

heavy_actual_t4 = y_true_t4[heavy_mask_t4]
heavy_candidate_t4 = y_pred_t4[heavy_mask_t4]
heavy_persistence_t4 = y_persistence_t4[heavy_mask_t4]

heavy_results_t4 = {
    "observations": int(heavy_mask_t4.sum()),
    "actual_mean": float(heavy_actual_t4.mean()),
    "candidate_mean": float(heavy_candidate_t4.mean()),
    "persistence_mean": float(heavy_persistence_t4.mean()),
    "candidate_bias": float(np.mean(heavy_candidate_t4 - heavy_actual_t4)),
    "persistence_bias": float(np.mean(heavy_persistence_t4 - heavy_actual_t4)),
    "candidate_mae": float(mean_absolute_error(heavy_actual_t4, heavy_candidate_t4)),
    "persistence_mae": float(mean_absolute_error(heavy_actual_t4, heavy_persistence_t4)),
    "candidate_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t4, heavy_candidate_t4))),
    "persistence_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t4, heavy_persistence_t4))),
    "actual_p95": float(np.percentile(heavy_actual_t4, 95)),
    "candidate_p95": float(np.percentile(heavy_candidate_t4, 95)),
    "persistence_p95": float(np.percentile(heavy_persistence_t4, 95)),
    "actual_max": float(heavy_actual_t4.max()),
    "candidate_max": float(heavy_candidate_t4.max()),
    "persistence_max": float(heavy_persistence_t4.max())
}

print("T+4 INTENSITY COMPARISON SAVED")
print("Path:", t4_intensity_path)

print("\nT+4 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)")
for key, value in heavy_results_t4.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\nFinal holdout accessed: False")

T+4 INTENSITY COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t4_rainfall_intensity_comparison.csv

T+4 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)
observations: 18977
actual_mean: 77.4719
candidate_mean: 14.1108
persistence_mean: 16.9983
candidate_bias: -63.3611
persistence_bias: -60.4736
candidate_mae: 63.5643
persistence_mae: 63.0502
candidate_rmse: 74.2442
persistence_rmse: 75.0987
actual_p95: 142.4980
candidate_p95: 39.2729
persistence_p95: 60.3760
actual_max: 666.3700
candidate_max: 116.2748
persistence_max: 440.8500

Final holdout accessed: False


In [103]:
# WI-05 — Cell 83: Consolidate T+4 Findings

t4_findings = {
    "work_item": "WI-05",
    "target": "PRECTOTCORR",
    "horizon": "T+4",
    "model": "Two-stage HistGradientBoosting",
    "status": "provisional_candidate",
    "validation_observations": int(len(y_true_t4)),
    "validation_mae_mm": float(t4_mae),
    "validation_rmse_mm": float(t4_rmse),
    "baseline_comparison": t4_baseline_comparison.to_dict(orient="records"),
    "seasonal_comparison": t4_seasonal_validation.to_dict(orient="records"),
    "intensity_comparison": t4_intensity_comparison.to_dict(orient="records"),
    "heavy_rainfall_diagnostics": heavy_results_t4,
    "persistence_feature": "PRECTOTCORR_current",
    "final_holdout_accessed": False,
    "final_model_selected": False
}

t4_findings_path = REPORT_DIR / "WI_05_t4_consolidated_findings.json"

with open(t4_findings_path, "w", encoding="utf-8") as f:
    json.dump(
        t4_findings,
        f,
        indent=4,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

print("T+4 CONSOLIDATED FINDINGS SAVED")
print("Path:", t4_findings_path)
print("Status:", t4_findings["status"])
print("Validation observations:", t4_findings["validation_observations"])
print("Final model selected:", t4_findings["final_model_selected"])
print("Final holdout accessed:", t4_findings["final_holdout_accessed"])

T+4 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t4_consolidated_findings.json
Status: provisional_candidate
Validation observations: 2761920
Final model selected: False
Final holdout accessed: False


In [104]:
# WI-05 — Cell 84: T+5 Configuration

horizon = 5
target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

model_features_t5 = model_features_t2.copy()

print("WI-05 T+5 CONFIGURATION")
print("Horizon:", horizon)
print("Target:", target_col)
print("Number of features:", len(model_features_t5))
print("Feature consistency:",
      model_features_t5 == classifier_features == regressor_features)
print("Training period: 2013–2019")
print("Validation period: 2020–2022")
print("Final holdout: 2023–2024 (untouched)")

WI-05 T+5 CONFIGURATION
Horizon: 5
Target: PRECTOTCORR_target_t_plus_5
Number of features: 53
Feature consistency: True
Training period: 2013–2019
Validation period: 2020–2022
Final holdout: 2023–2024 (untouched)


In [105]:
# WI-05 — Cell 85: Build T+5 Training Sample

sample_parts_t5 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    train_df = batch_df[
        batch_df["target_date"].dt.year.between(2013, 2019)
    ].copy()

    train_df["target_year"] = train_df["target_date"].dt.year

    for group_index, (_, group) in enumerate(
        train_df.groupby(["cell_id", "target_year"], sort=False)
    ):
        sampled_group = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + group_index
        )
        sample_parts_t5.append(sampled_group)

    del batch_df, train_df

training_sample_df_t5 = pd.concat(
    sample_parts_t5,
    ignore_index=True
)

training_sample_df_t5.drop(columns=["target_year"], inplace=True)

print("T+5 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t5))
print("Unique grid cells:", training_sample_df_t5["cell_id"].nunique())
print("Missing targets:", training_sample_df_t5[target_col].isna().sum())
print(
    "Wet-day rate (%):",
    round((training_sample_df_t5[target_col] >= 1.0).mean() * 100, 4)
)
print("Target maximum:", training_sample_df_t5[target_col].max())
print("Target-date years:")
print(
    training_sample_df_t5["target_date"]
    .dt.year.value_counts()
    .sort_index()
)

T+5 TRAINING SAMPLE
Rows: 352800
Unique grid cells: 2520
Missing targets: 0
Wet-day rate (%): 38.5099
Target maximum: 488.99
Target-date years:
target_date
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
2018    50400
2019    50400
Name: count, dtype: int64


In [106]:
# WI-05 — Cell 86: Train T+5 Models

X_train_t5 = training_sample_df_t5[model_features_t5]
y_train_t5 = training_sample_df_t5[target_col]

wet_train_t5 = (y_train_t5 >= 1.0).astype(int)
wet_mask_t5 = y_train_t5 >= 1.0

classifier_t5 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

regressor_t5 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

classifier_t5.fit(X_train_t5, wet_train_t5)

regressor_t5.fit(
    X_train_t5.loc[wet_mask_t5],
    y_train_t5.loc[wet_mask_t5]
)

candidate_dir = REPORT_DIR.parent / "candidate_models"
candidate_dir.mkdir(parents=True, exist_ok=True)

classifier_path_t5 = candidate_dir / "WI_05_hgb_wet_classifier_t5_candidate.joblib"
regressor_path_t5 = candidate_dir / "WI_05_hgb_wet_amount_regressor_t5_candidate.joblib"

joblib.dump(classifier_t5, classifier_path_t5)
joblib.dump(regressor_t5, regressor_path_t5)

print("T+5 MODEL TRAINING COMPLETE")
print("Training rows:", len(X_train_t5))
print("Features:", X_train_t5.shape[1])
print("Wet observations:", int(wet_mask_t5.sum()))
print("Dry observations:", int((~wet_mask_t5).sum()))
print("Classifier:", classifier_path_t5)
print("Regressor:", regressor_path_t5)
print("Final holdout accessed: False")

T+5 MODEL TRAINING COMPLETE
Training rows: 352800
Features: 53
Wet observations: 135863
Dry observations: 216937
Classifier: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_classifier_t5_candidate.joblib
Regressor: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t5_candidate.joblib
Final holdout accessed: False


In [107]:
# WI-05 — Cell 87: T+5 Full Validation

actual_parts_t5 = []
probability_parts_t5 = []
conditional_amount_parts_t5 = []
persistence_parts_t5 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t5 + [target_col, "PRECTOTCORR_current"])

    if len(val_df) > 0:
        X_val_batch = val_df[model_features_t5]

        actual_parts_t5.append(val_df[target_col].to_numpy())
        probability_parts_t5.append(
            classifier_t5.predict_proba(X_val_batch)[:, 1]
        )
        conditional_amount_parts_t5.append(
            regressor_t5.predict(X_val_batch)
        )
        persistence_parts_t5.append(
            val_df["PRECTOTCORR_current"].to_numpy()
        )

    del batch_df, val_df

y_true_t5 = np.concatenate(actual_parts_t5)
y_probability_t5 = np.concatenate(probability_parts_t5)
y_conditional_t5 = np.concatenate(conditional_amount_parts_t5)
y_persistence_t5 = np.concatenate(persistence_parts_t5)

y_pred_t5 = y_probability_t5 * y_conditional_t5

t5_mae = mean_absolute_error(y_true_t5, y_pred_t5)
t5_rmse = np.sqrt(mean_squared_error(y_true_t5, y_pred_t5))

t5_persistence_mae = mean_absolute_error(y_true_t5, y_persistence_t5)
t5_persistence_rmse = np.sqrt(
    mean_squared_error(y_true_t5, y_persistence_t5)
)

print("T+5 FULL VALIDATION RESULTS")
print("Validation observations:", len(y_true_t5))
print("Candidate MAE:", round(t5_mae, 4))
print("Candidate RMSE:", round(t5_rmse, 4))
print("Persistence MAE:", round(t5_persistence_mae, 4))
print("Persistence RMSE:", round(t5_persistence_rmse, 4))
print(
    "MAE improvement over persistence (%):",
    round((1 - t5_mae / t5_persistence_mae) * 100, 2)
)
print("Final holdout accessed: False")

T+5 FULL VALIDATION RESULTS
Validation observations: 2761920
Candidate MAE: 3.6327
Candidate RMSE: 8.598
Persistence MAE: 4.6048
Persistence RMSE: 11.6422
MAE improvement over persistence (%): 21.11
Final holdout accessed: False


In [108]:
# WI-05 — Cell 88: T+5 Full Baseline Comparison

t5_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": t5_mae,
        "rmse_mm": t5_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": t5_persistence_mae,
        "rmse_mm": t5_persistence_rmse
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.8639,
        "rmse_mm": 12.3194
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t5_baseline_comparison["mae_rank"] = (
    t5_baseline_comparison["mae_mm"].rank(method="min").astype(int)
)
t5_baseline_comparison["rmse_rank"] = (
    t5_baseline_comparison["rmse_mm"].rank(method="min").astype(int)
)

baseline_path_t5 = REPORT_DIR / "WI_05_t5_full_baseline_comparison.csv"
t5_baseline_comparison.to_csv(baseline_path_t5, index=False)

print("T+5 FULL BASELINE COMPARISON")
print(t5_baseline_comparison.round(4).to_string(index=False))

print("\nMAE improvement over persistence:",
      round((1 - t5_mae / t5_persistence_mae) * 100, 2), "%")
print("MAE improvement over climatology:",
      round((1 - t5_mae / 3.7649) * 100, 2), "%")

print("\nSaved:", baseline_path_t5)
print("Final holdout accessed: False")

T+5 FULL BASELINE COMPARISON
                         model  mae_mm  rmse_mm  mae_rank  rmse_rank
Two-stage HistGradientBoosting  3.6327   8.5980         1          1
                   Persistence  4.6048  11.6422         3          3
                Seasonal naive  4.8639  12.3194         4          4
          Training climatology  3.7649   9.0620         2          2

MAE improvement over persistence: 21.11 %
MAE improvement over climatology: 3.51 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t5_full_baseline_comparison.csv
Final holdout accessed: False


In [109]:
# WI-05 — Cell 89: T+5 Seasonal Evaluation

t5_validation_df = pd.DataFrame({
    "actual": y_true_t5,
    "prediction": y_pred_t5,
    "persistence": y_persistence_t5
})

target_dates_parts_t5 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(subset=model_features_t5 + [target_col, "PRECTOTCORR_current"])

    if len(val_df) > 0:
        target_dates_parts_t5.append(
            val_df["target_date"].to_numpy()
        )

    del batch_df, val_df

t5_validation_df["target_date"] = np.concatenate(target_dates_parts_t5)

month = pd.to_datetime(t5_validation_df["target_date"]).dt.month

t5_validation_df["season"] = np.select(
    [
        month.isin([12, 1, 2]),
        month.isin([3, 4, 5]),
        month.isin([6, 7, 8, 9]),
        month.isin([10, 11])
    ],
    ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"],
    default="Unknown"
)

seasonal_rows_t5 = []

for season, group in t5_validation_df.groupby("season"):
    candidate_mae = mean_absolute_error(
        group["actual"], group["prediction"]
    )
    candidate_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["prediction"])
    )
    persistence_mae = mean_absolute_error(
        group["actual"], group["persistence"]
    )
    persistence_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["persistence"])
    )

    seasonal_rows_t5.append({
        "season": season,
        "observations": len(group),
        "actual_mean": group["actual"].mean(),
        "prediction_mean": group["prediction"].mean(),
        "candidate_mae": candidate_mae,
        "candidate_rmse": candidate_rmse,
        "persistence_mae": persistence_mae,
        "persistence_rmse": persistence_rmse,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100,
        "rmse_improvement_pct": (
            1 - candidate_rmse / persistence_rmse
        ) * 100
    })

t5_seasonal_validation = pd.DataFrame(seasonal_rows_t5)

print("T+5 SEASONAL VALIDATION")
print(t5_seasonal_validation.round(4).to_string(index=False))
print("Final holdout accessed: False")

T+5 SEASONAL VALIDATION
      season  observations  actual_mean  prediction_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct
     Monsoon        922320       7.4632           7.3973         6.3523         11.4976           7.9479           15.2588              20.0758               24.6494
Post-monsoon        461160       3.4938           3.1573         3.4263          8.3296           4.5691           11.7671              25.0121               29.2130
 Pre-monsoon        695520       2.3632           2.0554         2.4107          7.0472           3.0159            9.5931              20.0657               26.5381
      Winter        682920       1.0597           0.9042         1.3437          4.7964           1.7320            6.8183              22.4198               29.6532
Final holdout accessed: False


In [110]:
# WI-05 — Cell 90: Save T+5 Seasonal Comparison

t5_seasonal_path = REPORT_DIR / "WI_05_t5_seasonal_baseline_comparison.csv"

t5_seasonal_validation.to_csv(
    t5_seasonal_path,
    index=False
)

print("T+5 SEASONAL COMPARISON SAVED")
print("Path:", t5_seasonal_path)
print("Seasons:", len(t5_seasonal_validation))
print(
    "All seasons beat persistence:",
    bool(
        (t5_seasonal_validation["mae_improvement_pct"] > 0).all()
        and
        (t5_seasonal_validation["rmse_improvement_pct"] > 0).all()
    )
)
print("Final holdout accessed: False")

T+5 SEASONAL COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t5_seasonal_baseline_comparison.csv
Seasons: 4
All seasons beat persistence: True
Final holdout accessed: False


In [111]:
# WI-05 — Cell 91: T+5 Rainfall Intensity Evaluation

intensity_conditions_t5 = [
    y_true_t5 < 1,
    (y_true_t5 >= 1) & (y_true_t5 < 5),
    (y_true_t5 >= 5) & (y_true_t5 < 20),
    (y_true_t5 >= 20) & (y_true_t5 < 50),
    y_true_t5 >= 50
]

intensity_labels_t5 = [
    "Below 1 mm",
    "1-5 mm",
    "5-20 mm",
    "20-50 mm",
    "50+ mm"
]

intensity_rows_t5 = []

for label, mask in zip(intensity_labels_t5, intensity_conditions_t5):
    actual = y_true_t5[mask]
    candidate = y_pred_t5[mask]
    persistence = y_persistence_t5[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_rows_t5.append({
        "intensity_band": label,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true_t5) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100
    })

t5_intensity_comparison = pd.DataFrame(intensity_rows_t5)

print("T+5 RAINFALL INTENSITY COMPARISON")
print(t5_intensity_comparison.round(4).to_string(index=False))
print("\nFinal holdout accessed: False")

T+5 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.4891           1.5874               6.1910
        1-5 mm        575375          20.8324         3.3725           4.7721              29.3283
       5-20 mm        458822          16.6124         5.1838           8.5068              39.0635
      20-50 mm        105268           3.8114        20.1237          22.0192               8.6085
        50+ mm         18977           0.6871        63.6622          63.5390              -0.1940

Final holdout accessed: False


In [112]:
# WI-05 — Cell 92: T+5 Intensity and Heavy-Rainfall Diagnostics

t5_intensity_path = REPORT_DIR / "WI_05_t5_rainfall_intensity_comparison.csv"

t5_intensity_comparison.to_csv(
    t5_intensity_path,
    index=False
)

heavy_mask_t5 = y_true_t5 >= 50

heavy_actual_t5 = y_true_t5[heavy_mask_t5]
heavy_candidate_t5 = y_pred_t5[heavy_mask_t5]
heavy_persistence_t5 = y_persistence_t5[heavy_mask_t5]

heavy_results_t5 = {
    "observations": int(heavy_mask_t5.sum()),
    "actual_mean": float(heavy_actual_t5.mean()),
    "candidate_mean": float(heavy_candidate_t5.mean()),
    "persistence_mean": float(heavy_persistence_t5.mean()),
    "candidate_bias": float(np.mean(heavy_candidate_t5 - heavy_actual_t5)),
    "persistence_bias": float(np.mean(heavy_persistence_t5 - heavy_actual_t5)),
    "candidate_mae": float(mean_absolute_error(heavy_actual_t5, heavy_candidate_t5)),
    "persistence_mae": float(mean_absolute_error(heavy_actual_t5, heavy_persistence_t5)),
    "candidate_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t5, heavy_candidate_t5))),
    "persistence_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t5, heavy_persistence_t5))),
    "actual_p95": float(np.percentile(heavy_actual_t5, 95)),
    "candidate_p95": float(np.percentile(heavy_candidate_t5, 95)),
    "persistence_p95": float(np.percentile(heavy_persistence_t5, 95)),
    "actual_max": float(heavy_actual_t5.max()),
    "candidate_max": float(heavy_candidate_t5.max()),
    "persistence_max": float(heavy_persistence_t5.max())
}

print("T+5 INTENSITY COMPARISON SAVED")
print("Path:", t5_intensity_path)

print("\nT+5 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)")
for key, value in heavy_results_t5.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\nFinal holdout accessed: False")

T+5 INTENSITY COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t5_rainfall_intensity_comparison.csv

T+5 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)
observations: 18977
actual_mean: 77.4719
candidate_mean: 14.0348
persistence_mean: 16.2209
candidate_bias: -63.4371
persistence_bias: -61.2510
candidate_mae: 63.6622
persistence_mae: 63.5390
candidate_rmse: 74.2972
persistence_rmse: 75.4105
actual_p95: 142.4980
candidate_p95: 40.0633
persistence_p95: 59.5140
actual_max: 666.3700
candidate_max: 122.1049
persistence_max: 410.0500

Final holdout accessed: False


In [113]:
# WI-05 — Cell 93: Consolidate T+5 Findings

t5_findings = {
    "work_item": "WI-05",
    "target": "PRECTOTCORR",
    "horizon": "T+5",
    "model": "Two-stage HistGradientBoosting",
    "status": "provisional_candidate",
    "validation_observations": int(len(y_true_t5)),
    "validation_mae_mm": float(t5_mae),
    "validation_rmse_mm": float(t5_rmse),
    "baseline_comparison": t5_baseline_comparison.to_dict(orient="records"),
    "seasonal_comparison": t5_seasonal_validation.to_dict(orient="records"),
    "intensity_comparison": t5_intensity_comparison.to_dict(orient="records"),
    "heavy_rainfall_diagnostics": heavy_results_t5,
    "persistence_feature": "PRECTOTCORR_current",
    "final_holdout_accessed": False,
    "final_model_selected": False
}

t5_findings_path = REPORT_DIR / "WI_05_t5_consolidated_findings.json"

with open(t5_findings_path, "w", encoding="utf-8") as f:
    json.dump(
        t5_findings,
        f,
        indent=4,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

print("T+5 CONSOLIDATED FINDINGS SAVED")
print("Path:", t5_findings_path)
print("Status:", t5_findings["status"])
print("Validation observations:", t5_findings["validation_observations"])
print("Final model selected:", t5_findings["final_model_selected"])
print("Final holdout accessed:", t5_findings["final_holdout_accessed"])

T+5 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t5_consolidated_findings.json
Status: provisional_candidate
Validation observations: 2761920
Final model selected: False
Final holdout accessed: False


In [114]:
# WI-05 — Cell 94: T+6 Configuration

horizon = 6
target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

model_features_t6 = model_features_t2.copy()

print("WI-05 T+6 CONFIGURATION")
print("Horizon:", horizon)
print("Target:", target_col)
print("Number of features:", len(model_features_t6))
print(
    "Feature consistency:",
    model_features_t6 == classifier_features == regressor_features
)
print("Training period: 2013–2019")
print("Validation period: 2020–2022")
print("Final holdout: 2023–2024 (untouched)")

WI-05 T+6 CONFIGURATION
Horizon: 6
Target: PRECTOTCORR_target_t_plus_6
Number of features: 53
Feature consistency: True
Training period: 2013–2019
Validation period: 2020–2022
Final holdout: 2023–2024 (untouched)


In [115]:
# WI-05 — Cell 95: Build T+6 Training Sample

sample_parts_t6 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    train_df = batch_df[
        batch_df["target_date"].dt.year.between(2013, 2019)
    ].copy()

    train_df["target_year"] = train_df["target_date"].dt.year

    for group_index, (_, group) in enumerate(
        train_df.groupby(["cell_id", "target_year"], sort=False)
    ):
        sampled_group = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + group_index
        )
        sample_parts_t6.append(sampled_group)

    del batch_df, train_df

training_sample_df_t6 = pd.concat(
    sample_parts_t6,
    ignore_index=True
)

training_sample_df_t6.drop(columns=["target_year"], inplace=True)

print("T+6 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t6))
print("Unique grid cells:", training_sample_df_t6["cell_id"].nunique())
print("Missing targets:", training_sample_df_t6[target_col].isna().sum())
print(
    "Wet-day rate (%):",
    round((training_sample_df_t6[target_col] >= 1.0).mean() * 100, 4)
)
print("Target maximum:", training_sample_df_t6[target_col].max())
print("Target-date years:")
print(
    training_sample_df_t6["target_date"]
    .dt.year.value_counts()
    .sort_index()
)

T+6 TRAINING SAMPLE
Rows: 352800
Unique grid cells: 2520
Missing targets: 0
Wet-day rate (%): 38.521
Target maximum: 488.99
Target-date years:
target_date
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
2018    50400
2019    50400
Name: count, dtype: int64


In [116]:
# WI-05 — Cell 96: Train T+6 Models

X_train_t6 = training_sample_df_t6[model_features_t6]
y_train_t6 = training_sample_df_t6[target_col]

wet_train_t6 = (y_train_t6 >= 1.0).astype(int)
wet_mask_t6 = y_train_t6 >= 1.0

classifier_t6 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

regressor_t6 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

classifier_t6.fit(X_train_t6, wet_train_t6)

regressor_t6.fit(
    X_train_t6.loc[wet_mask_t6],
    y_train_t6.loc[wet_mask_t6]
)

candidate_dir = REPORT_DIR.parent / "candidate_models"
candidate_dir.mkdir(parents=True, exist_ok=True)

classifier_path_t6 = candidate_dir / "WI_05_hgb_wet_classifier_t6_candidate.joblib"
regressor_path_t6 = candidate_dir / "WI_05_hgb_wet_amount_regressor_t6_candidate.joblib"

joblib.dump(classifier_t6, classifier_path_t6)
joblib.dump(regressor_t6, regressor_path_t6)

print("T+6 MODEL TRAINING COMPLETE")
print("Training rows:", len(X_train_t6))
print("Features:", X_train_t6.shape[1])
print("Wet observations:", int(wet_mask_t6.sum()))
print("Dry observations:", int((~wet_mask_t6).sum()))
print("Classifier:", classifier_path_t6)
print("Regressor:", regressor_path_t6)
print("Final holdout accessed: False")

T+6 MODEL TRAINING COMPLETE
Training rows: 352800
Features: 53
Wet observations: 135902
Dry observations: 216898
Classifier: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_classifier_t6_candidate.joblib
Regressor: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t6_candidate.joblib
Final holdout accessed: False


In [117]:
# WI-05 — Cell 97: T+6 Full Validation

actual_parts_t6 = []
probability_parts_t6 = []
conditional_amount_parts_t6 = []
persistence_parts_t6 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(
        subset=model_features_t6 + [target_col, "PRECTOTCORR_current"]
    )

    if len(val_df) > 0:
        X_val_batch = val_df[model_features_t6]

        actual_parts_t6.append(val_df[target_col].to_numpy())
        probability_parts_t6.append(
            classifier_t6.predict_proba(X_val_batch)[:, 1]
        )
        conditional_amount_parts_t6.append(
            regressor_t6.predict(X_val_batch)
        )
        persistence_parts_t6.append(
            val_df["PRECTOTCORR_current"].to_numpy()
        )

    del batch_df, val_df

y_true_t6 = np.concatenate(actual_parts_t6)
y_probability_t6 = np.concatenate(probability_parts_t6)
y_conditional_t6 = np.concatenate(conditional_amount_parts_t6)
y_persistence_t6 = np.concatenate(persistence_parts_t6)

y_pred_t6 = y_probability_t6 * y_conditional_t6

t6_mae = mean_absolute_error(y_true_t6, y_pred_t6)
t6_rmse = np.sqrt(mean_squared_error(y_true_t6, y_pred_t6))

t6_persistence_mae = mean_absolute_error(y_true_t6, y_persistence_t6)
t6_persistence_rmse = np.sqrt(
    mean_squared_error(y_true_t6, y_persistence_t6)
)

print("T+6 FULL VALIDATION RESULTS")
print("Validation observations:", len(y_true_t6))
print("Candidate MAE:", round(t6_mae, 4))
print("Candidate RMSE:", round(t6_rmse, 4))
print("Persistence MAE:", round(t6_persistence_mae, 4))
print("Persistence RMSE:", round(t6_persistence_rmse, 4))
print(
    "MAE improvement over persistence (%):",
    round((1 - t6_mae / t6_persistence_mae) * 100, 2)
)
print("Final holdout accessed: False")

T+6 FULL VALIDATION RESULTS
Validation observations: 2761920
Candidate MAE: 3.6504
Candidate RMSE: 8.619
Persistence MAE: 4.6495
Persistence RMSE: 11.6799
MAE improvement over persistence (%): 21.49
Final holdout accessed: False


In [118]:
# WI-05 — Cell 98: T+6 Full Baseline Comparison

t6_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HistGradientBoosting",
        "mae_mm": t6_mae,
        "rmse_mm": t6_rmse
    },
    {
        "model": "Persistence",
        "mae_mm": t6_persistence_mae,
        "rmse_mm": t6_persistence_rmse
    },
    {
        "model": "Seasonal naive",
        "mae_mm": 4.8695,
        "rmse_mm": 12.3261
    },
    {
        "model": "Training climatology",
        "mae_mm": 3.7649,
        "rmse_mm": 9.0620
    }
])

t6_baseline_comparison["mae_rank"] = (
    t6_baseline_comparison["mae_mm"].rank(method="min").astype(int)
)
t6_baseline_comparison["rmse_rank"] = (
    t6_baseline_comparison["rmse_mm"].rank(method="min").astype(int)
)

baseline_path_t6 = REPORT_DIR / "WI_05_t6_full_baseline_comparison.csv"
t6_baseline_comparison.to_csv(baseline_path_t6, index=False)

print("T+6 FULL BASELINE COMPARISON")
print(t6_baseline_comparison.round(4).to_string(index=False))

print("\nMAE improvement over persistence:",
      round((1 - t6_mae / t6_persistence_mae) * 100, 2), "%")
print("MAE improvement over climatology:",
      round((1 - t6_mae / 3.7649) * 100, 2), "%")

print("\nSaved:", baseline_path_t6)
print("Final holdout accessed: False")

T+6 FULL BASELINE COMPARISON
                         model  mae_mm  rmse_mm  mae_rank  rmse_rank
Two-stage HistGradientBoosting  3.6504   8.6190         1          1
                   Persistence  4.6495  11.6799         3          3
                Seasonal naive  4.8695  12.3261         4          4
          Training climatology  3.7649   9.0620         2          2

MAE improvement over persistence: 21.49 %
MAE improvement over climatology: 3.04 %

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t6_full_baseline_comparison.csv
Final holdout accessed: False


In [119]:
# WI-05 — Cell 99: T+6 Seasonal Evaluation

t6_validation_df = pd.DataFrame({
    "actual": y_true_t6,
    "prediction": y_pred_t6,
    "persistence": y_persistence_t6
})

target_dates_parts_t6 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(horizon, unit="D")
    )

    val_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ].dropna(
        subset=model_features_t6 + [target_col, "PRECTOTCORR_current"]
    )

    if len(val_df) > 0:
        target_dates_parts_t6.append(
            val_df["target_date"].to_numpy()
        )

    del batch_df, val_df

t6_validation_df["target_date"] = np.concatenate(target_dates_parts_t6)

month = pd.to_datetime(t6_validation_df["target_date"]).dt.month

t6_validation_df["season"] = np.select(
    [
        month.isin([12, 1, 2]),
        month.isin([3, 4, 5]),
        month.isin([6, 7, 8, 9]),
        month.isin([10, 11])
    ],
    ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"],
    default="Unknown"
)

seasonal_rows_t6 = []

for season, group in t6_validation_df.groupby("season"):
    candidate_mae = mean_absolute_error(
        group["actual"], group["prediction"]
    )
    candidate_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["prediction"])
    )
    persistence_mae = mean_absolute_error(
        group["actual"], group["persistence"]
    )
    persistence_rmse = np.sqrt(
        mean_squared_error(group["actual"], group["persistence"])
    )

    seasonal_rows_t6.append({
        "season": season,
        "observations": len(group),
        "actual_mean": group["actual"].mean(),
        "prediction_mean": group["prediction"].mean(),
        "candidate_mae": candidate_mae,
        "candidate_rmse": candidate_rmse,
        "persistence_mae": persistence_mae,
        "persistence_rmse": persistence_rmse,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100,
        "rmse_improvement_pct": (
            1 - candidate_rmse / persistence_rmse
        ) * 100
    })

t6_seasonal_validation = pd.DataFrame(seasonal_rows_t6)

print("T+6 SEASONAL VALIDATION")
print(t6_seasonal_validation.round(4).to_string(index=False))
print("Final holdout accessed: False")

T+6 SEASONAL VALIDATION
      season  observations  actual_mean  prediction_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct
     Monsoon        922320       7.4632           7.4079         6.3869         11.5333           8.0410           15.2946              20.5700               24.5924
Post-monsoon        461160       3.4938           3.1832         3.4493          8.3413           4.6181           11.7327              25.3092               28.9051
 Pre-monsoon        695520       2.3632           2.0458         2.4147          7.0596           3.0112            9.6508              19.8098               26.8499
      Winter        682920       1.0597           0.9075         1.3488          4.8012           1.7590            6.9271              23.3187               30.6905
Final holdout accessed: False


In [120]:
# WI-05 — Cell 100: Save T+6 Seasonal Comparison

t6_seasonal_path = REPORT_DIR / "WI_05_t6_seasonal_baseline_comparison.csv"

t6_seasonal_validation.to_csv(
    t6_seasonal_path,
    index=False
)

print("T+6 SEASONAL COMPARISON SAVED")
print("Path:", t6_seasonal_path)
print("Seasons:", len(t6_seasonal_validation))
print(
    "All seasons beat persistence:",
    bool(
        (t6_seasonal_validation["mae_improvement_pct"] > 0).all()
        and
        (t6_seasonal_validation["rmse_improvement_pct"] > 0).all()
    )
)
print("Final holdout accessed: False")

T+6 SEASONAL COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t6_seasonal_baseline_comparison.csv
Seasons: 4
All seasons beat persistence: True
Final holdout accessed: False


In [121]:
# WI-05 — Cell 101: T+6 Rainfall Intensity Evaluation

intensity_conditions_t6 = [
    y_true_t6 < 1,
    (y_true_t6 >= 1) & (y_true_t6 < 5),
    (y_true_t6 >= 5) & (y_true_t6 < 20),
    (y_true_t6 >= 20) & (y_true_t6 < 50),
    y_true_t6 >= 50
]

intensity_labels_t6 = [
    "Below 1 mm",
    "1-5 mm",
    "5-20 mm",
    "20-50 mm",
    "50+ mm"
]

intensity_rows_t6 = []

for label, mask in zip(intensity_labels_t6, intensity_conditions_t6):
    actual = y_true_t6[mask]
    candidate = y_pred_t6[mask]
    persistence = y_persistence_t6[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_rows_t6.append({
        "intensity_band": label,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true_t6) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100
    })

t6_intensity_comparison = pd.DataFrame(intensity_rows_t6)

print("T+6 RAINFALL INTENSITY COMPARISON")
print(t6_intensity_comparison.round(4).to_string(index=False))
print("\nFinal holdout accessed: False")

T+6 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.5092           1.6426               8.1237
        1-5 mm        575375          20.8324         3.3832           4.7521              28.8052
       5-20 mm        458822          16.6124         5.1946           8.5626              39.3339
      20-50 mm        105268           3.8114        20.1440          22.1830               9.1916
        50+ mm         18977           0.6871        63.8456          63.7419              -0.1626

Final holdout accessed: False


In [122]:
# WI-05 — Cell 102: T+6 Intensity and Heavy-Rainfall Diagnostics

t6_intensity_path = REPORT_DIR / "WI_05_t6_rainfall_intensity_comparison.csv"

t6_intensity_comparison.to_csv(
    t6_intensity_path,
    index=False
)

heavy_mask_t6 = y_true_t6 >= 50

heavy_actual_t6 = y_true_t6[heavy_mask_t6]
heavy_candidate_t6 = y_pred_t6[heavy_mask_t6]
heavy_persistence_t6 = y_persistence_t6[heavy_mask_t6]

heavy_results_t6 = {
    "observations": int(heavy_mask_t6.sum()),
    "actual_mean": float(heavy_actual_t6.mean()),
    "candidate_mean": float(heavy_candidate_t6.mean()),
    "persistence_mean": float(heavy_persistence_t6.mean()),
    "candidate_bias": float(np.mean(heavy_candidate_t6 - heavy_actual_t6)),
    "persistence_bias": float(np.mean(heavy_persistence_t6 - heavy_actual_t6)),
    "candidate_mae": float(mean_absolute_error(heavy_actual_t6, heavy_candidate_t6)),
    "persistence_mae": float(mean_absolute_error(heavy_actual_t6, heavy_persistence_t6)),
    "candidate_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t6, heavy_candidate_t6))),
    "persistence_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t6, heavy_persistence_t6))),
    "actual_p95": float(np.percentile(heavy_actual_t6, 95)),
    "candidate_p95": float(np.percentile(heavy_candidate_t6, 95)),
    "persistence_p95": float(np.percentile(heavy_persistence_t6, 95)),
    "actual_max": float(heavy_actual_t6.max()),
    "candidate_max": float(heavy_candidate_t6.max()),
    "persistence_max": float(heavy_persistence_t6.max())
}

print("T+6 INTENSITY COMPARISON SAVED")
print("Path:", t6_intensity_path)

print("\nT+6 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)")
for key, value in heavy_results_t6.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\nFinal holdout accessed: False")

T+6 INTENSITY COMPARISON SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t6_rainfall_intensity_comparison.csv

T+6 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)
observations: 18977
actual_mean: 77.4719
candidate_mean: 13.8285
persistence_mean: 16.1630
candidate_bias: -63.6434
persistence_bias: -61.3089
candidate_mae: 63.8456
persistence_mae: 63.7419
candidate_rmse: 74.4223
persistence_rmse: 75.4055
actual_p95: 142.4980
candidate_p95: 38.3718
persistence_p95: 59.2440
actual_max: 666.3700
candidate_max: 118.1988
persistence_max: 383.4000

Final holdout accessed: False


In [123]:
# WI-05 — Cell 103: Consolidate T+6 Findings

t6_findings = {
    "work_item": "WI-05",
    "target": "PRECTOTCORR",
    "horizon": "T+6",
    "model": "Two-stage HistGradientBoosting",
    "status": "provisional_candidate",
    "validation_observations": int(len(y_true_t6)),
    "validation_mae_mm": float(t6_mae),
    "validation_rmse_mm": float(t6_rmse),
    "baseline_comparison": t6_baseline_comparison.to_dict(orient="records"),
    "seasonal_comparison": t6_seasonal_validation.to_dict(orient="records"),
    "intensity_comparison": t6_intensity_comparison.to_dict(orient="records"),
    "heavy_rainfall_diagnostics": heavy_results_t6,
    "persistence_feature": "PRECTOTCORR_current",
    "final_holdout_accessed": False,
    "final_model_selected": False
}

t6_findings_path = REPORT_DIR / "WI_05_t6_consolidated_findings.json"

with open(t6_findings_path, "w", encoding="utf-8") as f:
    json.dump(
        t6_findings,
        f,
        indent=4,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

print("T+6 CONSOLIDATED FINDINGS SAVED")
print("Path:", t6_findings_path)
print("Status:", t6_findings["status"])
print("Validation observations:", t6_findings["validation_observations"])
print("Final model selected:", t6_findings["final_model_selected"])
print("Final holdout accessed:", t6_findings["final_holdout_accessed"])

T+6 CONSOLIDATED FINDINGS SAVED
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t6_consolidated_findings.json
Status: provisional_candidate
Validation observations: 2761920
Final model selected: False
Final holdout accessed: False


In [124]:
# WI-05 — Cell 104: Configure T+7 Rainfall Model

horizon = 7
target_col_t7 = "PRECTOTCORR_target_t_plus_7"

model_features_t7 = model_features_t2.copy()

print("T+7 CONFIGURATION")
print("Target:", target_col_t7)
print("Horizon:", horizon)
print("Predictor count:", len(model_features_t7))
print("Feature consistency:", model_features_t7 == model_features_t2)
print("Training years: 2013–2019")
print("Validation years: 2020–2022")
print("Final holdout years: 2023–2024 (untouched)")
print("Persistence feature: PRECTOTCORR_current")

T+7 CONFIGURATION
Target: PRECTOTCORR_target_t_plus_7
Horizon: 7
Predictor count: 53
Feature consistency: True
Training years: 2013–2019
Validation years: 2020–2022
Final holdout years: 2023–2024 (untouched)
Persistence feature: PRECTOTCORR_current


In [126]:
# WI-05 — Cell 105: Build T+7 Training Sample (Corrected)

sample_parts_t7 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(7, unit="D")
    )
    batch_df["target_year"] = batch_df["target_date"].dt.year

    batch_df = batch_df[
        batch_df["target_year"].between(2013, 2019)
    ]

    required_cols = (
        ["cell_id", "date", "target_year", target_col_t7]
        + model_features_t7
    )

    batch_df = batch_df.dropna(subset=required_cols)

    for (cell_id, target_year), group in batch_df.groupby(
        ["cell_id", "target_year"],
        sort=False
    ):
        sample_n = min(20, len(group))

        if sample_n == 0:
            continue

        random_state = 42 + batch_index * 1000 + int(target_year)

        sampled_group = group.sample(
            n=sample_n,
            random_state=random_state
        )

        sample_parts_t7.append(
            sampled_group[required_cols]
        )

    del batch_df

training_sample_df_t7 = pd.concat(
    sample_parts_t7,
    ignore_index=True
)

training_sample_df_t7["is_wet"] = (
    training_sample_df_t7[target_col_t7] >= 1.0
).astype("int8")

print("T+7 TRAINING SAMPLE")
print("Rows:", len(training_sample_df_t7))
print("Unique cells:", training_sample_df_t7["cell_id"].nunique())
print("Target years:", sorted(training_sample_df_t7["target_year"].unique()))
print("Missing target values:", training_sample_df_t7[target_col_t7].isna().sum())
print("Wet-day rate (%):", round(training_sample_df_t7["is_wet"].mean() * 100, 4))
print("Target maximum:", training_sample_df_t7[target_col_t7].max())
print("Final holdout accessed: False")

T+7 TRAINING SAMPLE
Rows: 352800
Unique cells: 2520
Target years: [np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Missing target values: 0
Wet-day rate (%): 39.055
Target maximum: 393.88
Final holdout accessed: False


In [127]:
# WI-05 — Cell 106: Train T+7 Rainfall Models

X_train_t7 = training_sample_df_t7[model_features_t7]
y_train_t7 = training_sample_df_t7[target_col_t7]
y_wet_t7 = training_sample_df_t7["is_wet"]

wet_mask_t7 = y_wet_t7 == 1

classifier_t7 = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

regressor_t7 = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

classifier_t7.fit(X_train_t7, y_wet_t7)

regressor_t7.fit(
    X_train_t7.loc[wet_mask_t7],
    y_train_t7.loc[wet_mask_t7]
)

candidate_models_dir = REPORT_DIR / "candidate_models"
candidate_models_dir.mkdir(parents=True, exist_ok=True)

classifier_path_t7 = (
    candidate_models_dir / "WI_05_hgb_wet_classifier_t7_candidate.joblib"
)

regressor_path_t7 = (
    candidate_models_dir / "WI_05_hgb_wet_amount_regressor_t7_candidate.joblib"
)

joblib.dump(classifier_t7, classifier_path_t7)
joblib.dump(regressor_t7, regressor_path_t7)

print("T+7 MODEL TRAINING COMPLETE")
print("Training rows:", len(X_train_t7))
print("Predictors:", X_train_t7.shape[1])
print("Wet observations:", int(wet_mask_t7.sum()))
print("Dry observations:", int((~wet_mask_t7).sum()))
print("Classifier saved:", classifier_path_t7)
print("Regressor saved:", regressor_path_t7)
print("Final holdout accessed: False")

T+7 MODEL TRAINING COMPLETE
Training rows: 352800
Predictors: 53
Wet observations: 137786
Dry observations: 215014
Classifier saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_classifier_t7_candidate.joblib
Regressor saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\candidate_models\WI_05_hgb_wet_amount_regressor_t7_candidate.joblib
Final holdout accessed: False


In [128]:
# WI-05 — Cell 107: Full T+7 Validation

actual_parts_t7 = []
probability_parts_t7 = []
conditional_amount_parts_t7 = []
persistence_parts_t7 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(7, unit="D")
    )

    batch_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ]

    required_cols = (
        model_features_t7
        + [target_col_t7, "PRECTOTCORR_current"]
    )

    batch_df = batch_df.dropna(subset=required_cols)

    if len(batch_df) == 0:
        del batch_df
        continue

    X_batch = batch_df[model_features_t7]

    probabilities = classifier_t7.predict_proba(X_batch)[:, 1]
    conditional_amounts = regressor_t7.predict(X_batch)

    predicted_amounts = np.where(
        probabilities >= 0.40,
        conditional_amounts,
        0.0
    )

    actual_parts_t7.append(
        batch_df[target_col_t7].to_numpy()
    )
    probability_parts_t7.append(probabilities)
    conditional_amount_parts_t7.append(conditional_amounts)
    persistence_parts_t7.append(
        batch_df["PRECTOTCORR_current"].to_numpy()
    )

    del batch_df, X_batch

y_true_t7 = np.concatenate(actual_parts_t7)
y_probability_t7 = np.concatenate(probability_parts_t7)
y_conditional_t7 = np.concatenate(conditional_amount_parts_t7)
y_persistence_t7 = np.concatenate(persistence_parts_t7)

y_pred_t7 = np.where(
    y_probability_t7 >= 0.40,
    y_conditional_t7,
    0.0
)

t7_mae = mean_absolute_error(y_true_t7, y_pred_t7)
t7_rmse = np.sqrt(mean_squared_error(y_true_t7, y_pred_t7))

t7_persistence_mae = mean_absolute_error(
    y_true_t7, y_persistence_t7
)
t7_persistence_rmse = np.sqrt(
    mean_squared_error(y_true_t7, y_persistence_t7)
)

print("T+7 FULL VALIDATION")
print("Validation observations:", len(y_true_t7))
print("Candidate MAE:", round(t7_mae, 4))
print("Candidate RMSE:", round(t7_rmse, 4))
print("Persistence MAE:", round(t7_persistence_mae, 4))
print("Persistence RMSE:", round(t7_persistence_rmse, 4))

print(
    "MAE improvement (%):",
    round((1 - t7_mae / t7_persistence_mae) * 100, 4)
)
print(
    "RMSE improvement (%):",
    round((1 - t7_rmse / t7_persistence_rmse) * 100, 4)
)

print("Final holdout accessed: False")

T+7 FULL VALIDATION
Validation observations: 2761920
Candidate MAE: 3.9909
Candidate RMSE: 8.8456
Persistence MAE: 4.6772
Persistence RMSE: 11.6909
MAE improvement (%): 14.6745
RMSE improvement (%): 24.3379
Final holdout accessed: False


In [129]:
# WI-05 — Cell 108: T+7 Full Baseline Comparison

t7_baseline_comparison = pd.DataFrame([
    {
        "model": "Two-stage HGB candidate",
        "mae": t7_mae,
        "rmse": t7_rmse
    },
    {
        "model": "Persistence",
        "mae": t7_persistence_mae,
        "rmse": t7_persistence_rmse
    },
    {
        "model": "Seasonal naive",
        "mae": 4.8473,
        "rmse": 12.2411
    },
    {
        "model": "Grid-cell calendar-day climatology",
        "mae": 3.7649,
        "rmse": 9.0620
    }
])

t7_baseline_comparison["mae_rank"] = (
    t7_baseline_comparison["mae"].rank(method="min").astype(int)
)

t7_baseline_comparison["rmse_rank"] = (
    t7_baseline_comparison["rmse"].rank(method="min").astype(int)
)

t7_baseline_path = (
    REPORT_DIR / "WI_05_t7_full_baseline_comparison.csv"
)

t7_baseline_comparison.to_csv(t7_baseline_path, index=False)

print("T+7 BASELINE COMPARISON")
print(t7_baseline_comparison.round(4).to_string(index=False))

print("\nCandidate vs climatology:")
print(
    "MAE difference (%):",
    round((1 - t7_mae / 3.7649) * 100, 4)
)
print(
    "RMSE improvement (%):",
    round((1 - t7_rmse / 9.0620) * 100, 4)
)

print("\nSaved:", t7_baseline_path)
print("Final holdout accessed: False")

T+7 BASELINE COMPARISON
                             model    mae    rmse  mae_rank  rmse_rank
           Two-stage HGB candidate 3.9909  8.8456         2          1
                       Persistence 4.6772 11.6909         3          3
                    Seasonal naive 4.8473 12.2411         4          4
Grid-cell calendar-day climatology 3.7649  9.0620         1          2

Candidate vs climatology:
MAE difference (%): -6.0024
RMSE improvement (%): 2.3883

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t7_full_baseline_comparison.csv
Final holdout accessed: False


In [130]:
# WI-05 — Cell 109: T+7 Seasonal Evaluation

seasonal_parts_t7 = []

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(7, unit="D")
    )

    batch_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ]

    required_cols = (
        model_features_t7
        + [target_col_t7, "PRECTOTCORR_current"]
    )

    batch_df = batch_df.dropna(subset=required_cols)

    if len(batch_df) == 0:
        del batch_df
        continue

    seasonal_parts_t7.append(
        batch_df[["target_date"]].copy()
    )

    del batch_df

t7_dates = pd.concat(seasonal_parts_t7, ignore_index=True)

if len(t7_dates) != len(y_true_t7):
    raise ValueError(
        f"Date alignment error: {len(t7_dates)} dates vs "
        f"{len(y_true_t7)} validation predictions"
    )

month_t7 = t7_dates["target_date"].dt.month

season_t7 = np.select(
    [
        month_t7.isin([12, 1, 2]),
        month_t7.isin([3, 4, 5]),
        month_t7.isin([6, 7, 8, 9]),
        month_t7.isin([10, 11])
    ],
    [
        "Winter",
        "Pre-monsoon",
        "Monsoon",
        "Post-monsoon"
    ],
    default="Unknown"
)

t7_seasonal_validation = pd.DataFrame({
    "season": season_t7,
    "actual": y_true_t7,
    "candidate": y_pred_t7,
    "persistence": y_persistence_t7
}).groupby("season").apply(
    lambda group: pd.Series({
        "observations": len(group),
        "actual_mean": group["actual"].mean(),
        "candidate_mean": group["candidate"].mean(),
        "candidate_mae": mean_absolute_error(
            group["actual"], group["candidate"]
        ),
        "candidate_rmse": np.sqrt(mean_squared_error(
            group["actual"], group["candidate"]
        )),
        "persistence_mae": mean_absolute_error(
            group["actual"], group["persistence"]
        ),
        "persistence_rmse": np.sqrt(mean_squared_error(
            group["actual"], group["persistence"]
        ))
    }),
    include_groups=False
).reset_index()

t7_seasonal_validation["mae_improvement_pct"] = (
    1 - t7_seasonal_validation["candidate_mae"]
    / t7_seasonal_validation["persistence_mae"]
) * 100

t7_seasonal_validation["rmse_improvement_pct"] = (
    1 - t7_seasonal_validation["candidate_rmse"]
    / t7_seasonal_validation["persistence_rmse"]
) * 100

t7_seasonal_path = (
    REPORT_DIR / "WI_05_t7_seasonal_baseline_comparison.csv"
)

t7_seasonal_validation.to_csv(t7_seasonal_path, index=False)

print("T+7 SEASONAL VALIDATION")
print(t7_seasonal_validation.round(4).to_string(index=False))
print("\nSaved:", t7_seasonal_path)
print("Final holdout accessed: False")

T+7 SEASONAL VALIDATION
      season  observations  actual_mean  candidate_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct
     Monsoon      922320.0       7.4632          8.9581         7.1642         11.8032           8.0860           15.3398              11.3994               23.0551
Post-monsoon      461160.0       3.4938          3.7627         3.7728          8.6088           4.6528           11.6688              18.9146               26.2234
 Pre-monsoon      695520.0       2.3632          2.1558         2.5925          7.2101           3.0216            9.6906              14.2010               25.5973
      Winter      682920.0       1.0597          0.6941         1.2766          5.0297           1.7763            6.8830              28.1296               26.9255

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_devel

In [131]:
# WI-05 — Cell 110: T+7 Rainfall Intensity Evaluation

intensity_conditions_t7 = [
    y_true_t7 < 1,
    (y_true_t7 >= 1) & (y_true_t7 < 5),
    (y_true_t7 >= 5) & (y_true_t7 < 20),
    (y_true_t7 >= 20) & (y_true_t7 < 50),
    y_true_t7 >= 50
]

intensity_labels_t7 = [
    "Below 1 mm",
    "1-5 mm",
    "5-20 mm",
    "20-50 mm",
    "50+ mm"
]

intensity_rows_t7 = []

for label, mask in zip(
    intensity_labels_t7,
    intensity_conditions_t7
):
    actual = y_true_t7[mask]
    candidate = y_pred_t7[mask]
    persistence = y_persistence_t7[mask]

    candidate_mae = mean_absolute_error(actual, candidate)
    persistence_mae = mean_absolute_error(actual, persistence)

    intensity_rows_t7.append({
        "intensity_band": label,
        "observations": len(actual),
        "observation_pct": len(actual) / len(y_true_t7) * 100,
        "candidate_mae": candidate_mae,
        "persistence_mae": persistence_mae,
        "mae_improvement_pct": (
            1 - candidate_mae / persistence_mae
        ) * 100
    })

t7_intensity_comparison = pd.DataFrame(intensity_rows_t7)

print("T+7 RAINFALL INTENSITY COMPARISON")
print(t7_intensity_comparison.round(4).to_string(index=False))
print("\nFinal holdout accessed: False")

T+7 RAINFALL INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         1.6751           1.6816               0.3904
        1-5 mm        575375          20.8324         5.0569           4.7761              -5.8794
       5-20 mm        458822          16.6124         4.9067           8.5434              42.5678
      20-50 mm        105268           3.8114        18.8910          22.3009              15.2902
        50+ mm         18977           0.6871        62.5509          63.5597               1.5871

Final holdout accessed: False


In [132]:
# WI-05 — Cell 111: T+7 Heavy-Rainfall Diagnostics

heavy_mask_t7 = y_true_t7 >= 50

heavy_actual_t7 = y_true_t7[heavy_mask_t7]
heavy_candidate_t7 = y_pred_t7[heavy_mask_t7]
heavy_persistence_t7 = y_persistence_t7[heavy_mask_t7]

heavy_results_t7 = {
    "observations": int(heavy_mask_t7.sum()),
    "actual_mean": float(heavy_actual_t7.mean()),
    "candidate_mean": float(heavy_candidate_t7.mean()),
    "persistence_mean": float(heavy_persistence_t7.mean()),
    "candidate_bias": float(np.mean(heavy_candidate_t7 - heavy_actual_t7)),
    "persistence_bias": float(np.mean(heavy_persistence_t7 - heavy_actual_t7)),
    "candidate_mae": float(mean_absolute_error(heavy_actual_t7, heavy_candidate_t7)),
    "persistence_mae": float(mean_absolute_error(heavy_actual_t7, heavy_persistence_t7)),
    "candidate_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t7, heavy_candidate_t7))),
    "persistence_rmse": float(np.sqrt(mean_squared_error(heavy_actual_t7, heavy_persistence_t7))),
    "actual_p95": float(np.percentile(heavy_actual_t7, 95)),
    "candidate_p95": float(np.percentile(heavy_candidate_t7, 95)),
    "persistence_p95": float(np.percentile(heavy_persistence_t7, 95)),
    "actual_max": float(heavy_actual_t7.max()),
    "candidate_max": float(heavy_candidate_t7.max()),
    "persistence_max": float(heavy_persistence_t7.max())
}

print("T+7 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)")
for key, value in heavy_results_t7.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\nFinal holdout accessed: False")

T+7 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)
observations: 18977
actual_mean: 77.4719
candidate_mean: 15.2407
persistence_mean: 16.0553
candidate_bias: -62.2312
persistence_bias: -61.4166
candidate_mae: 62.5509
persistence_mae: 63.5597
candidate_rmse: 73.3555
persistence_rmse: 75.1474
actual_p95: 142.4980
candidate_p95: 40.7244
persistence_p95: 58.3920
actual_max: 666.3700
candidate_max: 134.7834
persistence_max: 464.9300

Final holdout accessed: False


In [133]:
# WI-05 — Cell 112: Save T+7 Consolidated Findings

t7_intensity_path = (
    REPORT_DIR / "WI_05_t7_rainfall_intensity_comparison.csv"
)

t7_intensity_comparison.to_csv(t7_intensity_path, index=False)

t7_findings = {
    "work_item": "WI-05",
    "target": "PRECTOTCORR",
    "horizon": "T+7",
    "model": "Two-stage HistGradientBoosting",
    "status": "provisional_candidate",
    "validation_observations": int(len(y_true_t7)),
    "validation_mae_mm": float(t7_mae),
    "validation_rmse_mm": float(t7_rmse),
    "baseline_comparison": t7_baseline_comparison.to_dict(orient="records"),
    "seasonal_comparison": t7_seasonal_validation.to_dict(orient="records"),
    "intensity_comparison": t7_intensity_comparison.to_dict(orient="records"),
    "heavy_rainfall_diagnostics": heavy_results_t7,
    "wet_probability_threshold": 0.40,
    "persistence_feature": "PRECTOTCORR_current",
    "final_holdout_accessed": False,
    "final_model_selected": False
}

t7_findings_path = (
    REPORT_DIR / "WI_05_t7_consolidated_findings.json"
)

with open(t7_findings_path, "w", encoding="utf-8") as f:
    json.dump(
        t7_findings,
        f,
        indent=4,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

print("T+7 FINDINGS SAVED")
print("Intensity CSV:", t7_intensity_path)
print("Consolidated JSON:", t7_findings_path)
print("Validation observations:", len(y_true_t7))
print("Candidate MAE:", round(t7_mae, 4))
print("Candidate RMSE:", round(t7_rmse, 4))
print("Status:", t7_findings["status"])
print("Final model selected:", t7_findings["final_model_selected"])
print("Final holdout accessed:", t7_findings["final_holdout_accessed"])

T+7 FINDINGS SAVED
Intensity CSV: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t7_rainfall_intensity_comparison.csv
Consolidated JSON: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t7_consolidated_findings.json
Validation observations: 2761920
Candidate MAE: 3.9909
Candidate RMSE: 8.8456
Status: provisional_candidate
Final model selected: False
Final holdout accessed: False


In [135]:
# WI-05 — Cell 114: Inspect T+1 to T+3 Validation Arrays

for name in [
    "y_pred_t1", "y_pred_t2", "y_pred_t3",
    "y_probability_t1", "y_probability_t2", "y_probability_t3",
    "actual_parts_t1", "actual_parts_t2", "actual_parts_t3",
    "t1_validation_df", "t2_validation_df", "t3_validation_df"
]:
    if name in globals():
        value = globals()[name]
        print(name, "|", type(value).__name__, "|", len(value))
    else:
        print(name, "| NOT AVAILABLE")

print("\nCorrected persistence arrays:")
for h in [1, 2, 3]:
    name = f"y_persistence_corrected_t{h}"
    if name in globals():
        print(name, "|", len(globals()[name]))

y_pred_t1 | NOT AVAILABLE
y_pred_t2 | ndarray | 2761920
y_pred_t3 | ndarray | 2761920
y_probability_t1 | NOT AVAILABLE
y_probability_t2 | ndarray | 2761920
y_probability_t3 | ndarray | 2761920
actual_parts_t1 | NOT AVAILABLE
actual_parts_t2 | list | 252
actual_parts_t3 | list | 252
t1_validation_df | NOT AVAILABLE
t2_validation_df | NOT AVAILABLE
t3_validation_df | NOT AVAILABLE

Corrected persistence arrays:
y_persistence_corrected_t1 | 2761920
y_persistence_corrected_t2 | 2761920
y_persistence_corrected_t3 | 2761920


In [136]:
# WI-05 — Cell 115: Verify T+2 and T+3 Corrected Persistence

for h in [2, 3]:
    actual = np.concatenate(globals()[f"actual_parts_t{h}"])

    candidate = globals()[f"y_pred_t{h}"]
    persistence = globals()[f"y_persistence_corrected_t{h}"]

    if not (len(actual) == len(candidate) == len(persistence)):
        raise ValueError(f"T+{h}: array alignment mismatch")

    candidate_mae = mean_absolute_error(actual, candidate)
    candidate_rmse = np.sqrt(mean_squared_error(actual, candidate))

    persistence_mae = mean_absolute_error(actual, persistence)
    persistence_rmse = np.sqrt(mean_squared_error(actual, persistence))

    print(f"\nT+{h} CORRECTED BASELINE")
    print("Observations:", len(actual))
    print("Candidate MAE:", round(candidate_mae, 4))
    print("Candidate RMSE:", round(candidate_rmse, 4))
    print("Corrected persistence MAE:", round(persistence_mae, 4))
    print("Corrected persistence RMSE:", round(persistence_rmse, 4))
    print(
        "MAE improvement (%):",
        round((1 - candidate_mae / persistence_mae) * 100, 4)
    )
    print(
        "RMSE improvement (%):",
        round((1 - candidate_rmse / persistence_rmse) * 100, 4)
    )

print("\nT+1 candidate predictions still need recovery.")
print("Final holdout accessed: False")


T+2 CORRECTED BASELINE
Observations: 2761920
Candidate MAE: 3.418
Candidate RMSE: 8.3131
Corrected persistence MAE: 4.0844
Corrected persistence RMSE: 10.843
MAE improvement (%): 16.3157
RMSE improvement (%): 23.3318

T+3 CORRECTED BASELINE
Observations: 2761920
Candidate MAE: 3.579
Candidate RMSE: 8.552
Corrected persistence MAE: 4.3918
Corrected persistence RMSE: 11.3489
MAE improvement (%): 18.5071
RMSE improvement (%): 24.6449

T+1 candidate predictions still need recovery.
Final holdout accessed: False


In [137]:
# WI-05 — Cell 116: Recover T+1 Validation Predictions

classifier_path_t1 = (
    REPORT_DIR / "candidate_models"
    / "WI_05_hgb_wet_classifier_t1_candidate.joblib"
)

regressor_path_t1 = (
    REPORT_DIR / "candidate_models"
    / "WI_05_hgb_wet_amount_regressor_t1_candidate.joblib"
)

classifier_t1 = joblib.load(classifier_path_t1)
regressor_t1 = joblib.load(regressor_path_t1)

actual_parts_t1 = []
probability_parts_t1 = []
conditional_amount_parts_t1 = []
persistence_parts_t1 = []

target_col_t1 = "PRECTOTCORR_target_t_plus_1"

for batch_index in range(252):
    batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
    batch_df = pd.read_parquet(batch_path)

    batch_df["date"] = pd.to_datetime(batch_df["date"])
    batch_df["target_date"] = (
        batch_df["date"] + pd.to_timedelta(1, unit="D")
    )

    batch_df = batch_df[
        batch_df["target_date"].dt.year.between(2020, 2022)
    ]

    required_cols = (
        model_features_t2
        + [target_col_t1, "PRECTOTCORR_current"]
    )

    batch_df = batch_df.dropna(subset=required_cols)

    if len(batch_df) == 0:
        del batch_df
        continue

    X_batch = batch_df[model_features_t2]

    probabilities = classifier_t1.predict_proba(X_batch)[:, 1]
    conditional_amounts = regressor_t1.predict(X_batch)

    actual_parts_t1.append(
        batch_df[target_col_t1].to_numpy()
    )
    probability_parts_t1.append(probabilities)
    conditional_amount_parts_t1.append(conditional_amounts)
    persistence_parts_t1.append(
        batch_df["PRECTOTCORR_current"].to_numpy()
    )

    del batch_df, X_batch

y_true_t1 = np.concatenate(actual_parts_t1)
y_probability_t1 = np.concatenate(probability_parts_t1)
y_conditional_t1 = np.concatenate(conditional_amount_parts_t1)

y_persistence_corrected_t1 = np.concatenate(
    persistence_parts_t1
)

y_pred_t1 = np.where(
    y_probability_t1 >= 0.40,
    y_conditional_t1,
    0.0
)

print("T+1 PREDICTIONS RECOVERED")
print("Validation observations:", len(y_true_t1))
print("Candidate MAE:", round(mean_absolute_error(y_true_t1, y_pred_t1), 4))
print(
    "Candidate RMSE:",
    round(np.sqrt(mean_squared_error(y_true_t1, y_pred_t1)), 4)
)
print("Corrected persistence observations:", len(y_persistence_corrected_t1))

if not (
    len(y_true_t1)
    == len(y_pred_t1)
    == len(y_persistence_corrected_t1)
    == 2761920
):
    raise ValueError("T+1 validation alignment check failed.")

print("Alignment: PASS")
print("Final holdout accessed: False")

T+1 PREDICTIONS RECOVERED
Validation observations: 2761920
Candidate MAE: 3.1016
Candidate RMSE: 7.6624
Corrected persistence observations: 2761920
Alignment: PASS
Final holdout accessed: False


In [138]:
# WI-05 — Cell 117: Corrected T+1 to T+3 Baseline Comparisons

baseline_values = {
    1: {"seasonal_mae": 4.8002, "seasonal_rmse": 12.1503},
    2: {"seasonal_mae": 4.7911, "seasonal_rmse": 12.1318},
    3: {"seasonal_mae": 4.8055, "seasonal_rmse": 12.1839}
}

corrected_baseline_tables = {}

for h in [1, 2, 3]:
    actual = globals()[f"y_true_t{h}"]
    candidate = globals()[f"y_pred_t{h}"]
    persistence = globals()[f"y_persistence_corrected_t{h}"]

    candidate_mae = mean_absolute_error(actual, candidate)
    candidate_rmse = np.sqrt(mean_squared_error(actual, candidate))

    persistence_mae = mean_absolute_error(actual, persistence)
    persistence_rmse = np.sqrt(mean_squared_error(actual, persistence))

    seasonal_mae = baseline_values[h]["seasonal_mae"]
    seasonal_rmse = baseline_values[h]["seasonal_rmse"]

    rows = [
        {
            "model": "Two-stage HGB candidate",
            "mae": candidate_mae,
            "rmse": candidate_rmse
        },
        {
            "model": "Corrected persistence",
            "mae": persistence_mae,
            "rmse": persistence_rmse
        },
        {
            "model": "Seasonal naive",
            "mae": seasonal_mae,
            "rmse": seasonal_rmse
        },
        {
            "model": "Grid-cell calendar-day climatology",
            "mae": 3.7649,
            "rmse": 9.0620
        }
    ]

    comparison = pd.DataFrame(rows)

    comparison["mae_rank"] = (
        comparison["mae"].rank(method="min").astype(int)
    )
    comparison["rmse_rank"] = (
        comparison["rmse"].rank(method="min").astype(int)
    )

    corrected_baseline_tables[h] = comparison

    output_path = (
        REPORT_DIR / f"WI_05_t{h}_full_baseline_comparison_corrected.csv"
    )

    comparison.to_csv(output_path, index=False)

    print(f"\nT+{h} CORRECTED BASELINE COMPARISON")
    print(comparison.round(4).to_string(index=False))
    print("Saved:", output_path)

print("\nAll comparisons use PRECTOTCORR_current for persistence.")
print("Final holdout accessed: False")


T+1 CORRECTED BASELINE COMPARISON
                             model    mae    rmse  mae_rank  rmse_rank
           Two-stage HGB candidate 3.1016  7.6624         1          1
             Corrected persistence 3.2361  9.2107         2          3
                    Seasonal naive 4.8002 12.1503         4          4
Grid-cell calendar-day climatology 3.7649  9.0620         3          2
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_full_baseline_comparison_corrected.csv

T+2 CORRECTED BASELINE COMPARISON
                             model    mae    rmse  mae_rank  rmse_rank
           Two-stage HGB candidate 3.4180  8.3131         1          1
             Corrected persistence 4.0844 10.8430         3          3
                    Seasonal naive 4.7911 12.1318         4          4
Grid-cell calendar-day climatology 3.7649  9.0620         2          2
Saved: C:\Users

In [139]:
# WI-05 — Cell 118: Corrected T+1 to T+3 Seasonal Comparisons

seasonal_corrected_tables = {}

for h in [1, 2, 3]:
    date_parts = []

    for batch_index in range(252):
        batch_path = FEATURES_DIR / f"batch_{batch_index:04d}.parquet"
        batch_df = pd.read_parquet(batch_path)

        batch_df["date"] = pd.to_datetime(batch_df["date"])
        target_date = batch_df["date"] + pd.to_timedelta(h, unit="D")

        validation_mask = target_date.dt.year.between(2020, 2022)

        required_cols = (
            model_features_t2
            + [target_cols_corrected[h], "PRECTOTCORR_current"]
        )

        validation_df = batch_df.loc[validation_mask].dropna(
            subset=required_cols
        )

        date_parts.append(
            target_date.loc[validation_df.index].to_numpy()
        )

        del batch_df, validation_df

    target_dates = pd.to_datetime(
        np.concatenate(date_parts)
    )

    actual = globals()[f"y_true_t{h}"]
    candidate = globals()[f"y_pred_t{h}"]
    persistence = globals()[f"y_persistence_corrected_t{h}"]

    if not (
        len(target_dates) == len(actual)
        == len(candidate) == len(persistence)
    ):
        raise ValueError(f"T+{h}: seasonal array alignment mismatch")

    months = target_dates.month

    seasons = np.select(
        [
            np.isin(months, [12, 1, 2]),
            np.isin(months, [3, 4, 5]),
            np.isin(months, [6, 7, 8, 9]),
            np.isin(months, [10, 11])
        ],
        [
            "Winter",
            "Pre-monsoon",
            "Monsoon",
            "Post-monsoon"
        ],
        default="Unknown"
    )

    seasonal_df = pd.DataFrame({
        "season": seasons,
        "actual": actual,
        "candidate": candidate,
        "persistence": persistence
    })

    comparison = seasonal_df.groupby("season").apply(
        lambda group: pd.Series({
            "observations": len(group),
            "actual_mean": group["actual"].mean(),
            "candidate_mean": group["candidate"].mean(),
            "candidate_mae": mean_absolute_error(
                group["actual"], group["candidate"]
            ),
            "candidate_rmse": np.sqrt(mean_squared_error(
                group["actual"], group["candidate"]
            )),
            "persistence_mae": mean_absolute_error(
                group["actual"], group["persistence"]
            ),
            "persistence_rmse": np.sqrt(mean_squared_error(
                group["actual"], group["persistence"]
            ))
        }),
        include_groups=False
    ).reset_index()

    comparison["mae_improvement_pct"] = (
        1 - comparison["candidate_mae"]
        / comparison["persistence_mae"]
    ) * 100

    comparison["rmse_improvement_pct"] = (
        1 - comparison["candidate_rmse"]
        / comparison["persistence_rmse"]
    ) * 100

    seasonal_corrected_tables[h] = comparison

    output_path = (
        REPORT_DIR / f"WI_05_t{h}_seasonal_baseline_comparison_corrected.csv"
    )

    comparison.to_csv(output_path, index=False)

    print(f"\nT+{h} CORRECTED SEASONAL COMPARISON")
    print(comparison.round(4).to_string(index=False))
    print("Saved:", output_path)

print("\nFinal holdout accessed: False")


T+1 CORRECTED SEASONAL COMPARISON
      season  observations  actual_mean  candidate_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct
     Monsoon      922320.0       7.4632          8.0725         5.5297         10.1052           5.7230           11.9479               3.3764               15.4226
Post-monsoon      461160.0       3.4938          3.5830         2.8434          7.5010           3.0325            9.1898               6.2346               18.3769
 Pre-monsoon      695520.0       2.3632          2.2637         2.0770          6.4230           2.1497            7.8577               3.3840               18.2582
      Winter      682920.0       1.0597          0.9158         1.0403          4.4186           1.1215            5.5139               7.2392               19.8644
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_m

In [140]:
# WI-05 — Cell 119: Corrected T+1 to T+3 Intensity Comparisons

intensity_corrected_tables = {}
heavy_corrected_results = {}

intensity_labels = [
    "Below 1 mm",
    "1-5 mm",
    "5-20 mm",
    "20-50 mm",
    "50+ mm"
]

for h in [1, 2, 3]:
    actual = globals()[f"y_true_t{h}"]
    candidate = globals()[f"y_pred_t{h}"]
    persistence = globals()[f"y_persistence_corrected_t{h}"]

    conditions = [
        actual < 1,
        (actual >= 1) & (actual < 5),
        (actual >= 5) & (actual < 20),
        (actual >= 20) & (actual < 50),
        actual >= 50
    ]

    rows = []

    for label, mask in zip(intensity_labels, conditions):
        actual_band = actual[mask]
        candidate_band = candidate[mask]
        persistence_band = persistence[mask]

        candidate_mae = mean_absolute_error(
            actual_band, candidate_band
        )
        persistence_mae = mean_absolute_error(
            actual_band, persistence_band
        )

        rows.append({
            "intensity_band": label,
            "observations": len(actual_band),
            "observation_pct": len(actual_band) / len(actual) * 100,
            "candidate_mae": candidate_mae,
            "persistence_mae": persistence_mae,
            "mae_improvement_pct": (
                1 - candidate_mae / persistence_mae
            ) * 100
        })

    comparison = pd.DataFrame(rows)
    intensity_corrected_tables[h] = comparison

    intensity_path = (
        REPORT_DIR / f"WI_05_t{h}_rainfall_intensity_comparison_corrected.csv"
    )
    comparison.to_csv(intensity_path, index=False)

    heavy_mask = actual >= 50
    heavy_actual = actual[heavy_mask]
    heavy_candidate = candidate[heavy_mask]
    heavy_persistence = persistence[heavy_mask]

    heavy_results = {
        "observations": int(heavy_mask.sum()),
        "actual_mean": float(heavy_actual.mean()),
        "candidate_mean": float(heavy_candidate.mean()),
        "persistence_mean": float(heavy_persistence.mean()),
        "candidate_bias": float(np.mean(heavy_candidate - heavy_actual)),
        "persistence_bias": float(np.mean(heavy_persistence - heavy_actual)),
        "candidate_mae": float(mean_absolute_error(
            heavy_actual, heavy_candidate
        )),
        "persistence_mae": float(mean_absolute_error(
            heavy_actual, heavy_persistence
        )),
        "candidate_rmse": float(np.sqrt(mean_squared_error(
            heavy_actual, heavy_candidate
        ))),
        "persistence_rmse": float(np.sqrt(mean_squared_error(
            heavy_actual, heavy_persistence
        ))),
        "actual_p95": float(np.percentile(heavy_actual, 95)),
        "candidate_p95": float(np.percentile(heavy_candidate, 95)),
        "persistence_p95": float(np.percentile(heavy_persistence, 95)),
        "actual_max": float(heavy_actual.max()),
        "candidate_max": float(heavy_candidate.max()),
        "persistence_max": float(heavy_persistence.max())
    }

    heavy_corrected_results[h] = heavy_results

    print(f"\nT+{h} CORRECTED INTENSITY COMPARISON")
    print(comparison.round(4).to_string(index=False))
    print("Saved:", intensity_path)

    print(f"\nT+{h} HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)")
    for key, value in heavy_results.items():
        print(
            f"{key}: {value:.4f}"
            if isinstance(value, float)
            else f"{key}: {value}"
        )

print("\nCorrected persistence feature: PRECTOTCORR_current")
print("Final holdout accessed: False")


T+1 CORRECTED INTENSITY COMPARISON
intensity_band  observations  observation_pct  candidate_mae  persistence_mae  mae_improvement_pct
    Below 1 mm       1603478          58.0566         0.8887           0.5803             -53.1426
        1-5 mm        575375          20.8324         3.9714           3.3185             -19.6745
       5-20 mm        458822          16.6124         4.7967           7.2234              33.5953
      20-50 mm        105268           3.8114        15.5755          17.5913              11.4588
        50+ mm         18977           0.6871        53.5433          49.1145              -9.0172
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_rainfall_intensity_comparison_corrected.csv

T+1 HEAVY-RAINFALL DIAGNOSTICS (>=50 mm)
observations: 18977
actual_mean: 77.4719
candidate_mean: 25.4038
persistence_mean: 36.1843
candidate_bias: -52.0681
pe

In [141]:
# WI-05 — Cell 120: Save Corrected T+1 to T+3 Findings

corrected_findings = {}

for h in [1, 2, 3]:
    actual = globals()[f"y_true_t{h}"]
    candidate = globals()[f"y_pred_t{h}"]

    baseline_table = corrected_baseline_tables[h]
    seasonal_table = seasonal_corrected_tables[h]
    intensity_table = intensity_corrected_tables[h]
    heavy_results = heavy_corrected_results[h]

    candidate_row = baseline_table[
        baseline_table["model"] == "Two-stage HGB candidate"
    ].iloc[0]

    findings = {
        "work_item": "WI-05",
        "target": "PRECTOTCORR",
        "horizon": f"T+{h}",
        "model": "Two-stage HistGradientBoosting",
        "status": "provisional_candidate",
        "validation_observations": int(len(actual)),
        "validation_mae_mm": float(candidate_row["mae"]),
        "validation_rmse_mm": float(candidate_row["rmse"]),
        "baseline_comparison": baseline_table.to_dict(orient="records"),
        "seasonal_comparison": seasonal_table.to_dict(orient="records"),
        "intensity_comparison": intensity_table.to_dict(orient="records"),
        "heavy_rainfall_diagnostics": heavy_results,
        "wet_probability_threshold": 0.40,
        "persistence_feature": "PRECTOTCORR_current",
        "corrected_persistence_evaluation": True,
        "final_holdout_accessed": False,
        "final_model_selected": False
    }

    output_path = (
        REPORT_DIR / f"WI_05_t{h}_consolidated_findings_corrected.json"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            findings,
            f,
            indent=4,
            default=lambda x: x.item() if hasattr(x, "item") else str(x)
        )

    corrected_findings[h] = findings

    print(f"T+{h} CORRECTED FINDINGS SAVED")
    print("Candidate MAE:", round(findings["validation_mae_mm"], 4))
    print("Candidate RMSE:", round(findings["validation_rmse_mm"], 4))
    print("Path:", output_path)

print("\nAll corrected findings saved.")
print("Final model selected: False")
print("Final holdout accessed: False")

T+1 CORRECTED FINDINGS SAVED
Candidate MAE: 3.1016
Candidate RMSE: 7.6624
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t1_consolidated_findings_corrected.json
T+2 CORRECTED FINDINGS SAVED
Candidate MAE: 3.418
Candidate RMSE: 8.3131
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t2_consolidated_findings_corrected.json
T+3 CORRECTED FINDINGS SAVED
Candidate MAE: 3.579
Candidate RMSE: 8.552
Path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_t3_consolidated_findings_corrected.json

All corrected findings saved.
Final model selected: False
Final holdout accessed: False


In [142]:
# WI-05 — Cell 121: Consolidated Seven-Horizon Comparison

horizon_metrics = [
    {
        "horizon": "T+1",
        "candidate_mae": 3.1016,
        "candidate_rmse": 7.6624,
        "persistence_mae": 3.2361,
        "persistence_rmse": 9.2107
    },
    {
        "horizon": "T+2",
        "candidate_mae": 3.4180,
        "candidate_rmse": 8.3131,
        "persistence_mae": 4.0844,
        "persistence_rmse": 10.8430
    },
    {
        "horizon": "T+3",
        "candidate_mae": 3.5790,
        "candidate_rmse": 8.5520,
        "persistence_mae": 4.3918,
        "persistence_rmse": 11.3489
    },
    {
        "horizon": "T+4",
        "candidate_mae": 3.6100,
        "candidate_rmse": 8.5761,
        "persistence_mae": 4.5194,
        "persistence_rmse": 11.5325
    },
    {
        "horizon": "T+5",
        "candidate_mae": 3.6327,
        "candidate_rmse": 8.5980,
        "persistence_mae": 4.6048,
        "persistence_rmse": 11.6422
    },
    {
        "horizon": "T+6",
        "candidate_mae": 3.6504,
        "candidate_rmse": 8.6190,
        "persistence_mae": 4.6495,
        "persistence_rmse": 11.6799
    },
    {
        "horizon": "T+7",
        "candidate_mae": 3.9909,
        "candidate_rmse": 8.8456,
        "persistence_mae": 4.6772,
        "persistence_rmse": 11.6909
    }
]

horizon_comparison = pd.DataFrame(horizon_metrics)

horizon_comparison["climatology_mae"] = 3.7649
horizon_comparison["climatology_rmse"] = 9.0620

horizon_comparison["seasonal_naive_mae"] = [
    4.8002, 4.7911, 4.8055, 4.8382, 4.8639, 4.8695, 4.8473
]

horizon_comparison["seasonal_naive_rmse"] = [
    12.1503, 12.1318, 12.1839, 12.2575, 12.3194, 12.3261, 12.2411
]

horizon_comparison["mae_improvement_vs_persistence_pct"] = (
    1 - horizon_comparison["candidate_mae"]
    / horizon_comparison["persistence_mae"]
) * 100

horizon_comparison["rmse_improvement_vs_persistence_pct"] = (
    1 - horizon_comparison["candidate_rmse"]
    / horizon_comparison["persistence_rmse"]
) * 100

horizon_comparison["mae_improvement_vs_climatology_pct"] = (
    1 - horizon_comparison["candidate_mae"]
    / horizon_comparison["climatology_mae"]
) * 100

horizon_comparison["rmse_improvement_vs_climatology_pct"] = (
    1 - horizon_comparison["candidate_rmse"]
    / horizon_comparison["climatology_rmse"]
) * 100

horizon_comparison["strongest_baseline_mae"] = horizon_comparison[
    ["persistence_mae", "seasonal_naive_mae", "climatology_mae"]
].min(axis=1)

horizon_comparison["strongest_baseline_rmse"] = horizon_comparison[
    ["persistence_rmse", "seasonal_naive_rmse", "climatology_rmse"]
].min(axis=1)

horizon_comparison["mae_rank"] = (
    horizon_comparison[
        ["candidate_mae", "persistence_mae",
         "seasonal_naive_mae", "climatology_mae"]
    ].rank(axis=1, method="min").iloc[:, 0].astype(int)
)

horizon_comparison["rmse_rank"] = (
    horizon_comparison[
        ["candidate_rmse", "persistence_rmse",
         "seasonal_naive_rmse", "climatology_rmse"]
    ].rank(axis=1, method="min").iloc[:, 0].astype(int)
)

comparison_path = (
    REPORT_DIR / "WI_05_seven_horizon_validation_comparison.csv"
)

horizon_comparison.to_csv(comparison_path, index=False)

print("WI-05 SEVEN-HORIZON VALIDATION COMPARISON")
print(
    horizon_comparison[
        [
            "horizon",
            "candidate_mae",
            "candidate_rmse",
            "persistence_mae",
            "climatology_mae",
            "strongest_baseline_mae",
            "mae_improvement_vs_persistence_pct",
            "mae_rank",
            "rmse_rank"
        ]
    ].round(4).to_string(index=False)
)

print("\nSaved:", comparison_path)
print("Final model selected: False")
print("Final holdout accessed: False")

WI-05 SEVEN-HORIZON VALIDATION COMPARISON
horizon  candidate_mae  candidate_rmse  persistence_mae  climatology_mae  strongest_baseline_mae  mae_improvement_vs_persistence_pct  mae_rank  rmse_rank
    T+1         3.1016          7.6624           3.2361           3.7649                  3.2361                              4.1562         1          1
    T+2         3.4180          8.3131           4.0844           3.7649                  3.7649                             16.3157         1          1
    T+3         3.5790          8.5520           4.3918           3.7649                  3.7649                             18.5072         1          1
    T+4         3.6100          8.5761           4.5194           3.7649                  3.7649                             20.1221         1          1
    T+5         3.6327          8.5980           4.6048           3.7649                  3.7649                             21.1106         1          1
    T+6         3.6504          8.

In [143]:
# WI-05 — Cell 122: Strongest Baseline Performance

horizon_comparison["mae_improvement_vs_strongest_baseline_pct"] = (
    1 - horizon_comparison["candidate_mae"]
    / horizon_comparison["strongest_baseline_mae"]
) * 100

horizon_comparison["rmse_improvement_vs_strongest_baseline_pct"] = (
    1 - horizon_comparison["candidate_rmse"]
    / horizon_comparison["strongest_baseline_rmse"]
) * 100

horizon_comparison["strongest_mae_baseline"] = horizon_comparison[
    ["persistence_mae", "seasonal_naive_mae", "climatology_mae"]
].idxmin(axis=1)

horizon_comparison["strongest_rmse_baseline"] = horizon_comparison[
    ["persistence_rmse", "seasonal_naive_rmse", "climatology_rmse"]
].idxmin(axis=1)

strongest_baseline_path = (
    REPORT_DIR / "WI_05_seven_horizon_strongest_baseline_comparison.csv"
)

horizon_comparison.to_csv(strongest_baseline_path, index=False)

print("SEVEN-HORIZON STRONGEST BASELINE COMPARISON")

print(
    horizon_comparison[
        [
            "horizon",
            "strongest_mae_baseline",
            "mae_improvement_vs_strongest_baseline_pct",
            "strongest_rmse_baseline",
            "rmse_improvement_vs_strongest_baseline_pct"
        ]
    ].round(4).to_string(index=False)
)

print("\nSaved:", strongest_baseline_path)
print("Final model selected: False")
print("Final holdout accessed: False")

SEVEN-HORIZON STRONGEST BASELINE COMPARISON
horizon strongest_mae_baseline  mae_improvement_vs_strongest_baseline_pct strongest_rmse_baseline  rmse_improvement_vs_strongest_baseline_pct
    T+1        persistence_mae                                     4.1562        climatology_rmse                                     15.4447
    T+2        climatology_mae                                     9.2141        climatology_rmse                                      8.2642
    T+3        climatology_mae                                     4.9377        climatology_rmse                                      5.6279
    T+4        climatology_mae                                     4.1143        climatology_rmse                                      5.3620
    T+5        climatology_mae                                     3.5114        climatology_rmse                                      5.1203
    T+6        climatology_mae                                     3.0412        climatology_rmse       

In [144]:
# WI-05 — Cell 123: Consolidate Seven-Horizon Seasonal and Intensity Results

seasonal_frames = []
intensity_frames = []

for h in range(1, 8):

    # Use corrected persistence reports for T+1 to T+3
    if h <= 3:
        seasonal_file = (
            REPORT_DIR /
            f"WI_05_t{h}_seasonal_baseline_comparison_corrected.csv"
        )
        intensity_file = (
            REPORT_DIR /
            f"WI_05_t{h}_rainfall_intensity_comparison_corrected.csv"
        )
    else:
        seasonal_file = (
            REPORT_DIR /
            f"WI_05_t{h}_seasonal_baseline_comparison.csv"
        )
        intensity_file = (
            REPORT_DIR /
            f"WI_05_t{h}_rainfall_intensity_comparison.csv"
        )

    if not seasonal_file.exists():
        raise FileNotFoundError(seasonal_file)

    if not intensity_file.exists():
        raise FileNotFoundError(intensity_file)

    seasonal_df = pd.read_csv(seasonal_file)
    intensity_df = pd.read_csv(intensity_file)

    seasonal_df.insert(0, "horizon", f"T+{h}")
    intensity_df.insert(0, "horizon", f"T+{h}")

    seasonal_frames.append(seasonal_df)
    intensity_frames.append(intensity_df)

seasonal_summary = pd.concat(
    seasonal_frames,
    ignore_index=True
)

intensity_summary = pd.concat(
    intensity_frames,
    ignore_index=True
)

seasonal_output = (
    REPORT_DIR / "WI_05_seven_horizon_seasonal_summary.csv"
)

intensity_output = (
    REPORT_DIR / "WI_05_seven_horizon_intensity_summary.csv"
)

seasonal_summary.to_csv(seasonal_output, index=False)
intensity_summary.to_csv(intensity_output, index=False)

print("SEVEN-HORIZON SEASONAL SUMMARY")
print("Rows:", len(seasonal_summary))
print("Horizons:", seasonal_summary["horizon"].nunique())
print("Seasons:", sorted(seasonal_summary["season"].unique()))
print("\n", seasonal_summary.round(4).to_string(index=False))

print("\nSEVEN-HORIZON INTENSITY SUMMARY")
print("Rows:", len(intensity_summary))
print("Horizons:", intensity_summary["horizon"].nunique())
print("Intensity bands:", intensity_summary["intensity_band"].nunique())
print("\n", intensity_summary.round(4).to_string(index=False))

print("\nSaved seasonal summary:", seasonal_output)
print("Saved intensity summary:", intensity_output)
print("Final holdout accessed: False")

SEVEN-HORIZON SEASONAL SUMMARY
Rows: 28
Horizons: 7
Seasons: ['Monsoon', 'Post-monsoon', 'Pre-monsoon', 'Winter']

 horizon       season  observations  actual_mean  candidate_mean  candidate_mae  candidate_rmse  persistence_mae  persistence_rmse  mae_improvement_pct  rmse_improvement_pct  prediction_mean
    T+1      Monsoon      922320.0       7.4632          8.0725         5.5297         10.1052           5.7230           11.9479               3.3764               15.4226              NaN
    T+1 Post-monsoon      461160.0       3.4938          3.5830         2.8434          7.5010           3.0325            9.1898               6.2346               18.3769              NaN
    T+1  Pre-monsoon      695520.0       2.3632          2.2637         2.0770          6.4230           2.1497            7.8577               3.3840               18.2582              NaN
    T+1       Winter      682920.0       1.0597          0.9158         1.0403          4.4186           1.1215            5

In [145]:
# WI-05 — Cell 124: Normalize Seven-Horizon Seasonal Summary

seasonal_summary_normalized = seasonal_summary.copy()

if "prediction_mean" in seasonal_summary_normalized.columns:
    seasonal_summary_normalized["candidate_mean"] = (
        seasonal_summary_normalized["candidate_mean"].fillna(
            seasonal_summary_normalized["prediction_mean"]
        )
    )

seasonal_summary_normalized = seasonal_summary_normalized.drop(
    columns=["prediction_mean"],
    errors="ignore"
)

if seasonal_summary_normalized["candidate_mean"].isna().any():
    raise ValueError("Candidate mean still contains missing values.")

seasonal_normalized_path = (
    REPORT_DIR / "WI_05_seven_horizon_seasonal_summary_normalized.csv"
)

seasonal_summary_normalized.to_csv(
    seasonal_normalized_path,
    index=False
)

print("NORMALIZED SEASONAL SUMMARY")
print("Rows:", len(seasonal_summary_normalized))
print("Missing candidate means:",
      seasonal_summary_normalized["candidate_mean"].isna().sum())
print("\nCandidate mean by horizon:")
print(
    seasonal_summary_normalized.groupby("horizon")["candidate_mean"]
    .mean()
    .round(4)
)

print("\nSaved:", seasonal_normalized_path)
print("Final holdout accessed: False")

NORMALIZED SEASONAL SUMMARY
Rows: 28
Missing candidate means: 0

Candidate mean by horizon:
horizon
T+1    3.7088
T+2    3.3950
T+3    3.4146
T+4    3.3838
T+5    3.3786
T+6    3.3861
T+7    3.8927
Name: candidate_mean, dtype: float64

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_seven_horizon_seasonal_summary_normalized.csv
Final holdout accessed: False


In [146]:
# WI-05 — Cell 125: Seven-Horizon Heavy-Rainfall Summary

heavy_rows = []

# Corrected diagnostics for T+1 to T+3
for h in [1, 2, 3]:
    result = heavy_corrected_results[h]

    heavy_rows.append({
        "horizon": f"T+{h}",
        **result
    })

# Previously calculated diagnostics for T+4 to T+7
heavy_results_later = {
    4: {
        "observations": 18977,
        "actual_mean": 77.4719,
        "candidate_mean": 14.1108,
        "persistence_mean": 16.9983,
        "candidate_bias": -63.3611,
        "persistence_bias": -60.4736,
        "candidate_mae": 63.5643,
        "persistence_mae": 63.0502,
        "candidate_rmse": 74.2442,
        "persistence_rmse": 75.0987,
        "actual_p95": 142.4980,
        "candidate_p95": 39.2729,
        "persistence_p95": 60.3760,
        "actual_max": 666.3700,
        "candidate_max": 116.2748,
        "persistence_max": 440.8500
    },
    5: {
        "observations": 18977,
        "actual_mean": 77.4719,
        "candidate_mean": 14.0348,
        "persistence_mean": 16.2209,
        "candidate_bias": -63.4371,
        "persistence_bias": -61.2510,
        "candidate_mae": 63.6622,
        "persistence_mae": 63.5390,
        "candidate_rmse": 74.2972,
        "persistence_rmse": 75.4105,
        "actual_p95": 142.4980,
        "candidate_p95": 40.0633,
        "persistence_p95": 59.5140,
        "actual_max": 666.3700,
        "candidate_max": 122.1049,
        "persistence_max": 410.0500
    },
    6: {
        "observations": 18977,
        "actual_mean": 77.4719,
        "candidate_mean": 13.8285,
        "persistence_mean": 16.1630,
        "candidate_bias": -63.6434,
        "persistence_bias": -61.3089,
        "candidate_mae": 63.8456,
        "persistence_mae": 63.7419,
        "candidate_rmse": 74.4223,
        "persistence_rmse": 75.4055,
        "actual_p95": 142.4980,
        "candidate_p95": 38.3718,
        "persistence_p95": 59.2440,
        "actual_max": 666.3700,
        "candidate_max": 118.1988,
        "persistence_max": 383.4000
    },
    7: {
        "observations": 18977,
        "actual_mean": 77.4719,
        "candidate_mean": 15.2407,
        "persistence_mean": 16.0553,
        "candidate_bias": -62.2312,
        "persistence_bias": -61.4166,
        "candidate_mae": 62.5509,
        "persistence_mae": 63.5597,
        "candidate_rmse": 73.3555,
        "persistence_rmse": 75.1474,
        "actual_p95": 142.4980,
        "candidate_p95": 40.7244,
        "persistence_p95": 58.3920,
        "actual_max": 666.3700,
        "candidate_max": 134.7834,
        "persistence_max": 464.9300
    }
}

for h, result in heavy_results_later.items():
    heavy_rows.append({
        "horizon": f"T+{h}",
        **result
    })

heavy_summary = pd.DataFrame(heavy_rows).sort_values("horizon")

heavy_summary["candidate_mae_improvement_pct"] = (
    1 - heavy_summary["candidate_mae"]
    / heavy_summary["persistence_mae"]
) * 100

heavy_summary["candidate_rmse_improvement_pct"] = (
    1 - heavy_summary["candidate_rmse"]
    / heavy_summary["persistence_rmse"]
) * 100

heavy_summary["candidate_mean_underestimation_mm"] = (
    heavy_summary["actual_mean"] - heavy_summary["candidate_mean"]
)

heavy_output = (
    REPORT_DIR / "WI_05_seven_horizon_heavy_rainfall_summary.csv"
)

heavy_summary.to_csv(heavy_output, index=False)

print("SEVEN-HORIZON HEAVY-RAINFALL SUMMARY (>=50 mm)")
print(
    heavy_summary[
        [
            "horizon",
            "observations",
            "candidate_mae",
            "persistence_mae",
            "candidate_rmse",
            "persistence_rmse",
            "candidate_mae_improvement_pct",
            "candidate_mean",
            "actual_mean",
            "candidate_mean_underestimation_mm"
        ]
    ].round(4).to_string(index=False)
)

print("\nSaved:", heavy_output)
print("Final holdout accessed: False")

SEVEN-HORIZON HEAVY-RAINFALL SUMMARY (>=50 mm)
horizon  observations  candidate_mae  persistence_mae  candidate_rmse  persistence_rmse  candidate_mae_improvement_pct  candidate_mean  actual_mean  candidate_mean_underestimation_mm
    T+1         18977        53.5433          49.1145         65.4931           63.0324                        -9.0172         25.4038      77.4719                            52.0681
    T+2         18977        61.3879          58.5887         72.4713           71.5838                        -4.7777         16.4462      77.4719                            61.0257
    T+3         18977        62.6588          61.5446         73.5146           73.5814                        -1.8105         15.2398      77.4719                            62.2321
    T+4         18977        63.5643          63.0502         74.2442           75.0987                        -0.8154         14.1108      77.4719                            63.3611
    T+5         18977        63.6622  

In [147]:
# WI-05 — Cell 126: Generate Completion Report

from pathlib import Path
import pandas as pd

# Load consolidated validation summaries
comparison = pd.read_csv(
    REPORT_DIR / "WI_05_seven_horizon_validation_comparison.csv"
)

seasonal = pd.read_csv(
    REPORT_DIR / "WI_05_seven_horizon_seasonal_summary_normalized.csv"
)

intensity = pd.read_csv(
    REPORT_DIR / "WI_05_seven_horizon_intensity_summary.csv"
)

heavy = pd.read_csv(
    REPORT_DIR / "WI_05_seven_horizon_heavy_rainfall_summary.csv"
)

# Basic integrity checks
assert len(comparison) == 7, "Expected seven horizon comparison rows"
assert len(seasonal) == 28, "Expected 28 seasonal summary rows"
assert len(intensity) == 35, "Expected 35 intensity summary rows"
assert len(heavy) == 7, "Expected seven heavy-rainfall summary rows"
assert seasonal["candidate_mean"].notna().all()

report = f"""
# WI-05 — Rainfall Model Development

**Project:** BharatCrop Intelligence  
**Work item:** WI-05  
**Status:** Model development and validation completed; final model selection pending WI-07  
**Target:** PRECTOTCORR (daily rainfall, mm/day)  
**Forecast horizons:** T+1 through T+7  
**Evaluation:** Validation period 2020–2022  
**Final holdout:** 2023–2024 — not accessed

---

## 1. Objective and scope

WI-05 develops and evaluates rainfall forecasting candidates for seven
forecast horizons, from one to seven days ahead.

The work is conducted at grid-cell level using the weather feature dataset.
It is independent of crop-yield modelling and APY data.

The purpose is to compare candidate rainfall models against established
forecasting baselines and understand their performance across seasons,
rainfall-intensity bands, and heavy-rainfall events.

## 2. Data and validation design

- Source dataset: weather-daily-v1.
- Feature dataset: weather-features-v1.
- Spatial resolution: grid-cell level.
- Training target dates: 2013–2019.
- Validation target dates: 2020–2022.
- Final holdout target dates: 2023–2024.
- Training sample used for candidate development: 352,800 rows per horizon.
- Predictor count: 53.
- Rainfall target: PRECTOTCORR, measured in mm/day.
- Wet-day threshold used for classification: 1.0 mm/day.

The split is based on target dates rather than random row assignment.
The final holdout was not accessed during this work.

## 3. Modelling approach

The principal candidate approach is a two-stage model:

1. A HistGradientBoostingClassifier estimates whether rainfall reaches
   the wet-day threshold.
2. A HistGradientBoostingRegressor estimates rainfall amount conditional
   on the wet-day stage.

The candidate models were developed independently for T+1 through T+7.

The wet-day threshold is an operational modelling threshold and should
not be interpreted as a universal meteorological definition.

Model selection remains provisional. Final model comparison and selection
belong to WI-07.

## 4. Seven-horizon validation comparison

{comparison.round(4).to_string(index=False)}

The candidate models outperform the persistence baseline on aggregate
MAE across all seven horizons. However, the strongest baseline differs
by horizon. Climatology is the strongest MAE baseline from T+2 through
T+7, while persistence is strongest at T+1.

At T+7, the candidate has worse aggregate MAE than climatology, despite
better RMSE. This trade-off must be considered during WI-07.

## 5. Seasonal performance

The seasonal evaluation contains 28 records: seven horizons across
four seasons.

{seasonal.round(4).to_string(index=False)}

Seasonal performance is not uniform. The results should be considered
alongside aggregate metrics rather than treated as evidence that the
candidate is equally reliable in every season.

## 6. Rainfall-intensity performance

The intensity evaluation contains 35 records across seven horizons
and five rainfall-intensity bands.

{intensity.round(4).to_string(index=False)}

The candidate's aggregate performance does not imply consistent
performance across rainfall intensities. In particular, performance
in the lower rainfall bands and the highest intensity band requires
separate consideration.

## 7. Heavy-rainfall diagnostics

Heavy rainfall is evaluated here using the 50+ mm/day band.

{heavy.round(4).to_string(index=False)}

The heavy-rainfall results reveal a major limitation:

- The actual mean rainfall in this band is approximately 77.47 mm/day.
- Candidate mean predictions are substantially lower across all horizons.
- Candidate heavy-band MAE is worse than persistence from T+1 through T+6.
- At T+7, candidate MAE improves on persistence by only approximately
  1.59%.
- Candidate predictions substantially underestimate heavy-rainfall
  intensity.

Therefore, these candidates should not be represented as validated
heavy-rainfall warning models.

## 8. Wet-day classification diagnostics

Detailed classifier threshold and calibration diagnostics were evaluated
for T+1.

At the 0.40 classification threshold:

- Precision: 0.8088
- Recall: 0.8654
- F1-score: 0.8361
- PR-AUC: 0.9147
- Brier score: 0.099175

The 0.40 threshold produced the best F1-score among the tested thresholds.
It remains a provisional operating threshold.

These classification results should not be generalized to T+2 through
T+7 without equivalent horizon-specific evaluation.

## 9. Key limitations

1. **Heavy-rainfall underestimation:** Candidate models substantially
   underestimate rainfall in the 50+ mm/day band.
2. **Uneven intensity performance:** Aggregate metrics conceal weaknesses
   in specific rainfall-intensity bands.
3. **T+7 aggregate trade-off:** T+7 candidate MAE is worse than climatology.
4. **Sampling:** Candidate development used a 352,800-row sample per
   horizon rather than the full feature-complete training dataset.
5. **Spatial scope:** Evaluation is grid-cell level; district-level
   aggregation and land masking are outside this work item's scope.
6. **Historical data limitation:** The source archive ends in 2024 and
   does not provide live weather forecasts.
7. **Agricultural interpretation:** Weather predictions alone do not
   establish crop damage, disease occurrence, or yield loss.
8. **Holdout:** The final 2023–2024 holdout remains untouched and requires
   a separately controlled evaluation.

## 10. Handoff to WI-07

WI-05 provides seven horizon-specific rainfall candidate models and
validation evidence for final comparison.

WI-07 must:

- Compare candidates consistently against WI-04 baselines.
- Review aggregate MAE and RMSE together.
- Consider seasonal and intensity-band performance.
- Explicitly assess heavy-rainfall underestimation.
- Decide whether one model family is suitable across all horizons or
  whether horizon-specific choices are justified.
- Preserve the final holdout for controlled evaluation after model
  selection decisions are complete.

No final production model is selected by WI-05.

## 11. Completion statement

WI-05 rainfall model development and validation diagnostics for T+1
through T+7 have been consolidated.

The work item is ready for handoff to WI-07 for model comparison and
final selection, subject to review of the documented limitations.

**Final holdout accessed: No.**
"""

report_path = REPORT_DIR / "WI_05_Rainfall_Model_Development_Completion_Report.md"
report_path.write_text(report.strip(), encoding="utf-8")

print("WI-05 COMPLETION REPORT GENERATED")
print("Report:", report_path)
print("Comparison rows:", len(comparison))
print("Seasonal rows:", len(seasonal))
print("Intensity rows:", len(intensity))
print("Heavy-rainfall rows:", len(heavy))
print("Final holdout accessed: False")

WI-05 COMPLETION REPORT GENERATED
Report: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_05_rainfall_model_development\WI_05_Rainfall_Model_Development_Completion_Report.md
Comparison rows: 7
Seasonal rows: 28
Intensity rows: 35
Heavy-rainfall rows: 7
Final holdout accessed: False


In [148]:
# WI-05 — Cell 127: Final Artifact and Model Audit

from pathlib import Path
import pandas as pd

project_root = REPORT_DIR.parents[3]

required_files = [
    "WI_05_Rainfall_Model_Development_Completion_Report.md",
    "WI_05_seven_horizon_validation_comparison.csv",
    "WI_05_seven_horizon_strongest_baseline_comparison.csv",
    "WI_05_seven_horizon_seasonal_summary.csv",
    "WI_05_seven_horizon_seasonal_summary_normalized.csv",
    "WI_05_seven_horizon_intensity_summary.csv",
    "WI_05_seven_horizon_heavy_rainfall_summary.csv",
]

print("REQUIRED REPORT AND SUMMARY FILES")
missing_files = []

for filename in required_files:
    path = REPORT_DIR / filename
    exists = path.exists()
    print(f"{'FOUND' if exists else 'MISSING'}: {filename}")

    if not exists:
        missing_files.append(filename)

# Find WI-05 candidate model artifacts across the project
model_files = [
    p for p in project_root.rglob("*.joblib")
    if "WI_05" in p.name.upper()
    and ("candidate" in p.name.lower())
]

print("\nCANDIDATE MODEL ARTIFACTS")
for path in sorted(model_files):
    print(path)

# Check that both classifier and regressor exist for each horizon
model_audit = []

for h in range(1, 8):
    horizon_token = f"t{h}"

    horizon_files = [
        p for p in model_files
        if horizon_token in p.name.lower()
    ]

    has_classifier = any(
        "classifier" in p.name.lower()
        for p in horizon_files
    )

    has_regressor = any(
        "regressor" in p.name.lower()
        for p in horizon_files
    )

    model_audit.append({
        "horizon": f"T+{h}",
        "classifier_found": has_classifier,
        "regressor_found": has_regressor,
        "model_files_found": len(horizon_files)
    })

model_audit_df = pd.DataFrame(model_audit)

print("\nSEVEN-HORIZON MODEL AUDIT")
print(model_audit_df.to_string(index=False))

print("\nFINAL AUDIT")
print("Missing required files:", len(missing_files))
print("Candidate model files found:", len(model_files))
print(
    "Horizons with both model stages:",
    int(
        (
            model_audit_df["classifier_found"]
            & model_audit_df["regressor_found"]
        ).sum()
    ),
    "/ 7"
)
print("Final holdout accessed: False")

REQUIRED REPORT AND SUMMARY FILES
FOUND: WI_05_Rainfall_Model_Development_Completion_Report.md
FOUND: WI_05_seven_horizon_validation_comparison.csv
FOUND: WI_05_seven_horizon_strongest_baseline_comparison.csv
FOUND: WI_05_seven_horizon_seasonal_summary.csv
FOUND: WI_05_seven_horizon_seasonal_summary_normalized.csv
FOUND: WI_05_seven_horizon_intensity_summary.csv
FOUND: WI_05_seven_horizon_heavy_rainfall_summary.csv

CANDIDATE MODEL ARTIFACTS
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t3_candidate.joblib
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t4_candidate.joblib
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\candidate_models\WI_05_hgb_wet_amount_regressor_t5_candidate.joblib

# WI-05 — Rainfall Model Development

**Project:** BharatCrop Intelligence  
**Work Item:** WI-05  
**Status:** Complete — Model development and validation completed  
**Target Variable:** `PRECTOTCORR` (daily rainfall, mm/day)  
**Forecast Horizons:** T+1 through T+7  
**Validation Period:** 2020–2022  
**Final Holdout:** 2023–2024 — Not accessed  

---

## 1. Objective and Scope

WI-05 develops and evaluates rainfall forecasting candidates for seven forecast horizons, from one to seven days ahead.

The work is conducted at grid-cell level using the weather feature dataset. It is independent of crop-yield modelling and APY data.

The objective is to compare candidate rainfall models against established forecasting baselines and evaluate their performance across seasons, rainfall-intensity bands, and heavy-rainfall events.

## 2. Data and Validation Design

- **Source dataset:** `weather-daily-v1`
- **Feature dataset:** `weather-features-v1`
- **Spatial resolution:** Grid-cell level
- **Training target dates:** 2013–2019
- **Validation target dates:** 2020–2022
- **Final holdout target dates:** 2023–2024
- **Training sample:** 352,800 rows per horizon
- **Predictors:** 53
- **Target:** `PRECTOTCORR`, measured in mm/day
- **Wet-day threshold:** 1.0 mm/day

The data split is based on target dates rather than random row assignment. The final holdout was not accessed during this work.

## 3. Modelling Approach

The principal candidate approach is a two-stage model:

1. A `HistGradientBoostingClassifier` estimates whether rainfall reaches the wet-day threshold.
2. A `HistGradientBoostingRegressor` estimates rainfall amount conditional on the wet-day stage.

Candidate models were developed independently for T+1 through T+7.

The wet-day threshold is an operational modelling threshold and should not be interpreted as a universal meteorological definition.

Model selection remains provisional. Final comparison and selection belong to WI-07.

## 4. Seven-Horizon Validation Comparison

| Horizon | Candidate MAE | Candidate RMSE | Persistence MAE | Climatology MAE |
|---|---:|---:|---:|---:|
| T+1 | 3.1016 | 7.6624 | 3.2361 | 3.7649 |
| T+2 | 3.4180 | 8.3131 | 4.0844 | 3.7649 |
| T+3 | 3.5790 | 8.5520 | 4.3918 | 3.7649 |
| T+4 | 3.6100 | 8.5761 | 4.5194 | 3.7649 |
| T+5 | 3.6327 | 8.5980 | 4.6048 | 3.7649 |
| T+6 | 3.6504 | 8.6190 | 4.6495 | 3.7649 |
| T+7 | 3.9909 | 8.8456 | 4.6772 | 3.7649 |

The candidate models outperform persistence on aggregate MAE across all seven horizons.

However, the strongest baseline differs by horizon:

- T+1: Persistence is the strongest MAE baseline.
- T+2 through T+7: Climatology is the strongest MAE baseline.

At T+7, the candidate has worse aggregate MAE than climatology, despite better RMSE. This trade-off must be considered during WI-07.

## 5. Seasonal Performance

Seasonal evaluation contains 28 records: seven horizons across four seasons.

The evaluation covers:

- Winter
- Pre-monsoon
- Monsoon
- Post-monsoon

The candidate generally improves on persistence across seasonal MAE and RMSE. However, performance is not uniform across all horizons and seasons.

The detailed results are stored in:

`WI_05_seven_horizon_seasonal_summary_normalized.csv`

## 6. Rainfall-Intensity Performance

The intensity evaluation contains 35 records across seven horizons and five rainfall-intensity bands.

The evaluation distinguishes performance at different rainfall magnitudes rather than relying exclusively on aggregate error.

The detailed results are stored in:

`WI_05_seven_horizon_intensity_summary.csv`

The candidate's aggregate performance does not imply consistent performance across rainfall intensities. The highest rainfall-intensity band requires particular attention.

## 7. Heavy-Rainfall Diagnostics

Heavy rainfall is evaluated using the 50+ mm/day band.

| Horizon | Candidate MAE | Persistence MAE | Candidate RMSE | Persistence RMSE | MAE Improvement |
|---|---:|---:|---:|---:|---:|
| T+1 | 53.5433 | 49.1145 | 65.4931 | 63.0324 | -9.02% |
| T+2 | 61.3879 | 58.5887 | 72.4713 | 71.5838 | -4.78% |
| T+3 | 62.6588 | 61.5446 | 73.5146 | 73.5814 | -1.81% |
| T+4 | 63.5643 | 63.0502 | 74.2442 | 75.0987 | -0.82% |
| T+5 | 63.6622 | 63.5390 | 74.2972 | 75.4105 | -0.19% |
| T+6 | 63.8456 | 63.7419 | 74.4223 | 75.4055 | -0.16% |
| T+7 | 62.5509 | 63.5597 | 73.3555 | 75.1474 | +1.59% |

The heavy-rainfall results reveal a major limitation:

- Actual mean rainfall in this band is approximately 77.47 mm/day.
- Candidate mean predictions range from approximately 13.83 to 25.40 mm/day.
- Candidate heavy-band MAE is worse than persistence from T+1 through T+6.
- At T+7, candidate MAE improves on persistence by only approximately 1.59%.
- Candidate predictions substantially underestimate heavy-rainfall intensity.

**Conclusion:** These candidates should not be represented as validated heavy-rainfall warning models.

## 8. Wet-Day Classification Diagnostics

Detailed classifier threshold and calibration diagnostics were evaluated for T+1.

At the 0.40 classification threshold:

| Metric | Result |
|---|---:|
| Precision | 0.8088 |
| Recall | 0.8654 |
| F1-score | 0.8361 |
| PR-AUC | 0.9147 |
| Brier score | 0.099175 |

The 0.40 threshold produced the best F1-score among the tested thresholds. It remains a provisional operating threshold.

These classification results should not be generalized to T+2 through T+7 without equivalent horizon-specific evaluation.

## 9. Key Limitations

1. **Heavy-rainfall underestimation:** Candidate models substantially underestimate rainfall in the 50+ mm/day band.
2. **Uneven intensity performance:** Aggregate metrics conceal weaknesses in specific rainfall-intensity bands.
3. **T+7 aggregate trade-off:** T+7 candidate MAE is worse than climatology.
4. **Sampling:** Candidate development used a 352,800-row sample per horizon rather than the full feature-complete training dataset.
5. **Spatial scope:** Evaluation is grid-cell level; district-level aggregation and land masking are outside this work item's scope.
6. **Historical data limitation:** The source archive ends in 2024 and does not provide live weather forecasts.
7. **Agricultural interpretation:** Weather predictions alone do not establish crop damage, disease occurrence, or yield loss.
8. **Holdout:** The final 2023–2024 holdout remains untouched and requires a separately controlled evaluation.

## 10. Handoff to WI-07

WI-05 provides seven horizon-specific rainfall candidate models and validation evidence for final comparison.

WI-07 must:

- Compare candidates consistently against WI-04 baselines.
- Review aggregate MAE and RMSE together.
- Consider seasonal and intensity-band performance.
- Explicitly assess heavy-rainfall underestimation.
- Decide whether one model family is suitable across all horizons or whether horizon-specific choices are justified.
- Preserve the final holdout for controlled evaluation after model-selection decisions are complete.

No final production model is selected by WI-05.

## 11. Artifact Inventory

**Completion report**
- `WI_05_Rainfall_Model_Development_Completion_Report.md`

**Validation summaries**
- `WI_05_seven_horizon_validation_comparison.csv`
- `WI_05_seven_horizon_strongest_baseline_comparison.csv`
- `WI_05_seven_horizon_seasonal_summary.csv`
- `WI_05_seven_horizon_seasonal_summary_normalized.csv`
- `WI_05_seven_horizon_intensity_summary.csv`
- `WI_05_seven_horizon_heavy_rainfall_summary.csv`

**Candidate model artifacts**
- Seven wet-day classifiers
- Seven conditional rainfall amount regressors

All 14 candidate model artifacts were found during the final filesystem audit.

## 12. Completion Statement

WI-05 rainfall model development and validation diagnostics for T+1 through T+7 have been consolidated.

The completion report, validation summaries, and candidate model artifacts passed the final filesystem audit.

The work item is complete and ready for handoff to WI-07 for model comparison and final selection, subject to review of the documented limitations.

**Final holdout accessed: No.**